# Gut bacterial communities in invasive and native birds of the Iberian Peninsula

## Objective

Characterise and compare the cloacal bacterial community of two invasive parrot species
(*Myiopsitta monachus*, *Psittacula krameri*) and three native passerines (*Passer domesticus*,
*Sturnus unicolor*, *Turdus merula*) sampled in urban parks in Seville, Spain. Communities were
profiled by 16S rRNA amplicon sequencing on the Oxford Nanopore platform and summarized as
per-sample genus-level count tables.

Sampling effort was not equal across the five species. *Myiopsitta monachus* yielded only a
handful of birds that clear every quality and completeness requirement, which is too few to
estimate its own mean beside a locality effect; it is therefore reported descriptively throughout
— diversity summaries, per-bird figures and composition profiles — and left out of the models,
under the rule fixed in section 02.2. The inferential comparison is consequently between
*Psittacula krameri* and the three native passerines.

The analysis asks three questions, in this order:

1. **Between species.** Does the host species predict the diversity and composition of its gut
   bacterial community, once sampling locality is accounted for?
2. **Within species.** Within a host species, is diversity related to locality, sex or body
   condition?
3. **Composition.** Which bacterial genera drive any differences, and how consistent are they
   between individuals of the same host species?

| Section | Content | Input matrix |
| --- | --- | --- |
| 01.0 | Locate and load input datasets | raw `Genus_counts.tsv` files + metadata |
| 02.0 | Taxonomic curation and the analytical matrix | `community_matrix_pre_curation` |
| 03.0 | Sequencing depth diagnostics | `community_matrix_analysis` |
| 04.0 | Rarefaction depth selection | `community_matrix_analysis` |
| 05.0 | Alpha diversity | `rarefied_matrix` (50 replicates) |
| 06.0 | Between-species models | `alpha_data` |
| 07.0 | Within-species models | `intra_specific_datasets` |
| 08.0 | Beta diversity | `rarefied_matrix` → Bray–Curtis |
| 09.0 | Taxonomic composition | `composition_counts` → CLR |

### How to read this notebook

Each analysis cell ends with a printed verification block — what it produced, the checks it ran, and
an `assert` wherever an invariant has to hold for the next cell to mean anything. Every result table
then follows in a cell of its own, rendered as a searchable, sortable table, and each main figure is
followed by an interactive version whose points can be hovered to identify the individual bird or
genus behind them. Exporting the notebook

```bash
jupyter nbconvert --to html --embed-images TFM_microbiome_pipeline.ipynb
```

gives one self-contained page on which all of that stays usable with no kernel running, which is how
a reader arriving from the manuscript is expected to consult it.

### How the analysis is organized

Every curation and modeling decision is made against an explicit, stated criterion rather than a
convenient constant, and each criterion is fixed before the data that it acts on are inspected.
Nothing is filtered silently: every exclusion is written to `results/qc_reports/` with the reason
attached. Each code cell ends with a printed summary and, where an invariant has to hold for the
next cell to mean anything, an `assert` that checks it.

### Software

`numpy`, `pandas`, `scipy`, `matplotlib`, `statsmodels`, `patsy` and `scikit-learn` for the
analysis, and `plotly` and `itables` for the interactive figures and tables
(`pip install -r requirements.txt`). Principal coordinates analysis, PERMANOVA, PERMDISP, the
centered log-ratio transform and multiplicative zero replacement are implemented directly from
their published definitions in sections 08.1 and 09.5, so the notebook depends on nothing beyond
that list and runs unchanged on Windows, macOS and Linux.

### Reproducibility

Restart the kernel and run from top to bottom. No cell depends on state left by an earlier run.
All stochastic steps — rarefaction, permutation tests, Monte-Carlo intervals — draw from a single
shared random generator seeded once in section 01.1.

## 01.0 — Locate and load input datasets

Reads the sample metadata (`Base_de_datos.csv` / `.xlsx`) and the per-sample `Genus_counts.tsv`
files produced by the NAMECO taxonomic assignment pipeline, and assembles them into one
sample × genus count matrix.

Sample identifiers are built from the folder structure rather than from any column inside the
files, so that a mislabeled header cannot silently attach a count table to the wrong bird. The
two sources are then matched on that identifier: samples present in only one of them are listed in
`results/qc_reports/` with the side they were missing from, and the analysis continues with the
intersection.

The cell that follows sets the project paths, the random seed and the species coding used
throughout, so that every later section refers to one definition of each.

In [ ]:
# Install the two packages that produce the interactive tables and figures.
# Everything else this notebook uses ships with a standard scientific Python
# install; the full pinned environment is in requirements.txt.
%pip install itables plotly

In [ ]:
# 01.1 — Imports, dependency check and the shared random generator
# ----------------------------------------------------------------------
from __future__ import annotations

import importlib.util
import sys

# Check the environment before anything else, and fail with one message
# naming every missing package. sys.executable is printed because the
# usual cause is a Jupyter kernel pointing at a different environment
# from the one the packages were installed into.
_REQUIRED = {
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "scipy": "scipy", "statsmodels": "statsmodels", "patsy": "patsy",
    "sklearn": "scikit-learn",
    # Interactive tables and figures in the exported HTML (01.2, 10.0).
    "itables": "itables", "plotly": "plotly",
}
_missing = sorted(pip_name for mod, pip_name in _REQUIRED.items() if importlib.util.find_spec(mod) is None)
if _missing:
    raise ModuleNotFoundError(
        "Missing required package(s): " + ", ".join(_missing) + "\n"
        f"This kernel is running: {sys.executable}\n"
        "Install them into THAT environment:\n"
        f"    pip install {' '.join(_missing)}\n"
        f"    conda install -c conda-forge {' '.join(_missing)}\n"
        "If the path above is not the environment you expected, switch the notebook kernel "
        "(Kernel > Change kernel) instead of installing again."
    )

import json
import re
import warnings
from itertools import combinations
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.optimize import curve_fit
from scipy.spatial.distance import pdist, squareform
from scipy.stats import hypergeom, spearmanr

# Reproducibility: a single, shared random Generator. Every stochastic
# step (rarefaction, permutation tests, ...) draws from a generator that
# is advanced sequentially, instead of re-seeding with the same value
# repeatedly -- re-seeding independent draws with e.g. seed=SEED + i
# silently correlates draws whose indices differ by a multiple of the
# seed spacing.
RANDOM_SEED = 42
_ROOT_RNG = np.random.default_rng(RANDOM_SEED)


def spawn_rng() -> np.random.Generator:
    """Fresh, independent RNG stream derived from the shared root generator."""
    return np.random.default_rng(_ROOT_RNG.integers(0, 2**63 - 1))


# VERBOSE_TAXA_LOG=False keeps the per-replicate "dropping N all-zero
# taxa" lines out of the notebook output (50 near-identical lines that
# hid the actual result); the counts are still summarized and saved.
VERBOSE_TAXA_LOG = False

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 130)

print("01.1 verification")
print("-" * 60)
print(f"Python executable : {sys.executable}")
print(f"pandas {pd.__version__} | numpy {np.__version__} | matplotlib {mpl.__version__}")
print(f"RANDOM_SEED = {RANDOM_SEED}")
assert callable(spawn_rng)
print("OK -- imports and configuration ready.")

In [ ]:
# 01.2 — Presentation layer: figure style, species colors and interactive output
# -----------------------------------------------------------------------------
# Every figure in this notebook reads categories from the SAME maps, so a
# color always means the same species everywhere. Requirements met here:
#
#  * one distinct hue per species, from the Okabe-Ito color-blind-safe
#    palette (no two species share a hue or a light/dark pair);
#  * origin (invasive / native) is encoded REDUNDANTLY -- warm hues and
#    filled markers for invasive, cool hues and open-centered markers for
#    native -- so the figure survives grayscale printing and does not
#    rely on color alone;
#  * a distinct marker shape per species, for the same reason;
#  * `figure_caption()` writes a self-contained caption under every
#    figure, so each one can be read without the surrounding code.

PLOT_RCPARAMS = {
    "figure.dpi": 110, "savefig.dpi": 300, "figure.figsize": (8, 5),
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 10.5,
    "xtick.labelsize": 9.5, "ytick.labelsize": 9.5, "legend.fontsize": 9,
    "legend.frameon": True, "legend.framealpha": 0.9, "legend.edgecolor": "0.8",
    "font.size": 10, "figure.facecolor": "white", "axes.facecolor": "white",
}
plt.rcParams.update(PLOT_RCPARAMS)

# Okabe-Ito: vermillion / reddish-purple (warm = invasive),
# blue / bluish-green / sky-blue (cool = native).
SPECIES_COLORS = {
    "MyoMon": "#D55E00",   # vermillion
    "PsiKra": "#CC79A7",   # reddish purple
    "PasDom": "#0072B2",   # blue
    "Stuuni": "#009E73",   # bluish green
    "TurMer": "#56B4E9",   # sky blue
}
SPECIES_MARKERS = {"MyoMon": "o", "PsiKra": "s", "PasDom": "^", "Stuuni": "D", "TurMer": "v"}

# Qualitative palette for genus-level figures (Paul Tol "muted"): 12
# hues, color-blind safe, and with no confusable light/dark pairs of the
# kind `tab20` produces (the previous stacked bar put two near-identical
# pinks next to each other).
GENUS_PALETTE = [
    "#332288", "#88CCEE", "#44AA99", "#117733", "#999933", "#DDCC77",
    "#CC6677", "#882255", "#AA4499", "#661100", "#6699CC", "#E69F00",
]
COLOR_OTHER = "#BBBBBB"        # "other confirmed genera" -- neutral gray
COLOR_BOOKKEEPING = "#FFFFFF"  # excluded/unresolved reads -- white + hatch
LOCALITY_PALETTE = ["#4477AA", "#EE6677", "#228833", "#CCBB44", "#AA3377", "#66CCEE"]


def figure_caption(fig, text: str, y: float = -0.02) -> None:
    """Self-contained caption under a figure: what is plotted, N, and the test behind it."""
    fig.text(0.5, y, text, ha="center", va="top", fontsize=8.5, color="0.25", wrap=True)


def format_p(value: float, label: str = "p") -> str:
    """Readable p/q value: never prints '0', never prints 15 decimals."""
    if not np.isfinite(value):
        return f"{label} = n/a"
    if value < 1e-4:
        return f"{label} < 0.0001"
    return f"{label} = {value:.4g}"


def italic(name: str) -> str:
    """Matplotlib mathtext italic, safe for names containing underscores."""
    return f"$\\it{{{str(name).replace('_', chr(92) + '_')}}}$"


def significance_stars(p: float) -> str:
    """Conventional significance marker; 'ns' when not significant."""
    if not np.isfinite(p):
        return "n/a"
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"


def save_figure(fig, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    return path


# ------------------------------------------------------------------
# Interactive output, for reading this notebook as an exported HTML page
# ------------------------------------------------------------------
# Every result in this notebook is reported twice, on purpose. The printed
# block at the end of each cell is the VERIFICATION: plain text, readable
# in any viewer, and the thing that confirms the cell did what its header
# says. Each result table then follows in a cell of its own, rendered as a
# sortable, searchable, paged table, and each main figure is followed by an
# interactive version whose points can be hovered to identify the
# individual bird or genus behind them.
#
# The point of this is that the exported HTML should be explorable on its
# own. Somebody who arrives from the manuscript can open one file, search
# a 200-genus composition table or find which bird is the outlier in an
# ordination, and never install anything or re-run a cell.
from IPython.display import HTML, display

import plotly
import plotly.graph_objects as go
import plotly.io as pio
from itables import init_notebook_mode
from itables import show as _interactive_table
import itables.options as itables_options

# connected=False writes the table library into the notebook itself
# instead of linking a copy on the internet, which is what keeps the
# exported HTML working offline and years from now.
init_notebook_mode(all_interactive=False, connected=False)
itables_options.maxBytes = 0            # never silently truncate a result table
itables_options.classes = "display compact"
itables_options.scrollX = True
# A table caption belongs above the table it names, not under it.
itables_options.style = "table-layout:auto;width:auto;margin:auto;caption-side:top"

# The "notebook" renderer embeds the plotting library once in this
# notebook, for the same reason: the interactive figures have to stay
# alive in the exported HTML with no kernel running behind it.
pio.renderers.default = "notebook"
PLOTLY_TEMPLATE = "simple_white"
# matplotlib marker codes -> the equivalent symbol names, so one species
# keeps its shape as well as its color across both kinds of figure.
PLOTLY_MARKERS = {"o": "circle", "s": "square", "^": "triangle-up",
                  "D": "diamond", "v": "triangle-down"}


def html_italic(name: str) -> str:
    """Italic name for an interactive figure label, where the renderer reads HTML."""
    return f"<i>{name}</i>"


def species_html(code_: str, *, origin: bool = False) -> str:
    """Italic binomial for an interactive legend, optionally with its origin."""
    label = html_italic(SPECIES_CODE_TO_NAME.get(code_, code_))
    return f"{label} ({SPECIES_ORIGIN.get(code_, '?')})" if origin else label


def show_table(frame: pd.DataFrame, caption: str, *, decimals: int = 4,
               index: bool = False, page_length: int = 10) -> None:
    """Render one result table as a sortable, searchable, paged table.

    Floating-point columns are rounded for display only; the CSV written
    beside it always carries the full precision, so nothing a reader
    might recompute from is lost to rounding here.
    """
    display_frame = frame.copy()
    if index:
        display_frame = display_frame.reset_index()
    float_columns = display_frame.select_dtypes(include=["float", "float32", "float64"]).columns
    display_frame[float_columns] = display_frame[float_columns].round(decimals)
    # The page-length menu always contains the page length this table opens
    # at, otherwise the selector renders with nothing chosen.
    length_menu = sorted({page_length, 10, 25, 50, 100})
    _interactive_table(
        display_frame, caption=caption, paging=len(display_frame) > page_length,
        pageLength=page_length, lengthMenu=length_menu, order=[], showIndex=False, scrollX=True,
    )


print("01.2 verification")
print("-" * 60)
assert format_p(1e-9) == "p < 0.0001" and format_p(np.nan) == "p = n/a"
print(f"Species colors : {SPECIES_COLORS}")
print(f"Species markers : {SPECIES_MARKERS}")
assert len(set(SPECIES_COLORS.values())) == len(SPECIES_COLORS), "Two species share a color."
assert len(set(SPECIES_MARKERS.values())) == len(SPECIES_MARKERS), "Two species share a marker."
assert set(SPECIES_MARKERS.values()) <= set(PLOTLY_MARKERS), \
    "A species marker has no interactive equivalent, so the two kinds of figure would disagree."
print(f"Interactive output : tables via itables, figures via plotly {plotly.__version__} "
      f"(both embedded, so the exported HTML needs no network)")
print("OK -- one distinct color and marker per species; shared by every figure below.")

In [ ]:
# 01.3 — Project directory layout and output locations
# ----------------------------------------------------------------------
# PROJECT_DIR is the single source of truth: every other path is derived
# from it. Default is the working directory the Jupyter kernel started
# in, which works with no edits as long as Jupyter is launched FROM the
# project root (the folder that directly contains data/, results/ and
# images_graphs/ -- e.g. D:\NAMECO on Windows).
#
# If your setup always opens Jupyter in a different fixed folder (common
# with Anaconda Navigator), set PROJECT_DIR_OVERRIDE to an absolute path.
# Use forward slashes even on Windows -- pathlib accepts them natively and
# they avoid backslash-escape mistakes:
#     PROJECT_DIR_OVERRIDE = "D:/NAMECO"
PROJECT_DIR_OVERRIDE: str | None = None

PROJECT_DIR = Path(PROJECT_DIR_OVERRIDE).resolve() if PROJECT_DIR_OVERRIDE else Path.cwd().resolve()

DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
DATA_SAMPLE_DIR = RAW_DATA_DIR / "data_sample"

RESULTS_DIR = PROJECT_DIR / "results"
TABLES_DIR = RESULTS_DIR / "tables"
STATISTICS_DIR = RESULTS_DIR / "statistics"
DIVERSITY_DIR = RESULTS_DIR / "diversity"
TAXONOMY_RESULTS_DIR = RESULTS_DIR / "taxonomy"
QC_REPORTS_DIR = RESULTS_DIR / "qc_reports"

FIGURES_DIR = PROJECT_DIR / "images_graphs"
QUALITY_DIR = FIGURES_DIR / "quality"
ALPHA_DIR = FIGURES_DIR / "alpha_diversity"
BETA_DIR = FIGURES_DIR / "beta_diversity"
TAXONOMY_FIG_DIR = FIGURES_DIR / "taxonomy"
# Residual and goodness-of-fit plots are figures, not result tables, so they
# live under images_graphs alongside the rest; results/diversity keeps the
# matrices and tables the diagnostics are computed from.
DIAGNOSTICS_FIG_DIR = FIGURES_DIR / "diagnostics"

ALL_OUTPUT_DIRECTORIES = [
    RAW_DATA_DIR, PROCESSED_DATA_DIR, TABLES_DIR, STATISTICS_DIR, DIVERSITY_DIR,
    TAXONOMY_RESULTS_DIR, QC_REPORTS_DIR, QUALITY_DIR, ALPHA_DIR, BETA_DIR, TAXONOMY_FIG_DIR,
    DIAGNOSTICS_FIG_DIR,
]
for _directory in ALL_OUTPUT_DIRECTORIES:
    _directory.mkdir(parents=True, exist_ok=True)

print("01.3 verification")
print("-" * 60)
print("PROJECT_DIR (auto-detected from the working directory unless overridden above):")
print(f"  {PROJECT_DIR}")
for _d in ALL_OUTPUT_DIRECTORIES:
    assert _d.exists(), f"Directory was not created: {_d}"
print(f"Output directories ready under results/ and images_graphs/ ({len(ALL_OUTPUT_DIRECTORIES)} total).")

# DATA_SAMPLE_DIR is an INPUT directory and is never created automatically.
assert DATA_SAMPLE_DIR.exists(), (
    f"data_sample/ was not found at {DATA_SAMPLE_DIR}. Expected it at "
    f"<PROJECT_DIR>/data/raw/data_sample/, with PROJECT_DIR = {PROJECT_DIR}. "
    f"If PROJECT_DIR above is not your project root, set PROJECT_DIR_OVERRIDE at the top of this cell."
)
_species_subfolders = sorted(p.name for p in DATA_SAMPLE_DIR.iterdir() if p.is_dir() and not p.name.startswith("."))
assert len(_species_subfolders) > 0, (
    f"DATA_SAMPLE_DIR exists ({DATA_SAMPLE_DIR}) but contains no species subfolders."
)
print(f"DATA_SAMPLE_DIR = {DATA_SAMPLE_DIR}")
print(f"  Species subfolders found ({len(_species_subfolders)}): {_species_subfolders}")
print("OK -- project directories ready.")

In [ ]:
# 01.4 — Host species codes, origin, reference level and metadata column names
# ----------------------------------------------------------------------
SPECIES_ORDER = ["MyoMon", "PasDom", "PsiKra", "Stuuni", "TurMer"]

# Reference level for every Species contrast. Declared once, here, and
# applied by making Species_code an ordered Categorical whose first
# category is this species, which patsy then takes as the treatment
# reference; the model formulas stay plain `C(Species_code)`.
#
# The choice of baseline is a reporting decision, not a statistical one:
# the global tests and the post-hoc pairwise comparisons are invariant to
# it, and only the regression coefficients and their intervals change.
# Turdus merula is used because it is well sampled (n = 30) and spread
# across three localities, so contrasts against it are precise and its
# intercept does not absorb a site effect. Sturnus unicolor has slightly
# more birds but 94% of them come from a single locality, which would put
# that locality inside the intercept. A baseline drawn from the smallest
# species would make every coefficient a contrast against the noisiest
# group and would obscure differences that the global test detects.
REFERENCE_SPECIES = "TurMer"

SPECIES_CODE_TO_NAME = {
    "MyoMon": "Myiopsitta monachus",
    "PsiKra": "Psittacula krameri",
    "PasDom": "Passer domesticus",
    "Stuuni": "Sturnus unicolor",
    "TurMer": "Turdus merula",
}
SPECIES_NAME_TO_CODE = {v: k for k, v in SPECIES_CODE_TO_NAME.items()}
SPECIES_LABELS_ITALIC = {
    code_: f"$\\it{{{name.replace(' ', '~')}}}$" for code_, name in SPECIES_CODE_TO_NAME.items()
}

INVASIVE_SPECIES = {"MyoMon", "PsiKra"}
NATIVE_SPECIES = {"PasDom", "Stuuni", "TurMer"}
SPECIES_ORIGIN = {s: ("Invasive" if s in INVASIVE_SPECIES else "Native") for s in SPECIES_ORDER}


def species_legend_label(code_: str, multiline: bool = False) -> str:
    """Italic binomial + origin, for figure legends that must stand alone."""
    sep = "\n" if multiline else " "
    return f"{SPECIES_LABELS_ITALIC.get(code_, code_)}{sep}({SPECIES_ORIGIN.get(code_, '?').lower()})"


def species_axis_label(code_: str) -> str:
    """Short two-line axis tick label: italic binomial over origin."""
    return f"{SPECIES_LABELS_ITALIC.get(code_, code_)}\n({SPECIES_ORIGIN.get(code_, '?').lower()})"


def _genus_abbreviation_length() -> int:
    """Shortest genus prefix that is unique across the study's species.

    A single initial is the usual convention, but this study has both
    Passer and Psittacula, so "P. domesticus" and "P. krameri" would both
    expand to a "P." that stands for two different genera. The prefix
    length is therefore derived from the actual species list instead of
    being assumed to be 1, and the same length is used for every species
    so the labels stay visually consistent.
    """
    genera = [SPECIES_CODE_TO_NAME[s].split(" ")[0] for s in SPECIES_ORDER]
    for length in range(1, max(len(g) for g in genera) + 1):
        if len({g[:length] for g in genera}) == len(set(genera)):
            return length
    return max(len(g) for g in genera)


GENUS_ABBREVIATION_LENGTH = _genus_abbreviation_length()


def abbreviated_binomial(code_: str) -> str:
    """Italic abbreviated binomial for narrow panels where the full name collides."""
    name = SPECIES_CODE_TO_NAME.get(code_)
    if not name or " " not in name:
        return code_
    genus, epithet = name.split(" ", 1)
    return f"$\\it{{{genus[:GENUS_ABBREVIATION_LENGTH]}.~{epithet.replace(' ', '~')}}}$"


# The raw Base_de_datos.xlsx uses Spanish column names with inconsistent
# trailing whitespace ("SEXO ", "TIPO DE MUESTRA "). Renamed once, right
# after loading.
METADATA_COLUMN_RENAME = {
    "ANILLA": "Ring_ID", "FECHA": "Date", "LOCALIDAD": "Locality", "ESPECIE": "Species",
    "EDAD": "Age", "SEXO": "Sex_code", "GRASA": "Fat", "ALA": "Wing", "PESO": "Mass",
    "TARSO": "Tarsus", "TIPO DE MUESTRA": "Sample_type", "OBSERVACIONES": "Observations",
}
# SEXO coding: 1=Male, 2=Female, 0=Unknown/undetermined (this ringing
# protocol codes undetermined sex as 0, not the more conventional 9). 0 is
# kept as its own "Unknown" category rather than assumed juvenile-only.
SEX_CODE_MAP = {0: "Unknown", 1: "Male", 2: "Female", 9: "Unknown"}

SAMPLE_TYPE_KEEP = "CLOACA"

print("01.4 verification")
print("-" * 60)
assert len(SPECIES_ORDER) == 5 == len(SPECIES_CODE_TO_NAME)
assert set(INVASIVE_SPECIES | NATIVE_SPECIES) == set(SPECIES_ORDER)
assert REFERENCE_SPECIES in SPECIES_ORDER
# The abbreviated labels must still identify a species uniquely.
_abbrev = [abbreviated_binomial(s) for s in SPECIES_ORDER]
assert len(set(_abbrev)) == len(_abbrev), f"Abbreviated species labels are not unique: {_abbrev}"
print(f"Species order      : {SPECIES_ORDER}")
print(f"Reference species  : {REFERENCE_SPECIES} ({SPECIES_CODE_TO_NAME[REFERENCE_SPECIES]})")
print(f"Invasive           : {sorted(INVASIVE_SPECIES)}")
print(f"Native             : {sorted(NATIVE_SPECIES)}")
print("OK -- species coding and metadata mapping ready.")

In [ ]:
# 01.5 — Locate the metadata file and the per-sample count tables
# ----------------------------------------------------------------------
def locate_metadata_file(raw_data_dir: Path) -> Path:
    """Accept Base_de_datos.xlsx OR .csv, in data/raw/ or in data/raw/data_sample/."""
    csv_candidates = sorted(raw_data_dir.rglob("Base_de_datos.csv"))
    xlsx_candidates = sorted(raw_data_dir.rglob("Base_de_datos.xlsx"))
    candidates = csv_candidates or xlsx_candidates
    if not candidates:
        raise FileNotFoundError(
            f"Neither Base_de_datos.csv nor Base_de_datos.xlsx was found under {raw_data_dir}."
        )
    if len(candidates) > 1:
        print(f"[WARN] {len(candidates)} copies found; using the first one: {candidates[0]}")
    if csv_candidates and xlsx_candidates:
        print(f"[NOTE] Both a .csv and a .xlsx copy exist; using the .csv one: {candidates[0]}")
    return candidates[0]


def locate_genus_count_files(data_sample_dir: Path) -> list[Path]:
    files = sorted(data_sample_dir.rglob("Genus_counts.tsv"))
    if not files:
        raise FileNotFoundError(f"No Genus_counts.tsv files found under {data_sample_dir}.")
    return files


metadata_file = locate_metadata_file(RAW_DATA_DIR)
genus_files = locate_genus_count_files(DATA_SAMPLE_DIR)

print("01.5 verification")
print("-" * 60)
print(f"Metadata file: {metadata_file}")
print(f"Genus_counts.tsv files found: {len(genus_files)}")
assert metadata_file.exists()
assert len(genus_files) > 0
print("OK -- input files located.")

In [ ]:
# 01.6 — Read the per-sample Genus_counts.tsv files into one sample x genus matrix
# ----------------------------------------------------------------------
def load_genus_count_file(file: Path) -> tuple[str, str, pd.Series]:
    """Return (species_code, sample_id, genus -> count Series) for one file."""
    df = pd.read_csv(file, sep="\t")
    if df.shape[1] < 2:
        raise ValueError(f"Unexpected structure in file: {file}")

    genus_col, abundance_col = df.columns[0], df.columns[-1]
    sample_id = str(abundance_col).split("_")[0]     # "M03NK_MyoMon_concat" -> "M03NK"
    species_code = file.parent.parent.name           # data_sample/<SpeciesCode>/<folder>/...

    table = df[[genus_col, abundance_col]].copy()
    table.columns = ["Genus", "Count"]
    table["Count"] = pd.to_numeric(table["Count"], errors="coerce")

    if table["Count"].isna().any():
        raise ValueError(f"Missing / non-numeric counts in {file}")
    if (table["Count"] < 0).any():
        raise ValueError(f"Negative counts in {file}")
    if not np.allclose(table["Count"], np.round(table["Count"])):
        raise ValueError(f"Non-integer counts in {file}")
    table["Count"] = table["Count"].astype(np.int64)

    return species_code, sample_id, table.groupby("Genus")["Count"].sum()


def build_raw_community_matrix(genus_files: list[Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    per_sample_series, inventory_rows = {}, []
    for file in genus_files:
        species_code, sample_id, counts = load_genus_count_file(file)
        unique_sample_id = f"{species_code}_{sample_id}"
        if unique_sample_id in per_sample_series:
            raise ValueError(f"Duplicate SampleID built from folder structure: {unique_sample_id} (file: {file})")
        per_sample_series[unique_sample_id] = counts
        inventory_rows.append({
            "SampleID": unique_sample_id, "ID": sample_id, "Species_code": species_code,
            "File": str(file), "Folder_sample": file.parent.name,
        })
    community_matrix = pd.DataFrame(per_sample_series).T.fillna(0).astype(np.int64)
    community_matrix.index.name = "SampleID"
    return community_matrix, pd.DataFrame(inventory_rows)


raw_community_matrix, sample_inventory = build_raw_community_matrix(genus_files)

print("01.6 verification")
print("-" * 60)
print(f"Raw community matrix: {raw_community_matrix.shape[0]} samples x {raw_community_matrix.shape[1]} genus-level labels")
assert not raw_community_matrix.isna().any().any()
assert (raw_community_matrix.values >= 0).all()
assert raw_community_matrix.index.is_unique
print("OK -- no missing values, no negative counts, no duplicate SampleIDs.")

In [ ]:
# 01.7 — Read and clean the sample metadata (species, locality, sex, morphometrics)
# ----------------------------------------------------------------------
def clean_id(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.upper()


# Morphometric columns that MUST end up as true floats: section 07.2
# computes log(Mass) and log(Tarsus), and a comma-decimal string that
# survived as text would break that or silently become NaN.
NUMERIC_METADATA_COLUMNS = ["Fat", "Wing", "Mass", "Tarsus"]


def _read_metadata_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        # sep=None + engine="python" auto-detects comma vs. semicolon (an
        # Excel "CSV" export from a Spanish locale is semicolon-delimited).
        # encoding="utf-8-sig" transparently strips Excel's BOM.
        return pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
    return pd.read_excel(path)


def load_and_clean_metadata(metadata_file: Path) -> pd.DataFrame:
    metadata = _read_metadata_table(metadata_file)
    metadata.columns = metadata.columns.astype(str).str.strip()

    unmapped_columns = set(METADATA_COLUMN_RENAME) - set(metadata.columns)
    if unmapped_columns:
        raise KeyError(
            f"Expected metadata columns not found (after stripping whitespace): "
            f"{sorted(unmapped_columns)}. Columns present: {sorted(metadata.columns)}"
        )
    metadata = metadata.rename(columns=METADATA_COLUMN_RENAME)

    for _col in NUMERIC_METADATA_COLUMNS:
        if _col not in metadata.columns:
            continue
        if not pd.api.types.is_numeric_dtype(metadata[_col]):
            metadata[_col] = metadata[_col].astype("string").str.strip().str.replace(",", ".", regex=False)
        metadata[_col] = pd.to_numeric(metadata[_col], errors="coerce").astype("float64")

    metadata["ID_clean"] = clean_id(metadata["ID"])
    metadata["Species_code"] = metadata["Species"].map(SPECIES_NAME_TO_CODE)
    unmapped_species = metadata.loc[metadata["Species_code"].isna(), "Species"].unique()
    if len(unmapped_species):
        raise ValueError(f"Species values with no entry in SPECIES_NAME_TO_CODE: {sorted(unmapped_species)}")

    metadata["Sex"] = metadata["Sex_code"].map(SEX_CODE_MAP)
    unmapped_sex = metadata.loc[metadata["Sex"].isna(), "Sex_code"].unique()
    if len(unmapped_sex):
        raise ValueError(f"Sex_code values with no entry in SEX_CODE_MAP: {sorted(unmapped_sex)}")
    return metadata


metadata = load_and_clean_metadata(metadata_file)

print("01.7 verification")
print("-" * 60)
print(f"Metadata file read as: {'.csv (auto-detected delimiter)' if metadata_file.suffix.lower() == '.csv' else '.xlsx'}")
print(f"Metadata rows: {len(metadata)}")
print(f"Species present: {sorted(metadata['Species_code'].unique())}")
assert metadata["Species_code"].notna().all()
assert metadata["Sex"].notna().all()

print("\nNumeric morphometric columns (dtype + missing values):")
for _col in NUMERIC_METADATA_COLUMNS:
    if _col not in metadata.columns:
        continue
    print(f"  {_col:8s}: dtype={metadata[_col].dtype}, missing={int(metadata[_col].isna().sum())}/{len(metadata)}")
    assert pd.api.types.is_numeric_dtype(metadata[_col]), f"{_col} did not parse as numeric."
print("\nOK -- metadata loaded, columns renamed, Species/Sex mapped, numeric columns confirmed numeric.")

In [ ]:
# 01.8 — Match count tables to metadata -> community_matrix_pre_curation + analysis_metadata_raw
# ----------------------------------------------------------------------
def match_samples_to_metadata(sample_inventory: pd.DataFrame, metadata: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """Match each microbiome sample to exactly one metadata row.

    Returns (matched, qc_reports); qc_reports documents every excluded
    sample and why, for the Methods / Supplementary Materials.
    """
    sample_inventory = sample_inventory.copy()
    sample_inventory["ID_clean"] = clean_id(sample_inventory["ID"])
    qc_reports: dict[str, pd.DataFrame] = {}

    qc_reports["duplicated_ids_in_metadata"] = metadata.loc[
        metadata["ID_clean"].duplicated(keep=False), ["ID", "Species", "ID_clean"]
    ].sort_values("ID_clean")

    # An original ID appearing under more than one species folder is an
    # upstream labeling error that cannot be resolved automatically --
    # excluded, not merged, and kept on record.
    ids_per_species = sample_inventory.groupby("ID_clean")["Species_code"].nunique()
    ambiguous_ids = ids_per_species[ids_per_species > 1].index.tolist()
    qc_reports["ambiguous_ids_multiple_species"] = sample_inventory[
        sample_inventory["ID_clean"].isin(ambiguous_ids)
    ].copy()
    clean_inventory = sample_inventory[~sample_inventory["ID_clean"].isin(ambiguous_ids)]

    merged = clean_inventory.merge(
        metadata, on=["ID_clean", "Species_code"], how="left",
        suffixes=("", "_metadata"), indicator=True,
    )
    qc_reports["unmatched_to_metadata"] = merged.loc[merged["_merge"] != "both", ["SampleID", "ID", "Species_code"]]
    matched = merged[merged["_merge"] == "both"].drop(columns=["_merge"])

    if "Sample_type" not in matched.columns:
        raise KeyError("Expected column 'Sample_type' not found in metadata.")
    is_cloaca = matched["Sample_type"].astype("string").str.strip().str.upper().eq(SAMPLE_TYPE_KEEP)
    qc_reports["excluded_non_cloacal_samples"] = matched.loc[
        ~is_cloaca, ["SampleID", "ID", "Species_code", "Sample_type"]
    ]
    matched = matched.loc[is_cloaca].copy()

    dup_sample_ids = matched[matched["SampleID"].duplicated(keep=False)]
    if not dup_sample_ids.empty:
        raise ValueError(f"Duplicate SampleID(s) remain after QC filtering: {dup_sample_ids['SampleID'].unique().tolist()}")
    return matched.set_index("SampleID"), qc_reports


matched_metadata, qc_reports = match_samples_to_metadata(sample_inventory, metadata)
for _name, _report in qc_reports.items():
    _report.to_csv(QC_REPORTS_DIR / f"{_name}.csv", index=False)

community_matrix_pre_curation = raw_community_matrix.loc[matched_metadata.index].copy()
analysis_metadata_raw = matched_metadata

print("01.8 verification")
print("-" * 60)
print(f"Ambiguous IDs excluded (appear under >1 species folder): {len(qc_reports['ambiguous_ids_multiple_species'])}")
print(f"Unmatched to metadata: {len(qc_reports['unmatched_to_metadata'])}")
print(f"Excluded non-cloacal samples: {len(qc_reports['excluded_non_cloacal_samples'])}")
print(f"Samples matched to metadata and confirmed cloacal: {len(matched_metadata)}")
print("\nSamples per species:")
print(analysis_metadata_raw["Species_code"].value_counts().reindex(SPECIES_ORDER).to_string())

assert community_matrix_pre_curation.index.equals(analysis_metadata_raw.index)
assert (community_matrix_pre_curation.sum(axis=1) > 0).all(), "Found a sample with zero reads after QC"
assert not community_matrix_pre_curation.isna().any().any()
assert (community_matrix_pre_curation >= 0).all().all()

community_matrix_pre_curation.to_csv(PROCESSED_DATA_DIR / "community_matrix_pre_curation.csv")
analysis_metadata_raw.to_csv(PROCESSED_DATA_DIR / "analysis_metadata_pre_curation.csv")
print(f"\nSaved matrix   -> {PROCESSED_DATA_DIR / 'community_matrix_pre_curation.csv'}")
print(f"Saved metadata -> {PROCESSED_DATA_DIR / 'analysis_metadata_pre_curation.csv'}")
print("OK -- section 01.0 complete: matched, cloaca-filtered, pre-curation dataset built.")

## 02.0 — Taxonomic curation and the analytical matrix

The raw matrix carries one column per label emitted by the assignment pipeline, and those labels
are not all genera: some are higher ranks, some are placeholder identifiers, and some are
synonyms of a genus already present as its own column. Treating them all as genera would inflate
richness with duplicates and with taxa that were never resolved to genus rank in the first place.

Each original label therefore receives exactly one of four decisions, recorded with its evidence
in `results/taxonomy/taxonomic_correspondence.csv`:

- **confirmed** — a valid genus, kept as its own column;
- **merged** — a synonym or spelling variant of another genus, its counts added to that genus;
- **excluded** — resolved above genus rank, or not a taxon at all;
- **unresolved** — a placeholder identifier with no genus behind it.

Reads under *excluded* and *unresolved* are not discarded quietly. They are carried as an explicit
per-sample total so that the read accounting closes exactly — curated reads plus excluded reads
equal the original library size for every sample — and so that the share of a library that never
reached a confirmed genus can be reported per host species, which section 02.5 does.

Two further steps produce the matrix the rest of the notebook uses. Genera that are zero across
every sample are dropped, and a low-support filter removes genera seen in fewer than
`MIN_SAMPLE_PREVALENCE` samples or with fewer than `MIN_TOTAL_COUNT` reads in total. The result is
`community_matrix_analysis`, the single curated matrix that sections 03.0 onward operate on.

In [ ]:
# 02.1 — Curation rules: how a column label is judged to be a genus
# ----------------------------------------------------------------------
# A rank must be CONFIRMED (from a reference taxonomy, or from
# unambiguous text such as "<Family> unclassified") before a label is
# excluded as "above genus". A bare alphanumeric code (UBA1234, ECMA0423,
# GCA_902362365, ...) is NOT assumed to be above genus just because it is
# not a Latin binomial: in GTDB it may well be a valid, unnamed,
# genus-level placeholder. Without confirmation it is "Unresolved" --
# neither excluded nor pooled.
def _locate_taxonomic_reference() -> "Path | None":
    for _candidate in [RAW_DATA_DIR / "gtdb_taxonomic_reference.tsv",
                       DATA_SAMPLE_DIR / "gtdb_taxonomic_reference.tsv"]:
        if _candidate.exists():
            return _candidate
    return None


TAXONOMIC_REFERENCE_PATH: Path | None = _locate_taxonomic_reference()
TAXONOMIC_REFERENCE_COLUMNS = {"label": "Original_taxon", "rank": "Rank", "accepted_genus": "Accepted_genus"}


def is_unclassified_taxon(taxon: str) -> bool:
    """True only when the label ITSELF states a higher rank / no resolution."""
    taxon_lower = str(taxon).strip().lower()
    return any(k in taxon_lower for k in ["unclassified", "uncultured", "unknown"])


_HIGHER_RANK_SUFFIXES = {
    "aceae": "Family", "ales": "Order", "ineae": "Suborder", "inae": "Subfamily",
    "bacteria": "Phylum/Domain", "archaea": "Phylum/Domain",
}


def guess_higher_rank(taxon: str) -> str:
    taxon_lower = str(taxon).strip().lower()
    first_word = taxon_lower.split(" ")[0] if " " in taxon_lower else taxon_lower
    for suffix, rank in _HIGHER_RANK_SUFFIXES.items():
        if first_word.endswith(suffix):
            return rank
    return "Unresolved-higher-rank"


# Explicit genus-level merges: correspondences that have actually been
# checked, never inferred from string shape. Empty by default -- four
# suffix-based guesses (Helicobacter_H, Campylobacter_D, Enterococcus_B/D)
# were removed once checked against the real GTDB reference: they are
# distinct valid genera, not synonyms.
GENUS_MERGE_MAP: dict[str, str] = {}
UNRESOLVED_LABEL = "Unresolved"

print("02.1 verification")
print("-" * 60)
if TAXONOMIC_REFERENCE_PATH is not None:
    print(f"TAXONOMIC_REFERENCE_PATH = {TAXONOMIC_REFERENCE_PATH}  (found)")
else:
    print("[WARN] gtdb_taxonomic_reference.tsv was not found in data/raw/ or data/raw/data_sample/. "
          "Every label that is not an unambiguous Latin binomial will be left 'Unresolved'.")
print(f"GENUS_MERGE_MAP has {len(GENUS_MERGE_MAP)} manually validated entries.")
assert is_unclassified_taxon("Lactobacillaceae unclassified")
assert not is_unclassified_taxon("Escherichia")
print("OK -- taxonomic curation rules defined.")

In [ ]:
# 02.2 — Analysis parameters: filters, significance, FDR, and the depth and species policies
# ------------------------------------------------------------------------------------------
# NOT part of taxonomic curation (which only resolves identity and never
# removes a genuinely identified genus for being rare). Optional QC step
# applied AFTER curation, to confirmed genus-level taxa only.
APPLY_LOW_SUPPORT_FILTER = True
MIN_SAMPLE_PREVALENCE = 2
MIN_TOTAL_COUNT = 10

ALPHA_SIGNIFICANCE = 0.05
FDR_METHOD = "fdr_bh"
MIN_N_PER_LEVEL = 5

# -- Which species carry an inferential comparison ------------------------
# A host species enters the between-species models only if it has at least
# this many birds in the final complete-case design. The reason is not
# statistical fashion but what the estimate would be worth: a species mean
# that has to be estimated alongside a locality effect from a handful of
# birds comes with an interval wide enough to overlap every other species,
# so it can neither support nor refute anything -- and because the compact
# letter display and the pairwise contrasts are built from all species at
# once, that one unresolvable mean widens the comparisons among the species
# that ARE well sampled. Such a species is described in full (05.3, 05.4,
# 09.1-09.4) and left out of the models, which is the same policy already
# applied within species in 07.4, at the threshold that stage requires.
MIN_N_FOR_SPECIES_MODEL = 10

# -- Rarefaction-depth policy --------------------------------------------
# Defined HERE, at the top of the notebook, rather than in section 04.3
# where it is used. Two reasons: the quality floor is drawn on the
# sequencing-depth figure in 03.2, which runs earlier; and a depth rule is
# only meaningfully "pre-specified" if it is fixed before anything in the
# data has been looked at. Section 04.3 explains what each one does and
# re-prints them next to the trade-off they resolve.
MIN_LIBRARY_DEPTH = 10_000          # below this a library has failed, and is excluded
TARGET_SAMPLE_RETENTION = 0.95      # deepest depth keeping this share of QC-passing libraries
MIN_SPECIES_N_RETAINED = MIN_N_PER_LEVEL   # absolute floor of birds per species
MIN_ADEQUATE_ASYMPTOTE_PCT = 80.0   # adequacy check on the chosen depth
MIN_ASYMPTOTE_FIT_R2 = 0.90         # curve fits worse than this do not vote on adequacy


from statsmodels.stats.multitest import multipletests


def fdr_within_family(pvalues: dict[str, float], alpha: float = ALPHA_SIGNIFICANCE,
                      method: str = FDR_METHOD) -> pd.DataFrame:
    """Benjamini-Hochberg q-values for exactly this family of tests.

    Callers decide what belongs in one family; unrelated metrics or
    comparisons are never pooled into a single correction.
    """
    labels = list(pvalues.keys())
    raw = np.array([pvalues[k] for k in labels], dtype=float)
    finite = np.isfinite(raw)
    qvalues = np.full_like(raw, np.nan)
    reject = np.zeros(len(raw), dtype=bool)
    if finite.any():
        # A test that failed to converge (NaN) is excluded from the
        # correction rather than being treated as p = 1, which would both
        # dilute the correction and hide the failure.
        rej, q, _, _ = multipletests(raw[finite], alpha=alpha, method=method)
        qvalues[finite], reject[finite] = q, rej
    if (~finite).any():
        print(f"[WARN] {int((~finite).sum())} test(s) in this family returned a non-finite p-value and were "
              f"excluded from the FDR correction: {[l for l, f in zip(labels, finite) if not f]}")
    return pd.DataFrame({"Comparison": labels, "p_value": raw, "q_value": qvalues, "Significant_FDR": reject})

print("02.2 verification")
print("-" * 60)
print(f"APPLY_LOW_SUPPORT_FILTER={APPLY_LOW_SUPPORT_FILTER}, MIN_SAMPLE_PREVALENCE={MIN_SAMPLE_PREVALENCE}, MIN_TOTAL_COUNT={MIN_TOTAL_COUNT}")
print(f"ALPHA_SIGNIFICANCE={ALPHA_SIGNIFICANCE}, FDR_METHOD={FDR_METHOD!r}, MIN_N_PER_LEVEL={MIN_N_PER_LEVEL}")
print(f"Depth policy: MIN_LIBRARY_DEPTH={MIN_LIBRARY_DEPTH:,}, "
      f"TARGET_SAMPLE_RETENTION={TARGET_SAMPLE_RETENTION:.0%}, "
      f"MIN_SPECIES_N_RETAINED={MIN_SPECIES_N_RETAINED}, "
      f"MIN_ADEQUATE_ASYMPTOTE_PCT={MIN_ADEQUATE_ASYMPTOTE_PCT:.0f}%")
print("OK -- filter/statistical settings defined.")

In [ ]:
# 02.3 — External taxonomic reference (used to confirm genus names when available)
# ----------------------------------------------------------------------
def load_taxonomic_reference(path: "Path | None") -> "pd.DataFrame | None":
    if path is None:
        return None
    if not Path(path).exists():
        raise FileNotFoundError(f"TAXONOMIC_REFERENCE_PATH is set but not found: {path}")
    cols = TAXONOMIC_REFERENCE_COLUMNS
    ref = pd.read_csv(path, sep=None, engine="python")
    missing = {cols["label"], cols["rank"]} - set(ref.columns)
    if missing:
        raise KeyError(f"Taxonomic reference file is missing expected column(s) {missing}.")
    ref = ref.rename(columns={
        cols["label"]: "Original_taxon", cols["rank"]: "Reference_rank",
        cols.get("accepted_genus", ""): "Reference_accepted_genus",
    })
    keep = [c for c in ["Original_taxon", "Reference_rank", "Reference_accepted_genus"] if c in ref.columns]
    return ref[keep].drop_duplicates(subset="Original_taxon")


taxonomic_reference = load_taxonomic_reference(TAXONOMIC_REFERENCE_PATH)

print("02.3 verification")
print("-" * 60)
if taxonomic_reference is None:
    print("No reference connected -- unconfirmable labels will be marked 'Unresolved'.")
else:
    print(f"Reference loaded: {len(taxonomic_reference)} labels, columns {list(taxonomic_reference.columns)}")
    assert taxonomic_reference["Original_taxon"].is_unique
print("OK.")

In [ ]:
# 02.4 — Correspondence table: one curation decision per column of community_matrix_pre_curation
# ----------------------------------------------------------------------
def _looks_like_plain_genus_name(label: str) -> bool:
    """Conservative text-only check: capitalised, letters only, no suffix, no digits."""
    return bool(re.match(r"^[A-Z][a-z]+$", str(label).strip()))


def build_correspondence_table(original_labels: list[str], reference: "pd.DataFrame | None") -> pd.DataFrame:
    table = pd.DataFrame({"Original_taxon": [str(t) for t in original_labels]})
    table["Final_taxon"] = table["Original_taxon"]
    table["Taxonomic_rank"] = "Unresolved"
    table["Decision"] = UNRESOLVED_LABEL
    table["Evidence"] = "No confirmation available"
    if reference is not None:
        table = table.merge(reference, on="Original_taxon", how="left")

    ref_rank_col = table["Reference_rank"] if "Reference_rank" in table.columns else pd.Series(np.nan, index=table.index)
    ref_acc_col = table["Reference_accepted_genus"] if "Reference_accepted_genus" in table.columns else pd.Series(np.nan, index=table.index)

    decisions, ranks, finals, evidence = [], [], [], []
    for i, label in enumerate(table["Original_taxon"]):
        ref_rank, accepted = ref_rank_col.iloc[i], ref_acc_col.iloc[i]
        if is_unclassified_taxon(label):
            finals.append(label); ranks.append(guess_higher_rank(label)); decisions.append("Excluded")
            evidence.append("Label states a higher-rank fallback ('unclassified'/'uncultured'/'unknown')")
        elif pd.notna(ref_rank) and str(ref_rank).strip().lower() == "genus":
            if pd.notna(accepted) and str(accepted).strip() and str(accepted).strip() != label:
                finals.append(str(accepted).strip()); ranks.append("Genus"); decisions.append("Merged")
                evidence.append(f"Reference confirms genus rank; accepted name = {accepted}")
            else:
                finals.append(label); ranks.append("Genus"); decisions.append("Retained")
                evidence.append("Reference confirms genus rank")
        elif pd.notna(ref_rank):
            finals.append(label); ranks.append(str(ref_rank).strip()); decisions.append("Excluded")
            evidence.append(f"Reference confirms rank above genus ({ref_rank})")
        elif label in GENUS_MERGE_MAP:
            finals.append(GENUS_MERGE_MAP[label]); ranks.append("Genus"); decisions.append("Merged")
            evidence.append("Validated correspondence (GENUS_MERGE_MAP)")
        elif _looks_like_plain_genus_name(label):
            finals.append(label); ranks.append("Genus"); decisions.append("Retained")
            evidence.append("Unambiguous Latin genus name, no suffix/placeholder pattern")
        else:
            finals.append(label); ranks.append("Unresolved"); decisions.append(UNRESOLVED_LABEL)
            evidence.append("Label not found in the reference; rank not confirmable from text alone")

    table["Final_taxon"], table["Taxonomic_rank"] = finals, ranks
    table["Decision"], table["Evidence"] = decisions, evidence
    return table[["Original_taxon", "Final_taxon", "Taxonomic_rank", "Decision", "Evidence"]]


correspondence = build_correspondence_table(community_matrix_pre_curation.columns.tolist(), taxonomic_reference)

print("02.4 verification")
print("-" * 60)
decision_counts = correspondence["Decision"].value_counts()
for _decision in ["Retained", "Merged", "Excluded", "Unresolved"]:
    print(f"  {_decision:11s}: {int(decision_counts.get(_decision, 0))} label(s)")
assert set(correspondence["Decision"].unique()) <= {"Retained", "Merged", "Excluded", "Unresolved"}
assert correspondence["Original_taxon"].is_unique
print("OK -- every original label has exactly one decision.")

In [ ]:
# 02.5 — Apply curation, verify the read accounting, and report assignment success by species
# ----------------------------------------------------------------------
def apply_curation(community_matrix: pd.DataFrame, correspondence: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    quantitative = correspondence[correspondence["Decision"].isin(["Retained", "Merged"])]
    dropped = correspondence[~correspondence["Decision"].isin(["Retained", "Merged"])]
    kept_columns = [c for c in community_matrix.columns if str(c) in set(quantitative["Original_taxon"])]
    label_map = quantitative.set_index("Original_taxon")["Final_taxon"]
    curated_labels = [label_map[str(c)] for c in kept_columns]
    curated_matrix = community_matrix[kept_columns].T.groupby(curated_labels, sort=False).sum().T
    curated_matrix = curated_matrix.loc[community_matrix.index].round().astype(np.int64)
    curated_matrix.index.name = community_matrix.index.name
    dropped_columns = [c for c in community_matrix.columns if str(c) in set(dropped["Original_taxon"])]
    excluded_reads = (community_matrix[dropped_columns].sum(axis=1) if dropped_columns
                      else pd.Series(0, index=community_matrix.index))
    return curated_matrix, excluded_reads


def verify_curation_integrity(original: pd.DataFrame, curated: pd.DataFrame, excluded_reads: pd.Series) -> None:
    assert original.index.equals(curated.index), "Sample set changed during curation."
    assert not curated.index.duplicated().any(), "Duplicated samples after curation."
    reconstructed = curated.sum(axis=1) + excluded_reads
    original_totals = original.sum(axis=1)
    if not np.allclose(reconstructed.values, original_totals.values, rtol=1e-10, atol=1e-8):
        diff = reconstructed - original_totals
        bad = diff[~np.isclose(diff.values, 0, rtol=1e-10, atol=1e-8)]
        raise ValueError(f"Read accounting does not close for {len(bad)} sample(s): {bad.to_dict()}")


curated_matrix_unfiltered, excluded_reads = apply_curation(community_matrix_pre_curation, correspondence)
verify_curation_integrity(community_matrix_pre_curation, curated_matrix_unfiltered, excluded_reads)

print("02.5 verification")
print("-" * 60)
excluded_share_per_sample = (excluded_reads / community_matrix_pre_curation.sum(axis=1))
mean_excluded_share = float(excluded_share_per_sample.mean())
print(f"Confirmed genus-level taxa: {curated_matrix_unfiltered.shape[1]} (from {community_matrix_pre_curation.shape[1]} original labels)")
print(f"Mean share of reads per sample that are Excluded/Unresolved: {mean_excluded_share:.1%}")
if mean_excluded_share > 0.20:
    print("[WARN] Over 20% of reads are, on average, not assignable to a confirmed genus.")

# ------------------------------------------------------------------
# The OVERALL mean hides the thing that actually matters. Every
# between-species comparison in this notebook is computed on confirmed
# genus-level reads only, so if the share of reads that can be confirmed
# differs systematically BY SPECIES, then the species are not being
# compared on equivalent data: a host whose microbiota is poorly
# represented in the reference database enters every analysis with a
# different, smaller and non-randomly selected slice of its community.
# That is a confound for the study's central question, and a single
# pooled percentage cannot reveal it.
# ------------------------------------------------------------------
_species_of_sample = analysis_metadata_raw["Species_code"].reindex(excluded_share_per_sample.index)
assignment_by_species = (
    pd.DataFrame({"Species_code": _species_of_sample,
                  "Unresolved_share": excluded_share_per_sample})
    .dropna(subset=["Species_code"])
    .groupby("Species_code", observed=True)["Unresolved_share"]
    .agg(["count", "mean", "median", "min", "max"])
    .rename(columns={"count": "N", "mean": "Mean", "median": "Median",
                     "min": "Min", "max": "Max"})
    .sort_values("Mean", ascending=False)
)
assignment_by_species.to_csv(QC_REPORTS_DIR / "unresolved_read_share_by_species.csv")

_spread = float(assignment_by_species["Mean"].max() - assignment_by_species["Mean"].min())
if _spread >= 0.20:
    _worst = assignment_by_species.index[0]
    print(f"\n[CONFOUND WARNING] The share of unassignable reads differs by "
          f"{_spread:.0%} between host species ({_worst} is highest at "
          f"{assignment_by_species['Mean'].iloc[0]:.0%}, against "
          f"{assignment_by_species['Mean'].iloc[-1]:.0%} for "
          f"{assignment_by_species.index[-1]}). Every diversity metric in this notebook is computed "
          f"on confirmed reads only, so these species are NOT being compared on equivalent fractions "
          f"of their communities. This is a property of the reference database's coverage of each "
          f"host's microbiota, not something the statistics can correct. It must be reported in the "
          f"Methods, and any species-level result involving {_worst} must be stated with this "
          f"limitation attached.")
elif _spread >= 0.10:
    print(f"\n[NOTE] The share of unassignable reads varies by {_spread:.0%} between host species; "
          f"worth a line in the Methods, but not large enough to undermine between-species comparison.")
print("OK -- read accounting verified: curated + excluded/unresolved reads == original reads, every sample.")

In [ ]:
# Table · 02.5 — Taxonomic assignment success by host species
show_table(assignment_by_species, "Taxonomic assignment success by host species, from community_matrix_pre_curation. A high unresolved share means that species' diversity is computed from a small part of its reads, which is a limitation to state rather than a fault to fix.", index=True)

In [ ]:
# 02.6 — Drop all-zero genera and apply the low-support filter to the curated matrix
# ----------------------------------------------------------------------
def drop_zero_sum_taxa(matrix: pd.DataFrame, stage_label: str = "", verbose: bool | None = None) -> tuple[pd.DataFrame, int]:
    """Drop taxa that are zero across every sample; return (matrix, n_dropped).

    Checkpoint #1 is right after curation (here); checkpoint #2 is after
    each rarefaction replicate (section 05.0), since rarefaction can
    independently zero out a low-abundance taxon by chance.
    """
    verbose = VERBOSE_TAXA_LOG if verbose is None else verbose
    totals = matrix.sum(axis=0)
    zero_cols = totals[totals == 0].index.tolist()
    if zero_cols and verbose:
        label = f"[{stage_label}] " if stage_label else ""
        print(f"{label}Dropping {len(zero_cols)} all-zero taxa: {zero_cols[:10]}{'...' if len(zero_cols) > 10 else ''}")
    return matrix.drop(columns=zero_cols), len(zero_cols)


def filter_low_support_genera(community_matrix: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    prevalence = (community_matrix > 0).sum(axis=0)
    total_reads = community_matrix.sum(axis=0)
    keep = (prevalence >= MIN_SAMPLE_PREVALENCE) & (total_reads >= MIN_TOTAL_COUNT)
    filtered = community_matrix.loc[:, keep].copy()
    log = {
        "applied": True, "n_genera_before": int(community_matrix.shape[1]),
        "n_genera_after": int(filtered.shape[1]), "n_genera_removed": int((~keep).sum()),
        "min_sample_prevalence": MIN_SAMPLE_PREVALENCE, "min_total_count": MIN_TOTAL_COUNT,
    }
    return filtered, log


curated_matrix_checkpoint1, n_zero_curation = drop_zero_sum_taxa(curated_matrix_unfiltered, "post-curation", verbose=True)
if APPLY_LOW_SUPPORT_FILTER:
    community_matrix_curated, filter_log = filter_low_support_genera(curated_matrix_checkpoint1)
else:
    community_matrix_curated, filter_log = curated_matrix_checkpoint1, {"applied": False}

print("02.6 verification")
print("-" * 60)
print(f"After zero-sum checkpoint #1: {curated_matrix_checkpoint1.shape[1]} taxa ({n_zero_curation} all-zero taxa dropped)")
if filter_log.get("applied"):
    print(f"Low-support filter: {filter_log['n_genera_before']} -> {filter_log['n_genera_after']} genera "
          f"(removed {filter_log['n_genera_removed']}, prevalence >= {filter_log['min_sample_prevalence']}, "
          f"total count >= {filter_log['min_total_count']})")
assert not community_matrix_curated.isna().any().any()
assert (community_matrix_curated >= 0).all().all()
assert not community_matrix_curated.columns.duplicated().any()
assert (community_matrix_curated.sum(axis=0) > 0).all(), "A zero-sum taxon survived the checkpoint."
print(f"OK -- final curated genus matrix: {community_matrix_curated.shape[0]} samples x {community_matrix_curated.shape[1]} confirmed genera")

In [ ]:
# 02.7 — community_matrix_analysis: the curated matrix every later section works on
# ----------------------------------------------------------------------
# `community_matrix_analysis` is the SINGLE matrix every later section
# reads from -- never an undefined variable or a per-analysis rebuild.
analysis_metadata = analysis_metadata_raw.loc[community_matrix_curated.index].copy()
community_matrix_analysis = community_matrix_curated.loc[analysis_metadata.index].copy()

curated_matrix_path = PROCESSED_DATA_DIR / "community_matrix_curated.csv"
correspondence_path = TAXONOMY_RESULTS_DIR / "taxonomic_correspondence_table.csv"
excluded_reads_path = QC_REPORTS_DIR / "excluded_reads_summary.csv"
analysis_metadata_path = PROCESSED_DATA_DIR / "analysis_metadata.csv"

community_matrix_analysis.to_csv(curated_matrix_path)
analysis_metadata.to_csv(analysis_metadata_path)
correspondence.to_csv(correspondence_path, index=False)

excluded_summary = pd.DataFrame({
    "SampleID": community_matrix_pre_curation.index,
    "Total_reads": community_matrix_pre_curation.sum(axis=1).values,
    "Excluded_or_unresolved_reads": excluded_reads.reindex(community_matrix_pre_curation.index).values,
}).set_index("SampleID")
excluded_summary["Excluded_or_unresolved_share"] = (
    excluded_summary["Excluded_or_unresolved_reads"] / excluded_summary["Total_reads"]
)
excluded_summary.to_csv(excluded_reads_path)

with open(QC_REPORTS_DIR / "taxonomic_curation_log.json", "w") as _fh:
    json.dump({
        "n_original_labels": int(community_matrix_pre_curation.shape[1]),
        "n_confirmed_genus_taxa": int(community_matrix_analysis.shape[1]),
        "decision_counts": {k: int(v) for k, v in correspondence["Decision"].value_counts().items()},
        "reference_file_used": str(TAXONOMIC_REFERENCE_PATH) if taxonomic_reference is not None else None,
        "low_support_filter": filter_log,
    }, _fh, indent=2, default=str)

print("02.7 verification")
print("-" * 60)
print(f"Samples    : {community_matrix_analysis.shape[0]}")
print(f"Categories : {community_matrix_analysis.shape[1]} (confirmed genus-level taxa)")
assert community_matrix_analysis.index.equals(analysis_metadata.index)
assert community_matrix_analysis.index.is_unique and analysis_metadata.index.is_unique
assert not community_matrix_analysis.isna().any().any()
assert community_matrix_analysis.values.min() >= 0
for _p in [curated_matrix_path, analysis_metadata_path, correspondence_path, excluded_reads_path]:
    assert _p.exists()
print(f"\nSaved matrix (final analytical dataset) -> {curated_matrix_path}")
print(f"Saved metadata                          -> {analysis_metadata_path}")
print(f"Saved correspondence table              -> {correspondence_path}")
print(f"Saved excluded/unresolved read share    -> {excluded_reads_path}")
print("\nSamples per species (final analytical dataset):")
print(analysis_metadata["Species_code"].value_counts().reindex(SPECIES_ORDER).to_string())
print("\nOK -- section 02.0 complete.")

## 03.0 — Sequencing depth diagnostics

Library sizes are summarized from `community_matrix_analysis`, so that every depth statistic
quoted here refers to the same confirmed genus-level reads that the diversity analyses will use,
not to the raw output of the sequencer.

Depth matters for two reasons. It varies over three orders of magnitude between samples, which
makes raw diversity counts incomparable until depth is equalised; and it varies systematically
between host species, so any comparison that ignores it risks reporting sequencing effort as
biology. This section quantifies both, and section 04.0 decides what to do about them.

In [ ]:
# 03.1 — Library size summary from community_matrix_analysis, overall and by host species
# ----------------------------------------------------------------------
sequencing_depth = community_matrix_analysis.sum(axis=1).rename("Sequencing_depth")
depth_overall_summary = sequencing_depth.describe()
depth_overall_summary.to_csv(STATISTICS_DIR / "sequencing_depth_overall_summary.csv")

depth_by_species = sequencing_depth.to_frame().join(analysis_metadata["Species_code"])
depth_by_species_summary = (
    depth_by_species.groupby("Species_code")["Sequencing_depth"]
    .agg(["count", "mean", "std", "median", "min", "max"]).reindex(SPECIES_ORDER)
)
depth_by_species_summary.to_csv(STATISTICS_DIR / "sequencing_depth_by_species.csv")

print("03.1 verification")
print("-" * 60)
print("Sequencing depth (confirmed genus-level reads per sample), all species pooled:")
print(depth_overall_summary.apply(lambda v: f"{v:,.0f}").to_string())
print("\nSequencing depth by species:")
print(depth_by_species_summary.round(0).astype("Int64").to_string())
print(f"\nDepth spans {sequencing_depth.max() / sequencing_depth.min():,.0f}-fold "
      f"between the shallowest and deepest sample -- this is what section 04.0 has to reconcile.")
assert (sequencing_depth > 0).all(), "A sample has zero confirmed genus-level reads."
assert depth_by_species_summary["count"].sum() == len(analysis_metadata)
print("OK -- every sample retains a positive number of confirmed genus-level reads.")

In [ ]:
# 03.2 — Figure: library size per bird, by host species
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.5, 5.6))
plot_order = [s for s in SPECIES_ORDER if s in set(depth_by_species["Species_code"])]

for i, s in enumerate(plot_order, start=1):
    vals = depth_by_species.loc[depth_by_species["Species_code"] == s, "Sequencing_depth"].values
    bp = ax.boxplot([vals], positions=[i], widths=0.55, showfliers=False, patch_artist=True,
                    medianprops=dict(color="black", linewidth=1.6),
                    whiskerprops=dict(color="0.4"), capprops=dict(color="0.4"),
                    boxprops=dict(facecolor=SPECIES_COLORS[s], alpha=0.30, edgecolor=SPECIES_COLORS[s], linewidth=1.3))
    jitter = spawn_rng().normal(0, 0.06, size=len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals, s=34, zorder=3,
               marker=SPECIES_MARKERS[s], color=SPECIES_COLORS[s],
               edgecolor="white", linewidth=0.7, alpha=0.95)
    ax.text(i, ax.get_ylim()[1], "", ha="center")

ax.set_yscale("log")
ax.set_xticks(range(1, len(plot_order) + 1))
ax.set_xticklabels([f"{abbreviated_binomial(s)}\n({SPECIES_ORIGIN[s].lower()})" for s in plot_order])
ax.set_ylabel("Sequencing depth (confirmed genus-level reads per sample, log scale)")
ax.set_xlabel("")
ax.set_title(f"Sequencing depth per sample, by host species "
             f"({sequencing_depth.max() / sequencing_depth.min():,.0f}-fold range)", pad=26)
ax.text(0.5, 1.045, "Each point is one bird; box = median and interquartile range",
        transform=ax.transAxes, ha="center", va="bottom", fontsize=9.5, color="0.35")

# Annotate n per species directly on the figure.
for i, s in enumerate(plot_order, start=1):
    n = int((depth_by_species["Species_code"] == s).sum())
    ax.annotate(f"n = {n}", xy=(i, 1.0), xycoords=("data", "axes fraction"),
                xytext=(0, -12), textcoords="offset points", ha="center", fontsize=9, color="0.3")

ax.axhline(sequencing_depth.median(), color="0.35", linestyle="--", linewidth=1.1, zorder=0)
ax.annotate(f"overall median = {sequencing_depth.median():,.0f} reads",
            xy=(0.01, sequencing_depth.median()), xycoords=("axes fraction", "data"),
            xytext=(0, 5), textcoords="offset points", ha="left", fontsize=8.5, color="0.35")
# Moved outside the axes: inside, it sat on top of the low-depth points
# in the bottom-right corner, hiding exactly the libraries the figure is
# about. The x-axis is already labeled with the species, so this legend
# only carries the invasive/native color coding.
ax.legend(handles=[
    Patch(facecolor=SPECIES_COLORS[s], alpha=0.45, edgecolor=SPECIES_COLORS[s],
          label=species_legend_label(s)) for s in plot_order],
    loc="upper left", bbox_to_anchor=(1.01, 1.0), ncol=1,
    title="Species (origin)", title_fontsize=9, fontsize=8.5, borderaxespad=0)

# The quality floor belongs on this figure: it is the line that decides
# which libraries are analyzed at all, and it is set in section 04.3.
ax.axhline(MIN_LIBRARY_DEPTH, color="#B00020", linestyle="-.", linewidth=1.2, zorder=0)
ax.annotate(f"failed-library floor = {MIN_LIBRARY_DEPTH:,} reads",
            xy=(0.01, MIN_LIBRARY_DEPTH), xycoords=("axes fraction", "data"),
            xytext=(0, -12), textcoords="offset points", ha="left", fontsize=8.5, color="#B00020")

_below_floor = int((sequencing_depth < MIN_LIBRARY_DEPTH).sum())
figure_caption(fig,
    f"Sequencing depth after taxonomic curation, by species (N = {len(sequencing_depth)} cloacal samples). "
    f"Log scale. Range {sequencing_depth.min():,.0f}-{sequencing_depth.max():,.0f} reads. "
    f"The red line is the failed-library floor applied in section 04.4; {_below_floor} librar(ies) fall "
    f"below it and are excluded from every rarefaction-based analysis. Median depth differs markedly "
    f"between host species, which is why depth is equalised by rarefaction before any diversity metric "
    f"is compared. The spread shown here is what forces the rarefaction-depth trade-off analyzed in "
    f"section 04.0.")
plt.tight_layout()
depth_fig_path = save_figure(fig, QUALITY_DIR / "sequencing_depth_by_species.png")
plt.show(); plt.close(fig)

print("03.2 verification")
print("-" * 60)
assert depth_fig_path.exists()
print(f"Saved: {depth_fig_path}")
print("OK -- section 03.0 complete.")

## 04.0 — Rarefaction depth selection

Diversity estimated from a deeper library is larger simply because more of the community has been
sampled, so depth has to be equalised before species can be compared. Rarefaction does that by
subsampling every library to a common depth. The question this section answers is which depth, and
it answers it with a stated policy rather than a convenient round number.

### The trade-off

A deeper rarefaction describes each community better but discards every library that cannot reach
it; a shallower one keeps more birds but describes each of them from a smaller and more truncated
sample of its community. The cost and the benefit are therefore tabulated for every candidate
depth on a log-spaced grid spanning the full range of observed library sizes, and the choice is
made from that table.

### The policy

Three parameters, all fixed in section 02.2 before any depth is evaluated:

1. **`MIN_LIBRARY_DEPTH`** — a library below this has failed, and is excluded from
   rarefaction-based analyses and listed in `results/qc_reports/failed_libraries.csv`. One
   unusually shallow library should not set the depth for the whole study.
2. **`TARGET_SAMPLE_RETENTION`** — among the candidates, the deepest depth that still retains at
   least this share of the libraries that passed the floor.
3. **`MIN_SPECIES_N_RETAINED`** — no host species may be left with fewer than this many birds.
   This is an absolute count rather than a percentage, because on the smallest species a
   percentage threshold resolves to a whole number of birds and the rule would then be set by
   rounding rather than by design. The floor is the same `MIN_N_PER_LEVEL` a species needs to
   enter a model at all.

### Two coverage statistics, and which one governs

Coverage is reported two ways, and on these data they disagree: median **Good's coverage** reaches
1.000 at every candidate depth, while at the shallowest depths only a small minority of libraries
reach 90% of their fitted **Michaelis–Menten asymptote**. The disagreement is informative rather
than contradictory. Good's coverage is abundance-weighted — the estimated share of a community's
*individuals* that belong to an already-observed genus — so in communities dominated by one or two
genera it saturates almost immediately and says very little about the rare ones. Richness *counts
taxa*. The asymptotic statistic is therefore the one that decides whether a depth measures
richness, and the chosen depth is checked against `MIN_ADEQUATE_ASYMPTOTE_PCT` explicitly.

Failing that check would not invalidate the analysis: rarefied richness at a fixed depth is
Hurlbert's E[S*n*] and is strictly comparable between samples at any depth. What it changes is the
wording. A depth-truncated richness is reported as "genus richness at *N* reads", and differences
between species are not described as differences in total richness. The notebook sets
`RICHNESS_IS_DEPTH_TRUNCATED` from the diagnostic and the figures take their axis labels from it,
so the wording cannot drift away from the evidence.

The full trade-off across candidate depths is in
`results/statistics/rarefaction_depth_candidates.csv` and in the figure produced by 04.5. Section
06.13 then repeats the main between-species test across a span of depths, so the conclusions can
be shown not to depend on the point chosen here.

In [ ]:
# 04.1 — Rarefaction and alpha-diversity functions (analytical curves, subsampling, metrics)
# ----------------------------------------------------------------------
# Shared, testable functions used by sections 04, 05 and 08.

# -- Analytical rarefaction (Colwell/Heck hypergeometric expectation; no
# randomness, used only to CHOOSE a depth) --
def expected_rarefied_richness(counts: np.ndarray, depth: int) -> float:
    """Expected observed genus richness at `depth` reads (Heck et al. 1975)."""
    counts = np.asarray(counts, dtype=np.int64)
    total_reads = counts.sum()
    if depth > total_reads:
        return np.nan
    positive_counts = counts[counts > 0]
    return float(np.sum(1 - hypergeom.pmf(0, total_reads, positive_counts, depth)))


def expected_singletons(counts: np.ndarray, depth: int) -> float:
    """Expected number of genera seen exactly once in a subsample of `depth` reads."""
    counts = np.asarray(counts, dtype=np.int64)
    total_reads = counts.sum()
    if depth > total_reads:
        return np.nan
    positive_counts = counts[counts > 0]
    return float(np.sum(hypergeom.pmf(1, total_reads, positive_counts, depth)))


def goods_coverage(counts: np.ndarray, depth: int) -> float:
    """Good's coverage estimator C = 1 - f1/m at subsampling depth m.

    The share of the community's reads that belong to genera already
    observed. Unlike coverage against a fitted asymptote, this is
    computed directly from the data and needs no curve-fitting
    assumption, so it is the primary coverage number reported here.
    """
    f1 = expected_singletons(counts, depth)
    return np.nan if not np.isfinite(f1) else float(1.0 - f1 / depth)


def michaelis_menten(depth_k: np.ndarray, s_max: float, b: float) -> np.ndarray:
    """Asymptotic saturation curve; depth_k in thousands of reads, s_max = asymptotic richness."""
    return s_max * depth_k / (b + depth_k)


def fit_asymptotic_curve(depths: np.ndarray, richness: np.ndarray) -> dict:
    """Fit Michaelis-Menten to one sample's rarefaction curve; NaNs (never raises) if it fails."""
    x = np.asarray(depths, dtype=float) / 1000.0
    y = np.asarray(richness, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 3:
        return {"Smax": np.nan, "B": np.nan, "R2": np.nan}
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            params, _ = curve_fit(
                michaelis_menten, x, y, p0=[y.max() * 1.2, float(np.median(x))],
                bounds=([y.max(), 0], [np.inf, np.inf]), maxfev=20000,
            )
        s_max, b = params
        fitted = michaelis_menten(x, s_max, b)
        ss_res, ss_tot = np.sum((y - fitted) ** 2), np.sum((y - y.mean()) ** 2)
        return {"Smax": float(s_max), "B": float(b), "R2": float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan}
    except (RuntimeError, ValueError):
        return {"Smax": np.nan, "B": np.nan, "R2": np.nan}


# -- Stochastic rarefaction, used to build the matrix diversity is computed from --
def rarefy_counts(counts: np.ndarray, depth: int, rng: np.random.Generator) -> np.ndarray:
    """Subsample to exactly `depth` reads without replacement (exact multivariate hypergeometric)."""
    counts = np.asarray(counts, dtype=np.int64)
    total_reads = int(counts.sum())
    if total_reads < depth:
        raise ValueError(f"Sample has {total_reads:,} reads, below the requested depth of {depth:,}.")
    return rng.multivariate_hypergeometric(counts, depth)


def rarefy_matrix(community_matrix: pd.DataFrame, depth: int, rng: np.random.Generator) -> pd.DataFrame:
    """Rarefy every row to `depth` reads, advancing a single Generator (never re-seeded per sample)."""
    rarefied = {sid: rarefy_counts(row.values, depth, rng) for sid, row in community_matrix.iterrows()}
    out = pd.DataFrame(rarefied, index=community_matrix.columns).T.astype(np.int64)
    out.index.name = community_matrix.index.name
    return out


def rarefy_matrix_multiple(community_matrix: pd.DataFrame, depth: int, n_replicates: int, spawn_rng_fn) -> list[pd.DataFrame]:
    """`n_replicates` independent rarefactions, each from its own RNG stream."""
    return [rarefy_matrix(community_matrix, depth, spawn_rng_fn()) for _ in range(n_replicates)]


# -- Alpha-diversity metrics --
def calculate_alpha_diversity(counts: np.ndarray) -> dict:
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total == 0:
        return {"Richness": np.nan, "Shannon": np.nan, "Simpson": np.nan, "Pielou": np.nan}
    proportions = counts / total
    proportions = proportions[proportions > 0]
    richness = len(proportions)
    shannon = float(-np.sum(proportions * np.log(proportions)))
    return {
        "Richness": richness, "Shannon": shannon,
        "Simpson": float(1 - np.sum(proportions ** 2)),
        # Pielou is undefined for a one-genus community (log(1) = 0).
        "Pielou": float(shannon / np.log(richness)) if richness > 1 else np.nan,
    }


def alpha_diversity_table(matrix: pd.DataFrame) -> pd.DataFrame:
    records = []
    for sample_id, counts in matrix.iterrows():
        metrics = calculate_alpha_diversity(counts.values)
        metrics["SampleID"] = sample_id
        records.append(metrics)
    return pd.DataFrame(records).set_index("SampleID")[["Richness", "Shannon", "Simpson", "Pielou"]]


def alpha_diversity_over_replicates(replicate_matrices: list[pd.DataFrame]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Mean of each metric across replicates + its coefficient of variation across replicates.

    The CV is the direct, reportable answer to "how much do the results
    change between different rarefactions?" -- the diagnostic that
    exposed the previous depth choice as unusable.
    """
    per_replicate = [alpha_diversity_table(m) for m in replicate_matrices]
    stacked = pd.concat(per_replicate, keys=range(len(per_replicate)), names=["Replicate"])
    point_estimates = stacked.groupby("SampleID").mean()
    sd = stacked.groupby("SampleID").std()
    n_valid = stacked.groupby("SampleID").count()
    uncertainty = (sd / point_estimates.replace(0, np.nan)).abs()
    uncertainty.columns = [f"{c}_CV_across_replicates" for c in uncertainty.columns]
    n_valid.columns = [f"{c}_n_replicates" for c in n_valid.columns]
    return point_estimates, uncertainty.join(n_valid)


print("04.1 verification")
print("-" * 60)
_toy = np.array([50, 30, 15, 4, 1])
assert 0 < expected_rarefied_richness(_toy, 10) <= 5
assert 0 < goods_coverage(_toy, 10) <= 1
assert np.isnan(expected_rarefied_richness(_toy, 1000))
assert calculate_alpha_diversity(np.array([10, 0, 0]))["Richness"] == 1
assert np.isnan(calculate_alpha_diversity(np.array([10, 0, 0]))["Pielou"])
print("Self-tests passed (expected richness, Good's coverage, Pielou undefined at richness 1).")
print("OK -- rarefaction and alpha-diversity helper functions defined.")

In [ ]:
# 04.2 — Candidate depths, per-library rarefaction curves and Michaelis-Menten asymptotes
# ----------------------------------------------------------------------
def build_candidate_depths(sample_depths: pd.Series, min_depth: int, n_candidates: int = 24) -> np.ndarray:
    """Log-spaced candidate depths spanning the observed range of library sizes.

    The grid runs from the quality floor to the deepest library, so the
    retention/coverage trade-off is evaluated over the whole range the
    data can support rather than over an arbitrary window. Log spacing
    puts the candidates where the trade-off actually changes: retention
    falls and coverage rises roughly geometrically with depth.
    """
    top = float(sample_depths.max())
    if top <= min_depth:
        raise ValueError(
            f"No sample reaches MIN_LIBRARY_DEPTH = {min_depth:,} reads (deepest = {top:,.0f})."
        )
    grid = np.unique(np.round(np.geomspace(min_depth, top, n_candidates)).astype(int))
    return grid


def compute_rarefaction_curves(community_matrix: pd.DataFrame, candidate_depths: np.ndarray) -> pd.DataFrame:
    rows = []
    for sample_id, counts in community_matrix.iterrows():
        values = counts.values
        total_reads = values.sum()
        for depth in candidate_depths:
            if depth > total_reads:
                continue
            rows.append({
                "SampleID": sample_id, "Depth": int(depth),
                "Observed_genus_richness": expected_rarefied_richness(values, int(depth)),
                "Goods_coverage": goods_coverage(values, int(depth)),
            })
    return pd.DataFrame(rows)


def fit_all_asymptotic_curves(rarefaction_curves: pd.DataFrame) -> pd.DataFrame:
    results = []
    for sample_id, sample_curve in rarefaction_curves.groupby("SampleID"):
        fit = fit_asymptotic_curve(sample_curve["Depth"].values, sample_curve["Observed_genus_richness"].values)
        fit["SampleID"] = sample_id
        fit["N_depths_fitted"] = int(len(sample_curve))
        results.append(fit)
    return pd.DataFrame(results)[["SampleID", "Smax", "B", "R2", "N_depths_fitted"]]


print("04.2 verification")
print("-" * 60)
assert callable(build_candidate_depths) and callable(compute_rarefaction_curves) and callable(fit_all_asymptotic_curves)
print("OK -- candidate-grid, rarefaction-curve and curve-fitting functions defined.")

In [ ]:
# 04.3 — Depth selection rule and the coverage-adequacy check
# ----------------------------------------------------------------------
# --------------------------- POLICY PARAMETERS ------------------------
# Both are pre-specified here, BEFORE any depth is chosen or any result
# is seen, and both are reported in the Methods.
#
# MIN_LIBRARY_DEPTH: a library below this is a failed library. It is
#   excluded from rarefaction-based analyses and listed in
#   results/qc_reports/failed_libraries.csv, instead of being allowed to
#   drag the rarefaction depth for all 140 samples down to its own size.
#   10,000 reads is a conventional, commonly used floor for 16S amplicon
#   libraries and is far below this study's median library.
#
# TARGET_SAMPLE_RETENTION: of the libraries that pass that floor, the
#   deepest candidate depth that still retains this fraction is chosen.
#   0.95 accepts losing at most 1 sample in 20 in exchange for a depth
#   that is orders of magnitude more informative.
#
# MIN_SPECIES_N_RETAINED: the deepest-depth rule above is a rule about the
#   WHOLE dataset, and on an unbalanced design it can be satisfied while
#   taking most of its losses out of the smallest species -- here
#   Myiopsitta monachus, with the fewest individuals and the shallowest
#   libraries. A depth is therefore only admissible if EVERY species still
#   has at least this many samples afterwards.
#
#   Stated as an absolute count rather than a share. On the smallest
#   species a percentage threshold resolves to a whole number of birds --
#   with 7 QC-passing libraries, one bird is 14.3% and two are 28.6% --
#   so a percentage rule would be set by rounding rather than by design,
#   and the depth for the whole study would follow from it. The count
#   states the real requirement: a species needs enough birds to enter a
#   model at all, which is MIN_N_PER_LEVEL, and below that it is
#   descriptive-only whatever the depth.
#
# MIN_ADEQUATE_ASYMPTOTE_PCT: retention rules alone cannot tell you whether
#   the depth MEASURES anything. This is not a constraint on the choice
#   (that could make the problem infeasible); it is the adequacy check the
#   chosen depth has to pass, and failing it forces richness to be
#   reported as "richness at N reads" rather than as richness.
#
# MIN_ASYMPTOTE_FIT_R2: the asymptotic-coverage statistic is only as good
#   as the Michaelis-Menten fit it comes from. Libraries whose curve fits
#   worse than this are dropped from the adequacy PERCENTAGE (and counted
#   separately) rather than being quietly averaged in as if their
#   asymptote were known.
#
# The values themselves are set in section 02.2, before any of the data
# has been inspected. They are re-printed by the verification block below,
# next to the trade-off table they resolve.
# ----------------------------------------------------------------------


def summarize_candidate_depths(sample_depths: pd.Series, curves: pd.DataFrame,
                                asymptotic_fits: pd.DataFrame, candidate_depths: np.ndarray,
                                species_of_sample: pd.Series) -> pd.DataFrame:
    """Retention cost and coverage gain for every candidate depth."""
    # Only well-fitted curves contribute an asymptotic-coverage estimate:
    # a Michaelis-Menten fit with a poor R2 has an asymptote that is not
    # known, and averaging it in would report a precision that does not
    # exist. The count of libraries that DO contribute is carried in the
    # table so the percentage can never be read without its denominator.
    fits = asymptotic_fits.dropna(subset=["Smax"])
    fits = fits[fits["R2"] >= MIN_ASYMPTOTE_FIT_R2].set_index("SampleID")
    n_samples = len(sample_depths)
    species_totals = species_of_sample.loc[sample_depths.index].value_counts()
    rows = []
    for depth in candidate_depths:
        depth = int(depth)
        eligible = sample_depths.index[sample_depths >= depth]
        at_depth = curves.loc[curves["Depth"] == depth].set_index("SampleID")
        goods = at_depth.loc[at_depth.index.intersection(eligible), "Goods_coverage"]
        ids = fits.index.intersection(eligible)
        asym = (michaelis_menten(depth / 1000.0, fits.loc[ids, "Smax"], fits.loc[ids, "B"]) / fits.loc[ids, "Smax"]) if len(ids) else pd.Series(dtype=float)
        kept_per_species = species_of_sample.loc[eligible].value_counts().reindex(species_totals.index).fillna(0)
        worst_species_retention = float((kept_per_species / species_totals).min())
        rows.append({
            "Depth": depth,
            "N_samples_retained": int(len(eligible)),
            "Pct_samples_retained": 100.0 * len(eligible) / n_samples,
            "N_in_smallest_species": int(kept_per_species.min()),
            "Smallest_species": str(kept_per_species.idxmin()),
            "Pct_retained_in_worst_species": 100.0 * worst_species_retention,
            "Worst_species": str((kept_per_species / species_totals).idxmin()),
            "Median_Goods_coverage": float(goods.median()) if len(goods) else np.nan,
            "Pct_samples_Goods_coverage_ge_95": 100.0 * float((goods >= 0.95).mean()) if len(goods) else np.nan,
            "Mean_asymptotic_coverage": float(asym.mean()) if len(asym) else np.nan,
            "Pct_samples_reaching_90pct_asymptote": 100.0 * float((asym >= 0.90).mean()) if len(asym) else np.nan,
            "N_with_usable_asymptote_fit": int(len(asym)),
        })
    return pd.DataFrame(rows)


def select_rarefaction_depth(depth_summary: pd.DataFrame, target_retention: float,
                             min_species_n: int) -> tuple[int, str]:
    """Deepest candidate satisfying BOTH retention rules (overall %, and absolute n per species)."""
    feasible = depth_summary[
        (depth_summary["Pct_samples_retained"] >= 100.0 * target_retention)
        & (depth_summary["N_in_smallest_species"] >= min_species_n)
    ]
    if feasible.empty:
        raise ValueError(
            f"No candidate depth retains >= {100 * target_retention:.0f}% of samples overall AND "
            f">= {min_species_n} samples in every species. "
            f"Relax TARGET_SAMPLE_RETENTION / MIN_SPECIES_N_RETAINED, or raise MIN_LIBRARY_DEPTH."
        )
    chosen = feasible.loc[feasible["Depth"].idxmax()]
    justification = (
        f"Rarefaction depth = {int(chosen['Depth']):,} reads. Selected as the deepest candidate "
        f"retaining >= {100 * target_retention:.0f}% of the libraries that passed the "
        f"{MIN_LIBRARY_DEPTH:,}-read quality floor, while also leaving at least "
        f"{min_species_n} samples in every species "
        f"(smallest: {chosen['Smallest_species']}, n = {int(chosen['N_in_smallest_species'])}). "
        f"It keeps {int(chosen['N_samples_retained'])} samples ({chosen['Pct_samples_retained']:.1f}%), with a "
        f"median Good's coverage of {chosen['Median_Goods_coverage']:.3f} and "
        f"{chosen['Pct_samples_reaching_90pct_asymptote']:.1f}% of samples reaching >= 90% of their "
        f"fitted asymptotic richness. The full retention/coverage trade-off across all candidate "
        f"depths is in results/statistics/rarefaction_depth_candidates.csv and in Figure "
        f"'rarefaction_depth_selection.png'; section 06.13 repeats the main test across a span of depths."
    )
    return int(chosen["Depth"]), justification


def report_depth_adequacy(chosen_row: pd.Series) -> str:
    """Is the chosen depth deep enough to MEASURE richness, as opposed to merely compare it?

    Good's coverage and asymptotic coverage answer different questions and
    routinely disagree. Good's coverage is abundance-weighted -- it is the
    estimated share of the community's INDIVIDUALS that belong to a genus
    already seen, so it saturates as soon as the dominant genera are
    sampled and says almost nothing about rare ones. Asymptotic coverage
    is taxon-weighted. Because richness COUNTS taxa, the asymptotic
    statistic is the one that governs whether richness is being measured
    or merely truncated, and a median Good's coverage of 1.000 sitting
    next to 12% asymptotic coverage is not a contradiction: it is the
    expected reading of a community dominated by a few genera.
    """
    pct = float(chosen_row["Pct_samples_reaching_90pct_asymptote"])
    n_fit = int(chosen_row["N_with_usable_asymptote_fit"])
    if not np.isfinite(pct) or n_fit == 0:
        return ("[WARN] INCONCLUSIVE: no library has a usable Michaelis-Menten fit, so depth adequacy "
                "cannot be assessed from the asymptote. Treat richness as depth-truncated.")
    if pct >= MIN_ADEQUATE_ASYMPTOTE_PCT:
        return (f"ADEQUATE: {pct:.1f}% of the {n_fit} libraries with a usable curve fit reach >= 90% of "
                f"their fitted asymptotic richness at this depth (target >= "
                f"{MIN_ADEQUATE_ASYMPTOTE_PCT:.0f}%). Richness at this depth can be discussed as genus "
                f"richness.")
    return (f"[WARN] DEPTH-TRUNCATED: only {pct:.1f}% of the {n_fit} libraries with a usable curve fit "
            f"reach >= 90% of their fitted asymptotic richness (target >= "
            f"{MIN_ADEQUATE_ASYMPTOTE_PCT:.0f}%). Rarefied richness is still "
            f"a valid, strictly comparable statistic at a fixed depth -- it is Hurlbert's E[S_n] -- but it "
            f"estimates richness AT THIS DEPTH, not total richness, and at a shallow depth it is strongly "
            f"confounded with evenness: a bird whose community is dominated by one genus yields few genera "
            f"in a small subsample however many it actually carries. Report it as 'genus richness at "
            f"N reads' throughout, and do not describe between-species differences as differences in "
            f"total richness.")


print("04.3 verification")
print("-" * 60)
print(f"MIN_LIBRARY_DEPTH        = {MIN_LIBRARY_DEPTH:,} reads (failed-library floor)")
print(f"TARGET_SAMPLE_RETENTION  = {TARGET_SAMPLE_RETENTION:.0%} (of all QC-passing libraries)")
print(f"MIN_SPECIES_N_RETAINED   = {MIN_SPECIES_N_RETAINED} samples (absolute floor per species, "
      f"= MIN_N_PER_LEVEL)")
print(f"MIN_ADEQUATE_ASYMPTOTE_PCT = {MIN_ADEQUATE_ASYMPTOTE_PCT:.0f}% (adequacy check on the chosen depth)")
assert 0 < TARGET_SAMPLE_RETENTION <= 1 and MIN_LIBRARY_DEPTH > 0 and MIN_SPECIES_N_RETAINED >= 1
print("OK -- depth policy parameters pre-specified; selection functions defined.")

In [ ]:
# 04.4 — Apply the quality floor and select the rarefaction depth
# ----------------------------------------------------------------------
# Step 1: failed libraries, documented and excluded from rarefaction.
failed_libraries = sequencing_depth[sequencing_depth < MIN_LIBRARY_DEPTH].sort_values()
failed_library_report = (
    failed_libraries.to_frame("Sequencing_depth").join(analysis_metadata[["Species_code", "Locality"]])
)
failed_library_report.to_csv(QC_REPORTS_DIR / "failed_libraries.csv")

qc_passing_ids = sequencing_depth.index[sequencing_depth >= MIN_LIBRARY_DEPTH]
qc_passing_depths = sequencing_depth.loc[qc_passing_ids]

print("04.4 verification")
print("-" * 60)
print(f"Samples: {community_matrix_analysis.shape[0]}   Genera: {community_matrix_analysis.shape[1]}")
print(f"Sequencing depth: min={sequencing_depth.min():,.0f}  median={sequencing_depth.median():,.0f}  max={sequencing_depth.max():,.0f}")
print(f"\nFailed libraries (< {MIN_LIBRARY_DEPTH:,} reads): {len(failed_libraries)}")
if len(failed_libraries):
    print(failed_library_report.to_string())
    print(f"  -> excluded from rarefaction-based analyses; saved to {QC_REPORTS_DIR / 'failed_libraries.csv'}")
print(f"Libraries passing the quality floor: {len(qc_passing_ids)}/{len(sequencing_depth)}")

# Step 2: candidate depths and their trade-off.
candidate_depths = build_candidate_depths(qc_passing_depths, MIN_LIBRARY_DEPTH)
rarefaction_curves = compute_rarefaction_curves(community_matrix_analysis.loc[qc_passing_ids], candidate_depths)
asymptotic_fits = fit_all_asymptotic_curves(rarefaction_curves)
depth_summary = summarize_candidate_depths(qc_passing_depths, rarefaction_curves, asymptotic_fits,
                                           candidate_depths, analysis_metadata["Species_code"])

print(f"\nCandidate depths ({len(candidate_depths)} log-spaced values from {candidate_depths.min():,} "
      f"to {candidate_depths.max():,} reads).")

# Step 3: selection.
rarefaction_depth, depth_justification = select_rarefaction_depth(
    depth_summary, TARGET_SAMPLE_RETENTION, MIN_SPECIES_N_RETAINED)
rarefied_sample_ids = qc_passing_depths.index[qc_passing_depths >= rarefaction_depth].tolist()
dropped_for_depth = sorted(set(qc_passing_ids) - set(rarefied_sample_ids))

print(f"\n>>> Selected rarefaction depth: {rarefaction_depth:,} reads")
print(depth_justification)
print(f"\nSamples entering the rarefied analyses: {len(rarefied_sample_ids)}/{len(sequencing_depth)}")
if dropped_for_depth:
    print(f"  Dropped for insufficient depth (passed the QC floor but < {rarefaction_depth:,} reads): {dropped_for_depth}")
print("\nSamples per species after the depth filter:")
_after = analysis_metadata.loc[rarefied_sample_ids, "Species_code"].value_counts().reindex(SPECIES_ORDER).fillna(0).astype(int)
_before = analysis_metadata["Species_code"].value_counts().reindex(SPECIES_ORDER).fillna(0).astype(int)
print(pd.DataFrame({"Before": _before, "After": _after, "Lost": _before - _after}).to_string())

rarefaction_curves.to_csv(STATISTICS_DIR / "rarefaction_curves.csv", index=False)
asymptotic_fits.to_csv(STATISTICS_DIR / "rarefaction_asymptotic_fits.csv", index=False)
depth_summary.to_csv(STATISTICS_DIR / "rarefaction_depth_candidates.csv", index=False)
with open(STATISTICS_DIR / "rarefaction_depth_selection.txt", "w") as _fh:
    _fh.write(f"MIN_LIBRARY_DEPTH={MIN_LIBRARY_DEPTH}\nTARGET_SAMPLE_RETENTION={TARGET_SAMPLE_RETENTION}\n"
              f"MIN_SPECIES_N_RETAINED={MIN_SPECIES_N_RETAINED}\n"
              f"RAREFACTION_DEPTH={rarefaction_depth}\nN_SAMPLES_RAREFIED={len(rarefied_sample_ids)}\n\n{depth_justification}\n")

_chosen_row = depth_summary.set_index("Depth").loc[rarefaction_depth]
poor_fits = asymptotic_fits[~(asymptotic_fits["R2"] >= MIN_ASYMPTOTE_FIT_R2)]
if not poor_fits.empty:
    print(f"\n[NOTE] {len(poor_fits)}/{len(asymptotic_fits)} librar(ies) have a Michaelis-Menten fit with "
          f"R2 < {MIN_ASYMPTOTE_FIT_R2:.2f} (or no fit at all); their asymptote is not known, so they are "
          f"excluded from the adequacy percentage below, which is computed over the "
          f"{int(_chosen_row['N_with_usable_asymptote_fit'])} librar(ies) that do have a usable fit. "
          f"Good's coverage (median {_chosen_row['Median_Goods_coverage']:.3f}) needs no curve fit, but "
          f"being abundance-weighted it cannot stand in for this check.")

# The adequacy verdict, which decides how richness may be WORDED in the
# manuscript. RICHNESS_IS_DEPTH_TRUNCATED is read by the figures and by
# section 06.0 so the wording cannot drift away from the diagnostic.
depth_adequacy_note = report_depth_adequacy(_chosen_row)
# A missing adequacy statistic counts as NOT adequate: an unknown
# asymptote is a reason for the cautious wording, not a reason to skip it.
RICHNESS_IS_DEPTH_TRUNCATED = not (
    float(_chosen_row["N_with_usable_asymptote_fit"]) > 0
    and np.isfinite(_chosen_row["Pct_samples_reaching_90pct_asymptote"])
    and _chosen_row["Pct_samples_reaching_90pct_asymptote"] >= MIN_ADEQUATE_ASYMPTOTE_PCT)
RICHNESS_LABEL = (f"genus richness at {rarefaction_depth:,} reads" if RICHNESS_IS_DEPTH_TRUNCATED
                  else "genus richness")
print(f"\nDepth adequacy: {depth_adequacy_note}")
with open(STATISTICS_DIR / "rarefaction_depth_selection.txt", "a") as _fh:
    _fh.write(f"\nADEQUACY: {depth_adequacy_note}\n")

assert rarefaction_depth >= MIN_LIBRARY_DEPTH
assert len(rarefied_sample_ids) >= TARGET_SAMPLE_RETENTION * len(qc_passing_ids)
assert (_after.reindex(_before.index[_before > 0]).fillna(0) >= MIN_SPECIES_N_RETAINED).all(), \
    f"A species fell below {MIN_SPECIES_N_RETAINED} samples at the selected depth: {_after.to_dict()}"
assert (sequencing_depth.loc[rarefied_sample_ids] >= rarefaction_depth).all(), "Selected depth exceeds a retained sample's reads."
assert _after.sum() == len(rarefied_sample_ids)
assert (_after > 0).all(), "The depth filter removed an entire species -- lower TARGET_SAMPLE_RETENTION."
print(f"\nOK -- rarefaction_depth = {rarefaction_depth:,} reads on {len(rarefied_sample_ids)} samples; every species still represented.")

In [ ]:
# Table · 04.4 — Candidate rarefaction depths and what each one retains
show_table(depth_summary, "Candidate rarefaction depths evaluated on community_matrix_analysis: how many libraries and how many birds per species each depth retains, and the coverage it reaches. The selected depth is the deepest one meeting every pre-specified condition.")

In [ ]:
# 04.5 — Figure: the retention/coverage trade-off and the rarefaction curves
# ----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))

# -- Panel A: retention vs coverage across candidate depths --
ax = axes[0]
ax.plot(depth_summary["Depth"], depth_summary["Pct_samples_retained"],
        marker="o", markersize=4.5, color="#0072B2", linewidth=2, label="Samples retained (%)")
ax.plot(depth_summary["Depth"], 100 * depth_summary["Median_Goods_coverage"],
        marker="s", markersize=4.5, color="#D55E00", linewidth=2, label="Median Good's coverage (%)")
ax.plot(depth_summary["Depth"], depth_summary["Pct_samples_reaching_90pct_asymptote"],
        marker="^", markersize=4.5, color="#009E73", linewidth=2, linestyle="--",
        label="Samples reaching 90% of\nasymptotic richness (%)")
ax.axvline(rarefaction_depth, color="black", linewidth=1.4, linestyle=":")
ax.axhline(100 * TARGET_SAMPLE_RETENTION, color="#0072B2", linewidth=1, linestyle=":", alpha=0.6)
# Annotated against the RIGHT edge in axes coordinates: anchoring the text
# at the first data point put it hard against the spine, where the leading
# character was clipped.
ax.annotate(f"retention target {TARGET_SAMPLE_RETENTION:.0%}",
            xy=(0.99, 100 * TARGET_SAMPLE_RETENTION), xycoords=("axes fraction", "data"),
            xytext=(0, 5), textcoords="offset points", fontsize=8, color="#0072B2", ha="right")
ax.axhline(MIN_ADEQUATE_ASYMPTOTE_PCT, color="#009E73", linewidth=1, linestyle=":", alpha=0.6)
ax.annotate(f"richness-adequacy target {MIN_ADEQUATE_ASYMPTOTE_PCT:.0f}%",
            xy=(0.99, MIN_ADEQUATE_ASYMPTOTE_PCT), xycoords=("axes fraction", "data"),
            xytext=(0, -11), textcoords="offset points", fontsize=8, color="#009E73", ha="right")
ax.set_xscale("log")
ax.set_xlabel("Candidate rarefaction depth (reads, log scale)")
ax.set_ylabel("Percentage of libraries (%)")
ax.set_ylim(-3, 112)
ax.set_title(f"A. Depth is a trade-off: libraries kept vs. community captured\n"
             f"dotted line = selected depth ({rarefaction_depth:,} reads)", fontsize=11)
ax.legend(loc="lower left", fontsize=7.6, framealpha=0.92)

# A second y-axis for the absolute count that actually binds the choice.
# It is a count of birds, not a percentage, so it cannot share the left
# axis without being silently rescaled into something it is not.
ax_n = ax.twinx()
ax_n.plot(depth_summary["Depth"], depth_summary["N_in_smallest_species"],
          marker="v", markersize=4.5, color="#AA3377", linewidth=1.6, linestyle="-.")
ax_n.axhline(MIN_SPECIES_N_RETAINED, color="#AA3377", linewidth=1, linestyle=":", alpha=0.7)
ax_n.set_ylabel("Birds left in the smallest species (count)", color="#AA3377", fontsize=9)
ax_n.tick_params(axis="y", labelcolor="#AA3377", labelsize=8)
ax_n.set_ylim(bottom=0)
ax_n.grid(visible=False)
ax_n.annotate(f"floor = {MIN_SPECIES_N_RETAINED}", xy=(0.01, MIN_SPECIES_N_RETAINED),
              xycoords=("axes fraction", "data"), xytext=(0, 3), textcoords="offset points",
              fontsize=8, color="#AA3377")

# -- Panel B: rarefaction curves, all samples, colored by species --
ax = axes[1]
curve_species = rarefaction_curves.merge(
    analysis_metadata["Species_code"], left_on="SampleID", right_index=True, how="left")
# A species mean is drawn only where enough libraries are still deep
# enough to contribute to it. At the deepest candidates only one or two
# libraries survive per species, and a mean over those would move with
# the sample size rather than with the communities while being drawn at
# the same weight as the rest of the curve.
MIN_LIBRARIES_FOR_MEAN_CURVE = 5
for species in SPECIES_ORDER:
    sub = curve_species[curve_species["Species_code"] == species]
    if sub.empty:
        continue
    for _, sample_curve in sub.groupby("SampleID"):
        ax.plot(sample_curve["Depth"], sample_curve["Observed_genus_richness"],
                color=SPECIES_COLORS[species], alpha=0.30, linewidth=1.0)
    grouped = sub.groupby("Depth")["Observed_genus_richness"]
    mean_curve = grouped.mean()[grouped.count() >= MIN_LIBRARIES_FOR_MEAN_CURVE]
    ax.plot(mean_curve.index, mean_curve.values, color=SPECIES_COLORS[species], linewidth=2.8,
            marker=SPECIES_MARKERS[species], markersize=5, markevery=3,
            markeredgecolor="white", markeredgewidth=0.6,
            label=species_legend_label(species), zorder=4)
ax.axvline(rarefaction_depth, color="black", linewidth=1.4, linestyle=":")
ax.set_xscale("log")
ax.set_xlabel("Sequencing depth (reads, log scale)")
ax.set_ylabel(f"Expected genus richness\n(analytical rarefaction, E[S$_n$])")
ax.set_title(f"B. Expected genus richness against sequencing depth, per library\n"
             f"dotted line = selected depth ({rarefaction_depth:,} reads)", fontsize=11)
ax.legend(loc="upper left", fontsize=8, title="Species (origin)", title_fontsize=8,
          framealpha=0.92)

figure_caption(fig,
    f"Rarefaction-depth selection. A: for every candidate depth, the percentage of libraries retained "
    f"(blue), the median Good's coverage reached (orange) and the percentage of libraries reaching 90% of "
    f"their fitted asymptotic richness (green); the purple line, on the right-hand axis, is the number of "
    f"birds left in the smallest species, which is the constraint that binds. Good's coverage is "
    f"abundance-weighted and saturates as soon as the dominant genera are sampled, so it can sit at 100% "
    f"while most libraries are still far from their richness asymptote -- the green line, not the orange "
    f"one, is the criterion relevant to richness. The selected depth ({rarefaction_depth:,} reads, dotted "
    f"line) is the deepest candidate retaining >= {TARGET_SAMPLE_RETENTION:.0%} of the "
    f"{len(qc_passing_ids)} libraries above the {MIN_LIBRARY_DEPTH:,}-read quality floor AND leaving "
    f">= {MIN_SPECIES_N_RETAINED} birds in every species. "
    f"B: analytical (hypergeometric) rarefaction curve for every library, thin lines, colored by species; "
    f"thick lines are species means, drawn only where at least {MIN_LIBRARIES_FOR_MEAN_CURVE} libraries "
    f"still contribute. N = {len(rarefaction_curves['SampleID'].unique())} libraries.",
    y=-0.06)
plt.tight_layout()
rarefaction_fig_path = save_figure(fig, QUALITY_DIR / "rarefaction_depth_selection.png")
plt.show(); plt.close(fig)

print("04.5 verification")
print("-" * 60)
assert rarefaction_fig_path.exists()
print(f"Saved: {rarefaction_fig_path}")
print(f"Median Good's coverage at the selected depth        : {_chosen_row['Median_Goods_coverage']:.3f}")
print(f"Libraries reaching >= 90% of asymptotic richness     : "
      f"{_chosen_row['Pct_samples_reaching_90pct_asymptote']:.1f}%")
print(f"Richness will be reported as                         : '{RICHNESS_LABEL}'")
print("OK -- section 04.0 complete.")

## 05.0 — Alpha diversity

Builds the rarefied community matrix from `community_matrix_analysis` and computes four
alpha-diversity metrics on it: observed genus richness, Shannon diversity *H'*, Simpson diversity
(1 − *D*) and Pielou's evenness *J'*.

A single rarefaction is one random draw, and its result depends on which reads happen to be drawn.
Each metric is therefore averaged over `N_RAREFACTION_REPLICATES` independent rarefactions, and
the per-sample coefficient of variation across those replicates is reported as a diagnostic and
checked against `MAX_ACCEPTABLE_MEDIAN_CV`. If the point estimates moved appreciably between
replicates, everything built on them would inherit that noise as a loss of power, so this is
checked rather than assumed. Genera that a given replicate happens to reduce to zero are dropped
from that replicate before diversity is computed on it.

Three things are then established before any model is specified, because each of them decides how
the models in 06.0 and 07.0 must be built:

- **The support of each response**, which determines the candidate distribution family. Normality
  is deliberately not assessed on the raw per-species values — a model needs its *residuals* to
  behave, and those do not exist until a model is fitted.
- **Community dominance and metric redundancy.** When one genus holds most of a bird's reads,
  every alpha metric moves together, and four tests stop being four pieces of evidence. Both the
  dominance structure and the correlations between metrics are quantified here.
- **The sampling design.** Which Species × Locality and Species × Sex cells are populated well
  enough to support a term, and which host species are confined to a single locality and therefore
  cannot have their species effect fully separated from that site.

In [ ]:
# 05.1 — Replication parameters and the summary / design-check functions
# ----------------------------------------------------------------------
N_RAREFACTION_REPLICATES = 50
# If the median CV of a metric across replicates exceeds this, the
# rarefaction depth is too shallow for that metric to be modeled
# reliably, and that is reported rather than left for the reader to
# infer from wide confidence intervals later.
MAX_ACCEPTABLE_MEDIAN_CV = 0.15

# Four metrics are computed from each rarefied matrix, because they come
# from the same pass over the data and all four belong in the archived
# alpha-diversity table. Three of them are analyzed here.
#
# Shannon, Simpson and Pielou measure very nearly the same thing on these
# communities -- section 05.3 reports the correlations -- so modeling
# all three would present one finding three times and would make the
# multiplicity correction look more demanding than it is. Richness
# carries genuinely separate information and is kept; of the three
# evenness metrics, Shannon is the standard currency of the field and
# Simpson (1 - D, the probability that two reads come from different
# genera) states the dominance result most directly, so those two are
# modeled, while Pielou's evenness is computed and archived but not
# modeled. Anyone who wants it will find it in alpha_diversity_data.csv.
ALPHA_METRICS_COMPUTED = ["Richness", "Shannon", "Simpson", "Pielou"]
ALPHA_METRICS = ["Richness", "Shannon", "Simpson"]
# The metrics bounded in (0, 1), which are the ones modeled by beta
# regression. Derived from ALPHA_METRICS so that changing what is
# analyzed cannot leave a model fitted to a response nobody reports.
BETA_RESPONSE_METRICS = [m for m in ALPHA_METRICS if m in ("Simpson", "Pielou")]
# A bird whose single most abundant genus holds at least this share of its
# reads is called monodominant. 0.50 is a plain majority -- one genus
# outweighing everything else combined -- and is used only descriptively,
# never as a filter.
MONODOMINANCE_THRESHOLD = 0.50


def summarize_by_species(alpha_data: pd.DataFrame) -> pd.DataFrame:
    return (alpha_data.groupby("Species_code")[ALPHA_METRICS]
            .agg(["count", "mean", "std", "median", "min", "max", "skew"]).reindex(SPECIES_ORDER))


def metric_support_report(alpha_data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for metric in ALPHA_METRICS:
        values = alpha_data[metric].dropna()
        rows.append({
            "Metric": metric, "N": len(values), "Min": values.min(), "Max": values.max(),
            "N_at_or_below_0": int((values <= 0).sum()),
            "N_at_or_above_1": int((values >= 1).sum()) if metric in ("Simpson", "Pielou") else np.nan,
            "Is_integer_valued": bool(np.allclose(values, np.round(values))) if metric == "Richness" else np.nan,
            "Skewness": float(values.skew()),
        })
    return pd.DataFrame(rows)


def sampling_design_report(metadata: pd.DataFrame) -> dict[str, pd.DataFrame]:
    reports = {}
    reports["species_by_locality"] = pd.crosstab(metadata["Species_code"], metadata["Locality"])
    reports["species_by_sex"] = pd.crosstab(metadata["Species_code"], metadata["Sex"])
    reports["species_by_locality_by_sex"] = (
        metadata.groupby(["Species_code", "Locality", "Sex"], observed=True).size().rename("N").reset_index())

    # Localities with a single species cannot separate a Locality effect
    # from a Species effect in the inter-specific model.
    locality_species_counts = metadata.groupby("Locality")["Species_code"].nunique()
    reports["localities_single_species"] = locality_species_counts[locality_species_counts == 1].to_frame("n_species")

    # A species almost entirely confined to one locality leaves its
    # Species effect nearly aliased with that locality even after the
    # exclusion above. This is a property of the sampling design and has
    # to be stated when the species effect is interpreted.
    crosstab = reports["species_by_locality"]
    concentration = (crosstab.max(axis=1) / crosstab.sum(axis=1)).rename("Max_share_in_one_locality").to_frame()
    concentration["Dominant_locality"] = crosstab.idxmax(axis=1)
    concentration["N"] = crosstab.sum(axis=1)
    reports["species_locality_concentration"] = concentration.sort_values("Max_share_in_one_locality", ascending=False)

    within_species_locality_n = metadata.groupby(["Species_code", "Locality"], observed=True).size().rename("N").reset_index()
    reports["within_species_low_n_localities"] = within_species_locality_n[within_species_locality_n["N"] < MIN_N_PER_LEVEL]
    within_species_sex_n = metadata.groupby(["Species_code", "Sex"], observed=True).size().rename("N").reset_index()
    reports["within_species_low_n_sex_levels"] = within_species_sex_n[within_species_sex_n["N"] < MIN_N_PER_LEVEL]
    return reports


print("05.1 verification")
print("-" * 60)
print(f"N_RAREFACTION_REPLICATES = {N_RAREFACTION_REPLICATES}")
print(f"MAX_ACCEPTABLE_MEDIAN_CV = {MAX_ACCEPTABLE_MEDIAN_CV}")
assert callable(summarize_by_species) and callable(metric_support_report) and callable(sampling_design_report)
print("OK -- summary and design-check helpers defined.")

In [ ]:
# 05.2 — Rarefy community_matrix_analysis 50 times -> alpha_data, with replicate stability
# ----------------------------------------------------------------------
rarefaction_input = community_matrix_analysis.loc[rarefied_sample_ids]
print(f"Rarefying {rarefaction_input.shape[0]} samples to {rarefaction_depth:,} reads "
      f"x {N_RAREFACTION_REPLICATES} independent replicates ...")

replicate_matrices = rarefy_matrix_multiple(rarefaction_input, rarefaction_depth, N_RAREFACTION_REPLICATES, spawn_rng)

# Zero-sum-taxa checkpoint #2, per replicate (which genus drops out can
# differ between replicates). Counts are summarized, not printed 50 times.
cleaned, zero_counts = [], []
for i, m in enumerate(replicate_matrices):
    m_clean, n_dropped = drop_zero_sum_taxa(m, stage_label=f"post-rarefaction replicate {i}")
    cleaned.append(m_clean); zero_counts.append(n_dropped)
replicate_matrices = cleaned

# Canonical single realisation, reused by beta diversity (section 08.0).
rarefied_matrix = replicate_matrices[0]
rarefied_matrix.to_csv(PROCESSED_DATA_DIR / "rarefied_matrix.csv")

alpha_point_estimates, alpha_uncertainty = alpha_diversity_over_replicates(replicate_matrices)
alpha_diversity = alpha_point_estimates.join(alpha_uncertainty)
alpha_data = analysis_metadata.loc[alpha_diversity.index].join(alpha_diversity)
alpha_data.to_csv(PROCESSED_DATA_DIR / "alpha_diversity_data.csv")

# Stability is reported for every metric in the archived table, not only
# for the ones modeled here: it is a statement about the data.
cv_summary = pd.DataFrame({
    "Median_CV": [alpha_uncertainty[f"{m}_CV_across_replicates"].median() for m in ALPHA_METRICS_COMPUTED],
    "Max_CV": [alpha_uncertainty[f"{m}_CV_across_replicates"].max() for m in ALPHA_METRICS_COMPUTED],
    "Min_replicates_contributing": [int(alpha_uncertainty[f"{m}_n_replicates"].min())
                                    for m in ALPHA_METRICS_COMPUTED],
}, index=ALPHA_METRICS_COMPUTED)
cv_summary.to_csv(STATISTICS_DIR / "alpha_diversity_rarefaction_stability.csv")

print("\n05.2 verification")
print("-" * 60)
print(f"Rarefied matrix: {rarefied_matrix.shape[0]} samples x {rarefied_matrix.shape[1]} genera "
      f"(median {int(np.median(zero_counts))} all-zero genera dropped per replicate)")
print(f"Saved canonical rarefied matrix -> {PROCESSED_DATA_DIR / 'rarefied_matrix.csv'}")
print(f"\nRarefaction stability across {N_RAREFACTION_REPLICATES} replicates "
      f"(coefficient of variation per sample; lower is better).")

_unstable = cv_summary.index[cv_summary["Median_CV"] > MAX_ACCEPTABLE_MEDIAN_CV].tolist()
if _unstable:
    print(f"\n[WARN] Median CV above {MAX_ACCEPTABLE_MEDIAN_CV} for: {_unstable}. These metrics are "
          f"sensitive to which random subsample is drawn; raise the rarefaction depth or treat them "
          f"as descriptive only.")
else:
    print(f"\nAll four metrics have a median replicate-to-replicate CV below {MAX_ACCEPTABLE_MEDIAN_CV} "
          f"-- the point estimates carried into sections 06/07 are stable with respect to the random subsample.")

# Pielou is undefined when a rarefied sample contains a single genus; if
# that happened in some replicates the mean is over fewer than
# N_RAREFACTION_REPLICATES values, which must be visible, not silent.
_min_reps = int(alpha_uncertainty["Pielou_n_replicates"].min())
if _min_reps < N_RAREFACTION_REPLICATES:
    print(f"[NOTE] Pielou was undefined (single-genus rarefied community) in some replicates; "
          f"the worst-case sample averages over {_min_reps}/{N_RAREFACTION_REPLICATES} replicates.")

assert len(replicate_matrices) == N_RAREFACTION_REPLICATES
assert all((m.sum(axis=1) == rarefaction_depth).all() for m in replicate_matrices), "A replicate is not rarefied to the selected depth."
assert set(alpha_data.index) == set(rarefied_sample_ids)
assert alpha_data[ALPHA_METRICS_COMPUTED].notna().all().all(), "Unexpected missing alpha-diversity value."
print(f"\nSaved alpha-diversity + metadata table -> {PROCESSED_DATA_DIR / 'alpha_diversity_data.csv'}")
print(f"  It holds all {len(ALPHA_METRICS_COMPUTED)} metrics {ALPHA_METRICS_COMPUTED}; "
      f"this notebook models {ALPHA_METRICS}.")
print(f"OK -- alpha diversity computed for {len(alpha_data)} samples.")

In [ ]:
# Table · 05.2 — Replicate-to-replicate stability of each diversity measure
show_table(cv_summary, "Replicate-to-replicate stability of each metric across the {} rarefaction draws, from alpha_data: a low coefficient of variation means the value does not depend on which random subsample was taken.".format(N_RAREFACTION_REPLICATES))

In [ ]:
# Table · 05.2 — Per-bird alpha diversity, the table every model is fitted to
show_table(alpha_data[["Species_code", "Locality", "Sex"] + ALPHA_METRICS_COMPUTED],
           "Per-bird alpha diversity (alpha_diversity_data.csv): the mean over the rarefaction replicates for every bird, with its species, locality and sex. This is the table every model in sections 06.0 and 07.0 is fitted to.", index=True, page_length=15)

In [ ]:
# 05.3 — alpha_data: species summaries, response support, dominance, and the sampling design
# ----------------------------------------------------------------------
species_summary = summarize_by_species(alpha_data)
species_summary.to_csv(STATISTICS_DIR / "alpha_diversity_species_summary.csv")
print(f"Alpha diversity by species (mean +/- SD), at {rarefaction_depth:,} reads:")
for metric in ALPHA_METRICS:
    means = alpha_data.groupby("Species_code")[metric].mean().reindex(SPECIES_ORDER)
    stds = alpha_data.groupby("Species_code")[metric].std().reindex(SPECIES_ORDER)
    ns = alpha_data.groupby("Species_code")[metric].count().reindex(SPECIES_ORDER)
    print(f"  {metric:9s}: " + ", ".join(f"{s}={m:.2f}+/-{sd:.2f} (n={int(n)})"
                                          for s, m, sd, n in zip(SPECIES_ORDER, means, stds, ns)))

support = metric_support_report(alpha_data)
support.to_csv(STATISTICS_DIR / "alpha_diversity_metric_support.csv", index=False)
print("\nResponse support (decides the candidate model family in sections 06.0/07.0):")
print(support.round(4).to_string(index=False))
print("  Richness  -> counts, over-dispersed          => Negative Binomial (dispersion estimated)")
print("  Shannon   -> continuous, strictly positive   => Gamma GLM with a log link (06.5)")
print("  Simpson   -> strictly inside (0, 1)          => Beta regression")
print("  Pielou    -> strictly inside (0, 1)          => not modeled (see 06.0); archived only)")

# ----------------------------------------------------------------
# Dominance: what the four metrics are actually responding to.
#
# These communities are not evenly structured. When a bird's reads are
# overwhelmingly one genus, EVERY alpha metric moves together -- richness
# falls (few other genera survive a fixed-size subsample), Shannon,
# Simpson and Pielou fall by construction -- so the four metrics stop
# being four pieces of evidence and become four views of one quantity.
# That matters twice over: it is the biological finding (which hosts
# carry monodominant communities), and it is the reason four FDR-adjusted
# tests should not be read as four independent confirmations.
# ----------------------------------------------------------------
_rel = rarefied_matrix.div(rarefied_matrix.sum(axis=1), axis=0)
dominance = pd.DataFrame({
    "Dominant_genus": _rel.idxmax(axis=1),
    "Dominant_share": _rel.max(axis=1),
}).join(analysis_metadata["Species_code"])
dominance["Monodominant"] = dominance["Dominant_share"] >= MONODOMINANCE_THRESHOLD
dominance.to_csv(STATISTICS_DIR / "sample_dominance.csv")

dominance_by_species = (dominance.groupby("Species_code", observed=True)
                        .agg(N=("Dominant_share", "size"),
                             Median_dominant_share=("Dominant_share", "median"),
                             Pct_monodominant=("Monodominant", lambda v: 100.0 * v.mean()))
                        .reindex(SPECIES_ORDER).dropna(how="all"))
dominance_by_species["Most_common_dominant_genus"] = (
    dominance.groupby("Species_code", observed=True)["Dominant_genus"]
    .agg(lambda v: v.value_counts().idxmax() if len(v) else "")
    .reindex(dominance_by_species.index))
dominance_by_species.to_csv(STATISTICS_DIR / "dominance_by_species.csv")

print(f"\nCommunity dominance (share of reads held by each bird's single most abundant genus,")
print(f"at {rarefaction_depth:,} reads; 'monodominant' = >= {MONODOMINANCE_THRESHOLD:.0%}).")

_mono_overall = 100.0 * float(dominance["Monodominant"].mean())
print(f"Overall, {_mono_overall:.1f}% of birds carry a monodominant community.")
if dominance_by_species["Pct_monodominant"].max() - dominance_by_species["Pct_monodominant"].min() >= 30:
    _top = dominance_by_species["Pct_monodominant"].idxmax()
    print(f"[READING THE TABLE] Monodominance is strongly species-structured: {_top} reaches "
          f"{dominance_by_species.loc[_top, 'Pct_monodominant']:.0f}% of birds, usually "
          f"{dominance_by_species.loc[_top, 'Most_common_dominant_genus']}. Differences in the alpha "
          f"metrics between species are therefore, to a large extent, differences in HOW OFTEN a bird's "
          f"community is taken over by one genus -- which is a more specific and more interesting claim "
          f"than 'lower diversity', and is the one the Discussion should make.")

# How redundant are the four metrics on THIS dataset? Spearman, because
# the relationships are monotone but not linear.
metric_correlation = alpha_data[ALPHA_METRICS_COMPUTED].corr(method="spearman")
metric_correlation.to_csv(STATISTICS_DIR / "alpha_metric_correlations.csv")
print("\nSpearman correlation between the four alpha metrics (how much independent information they carry):")
print(metric_correlation.round(3).to_string())
_offdiag = metric_correlation.where(~np.eye(len(ALPHA_METRICS_COMPUTED), dtype=bool)).abs().stack()
print(f"Median |rho| between metrics: {_offdiag.median():.3f}  (max {_offdiag.max():.3f})")
if _offdiag.median() >= 0.70:
    print("[NOTE] The four metrics are strongly inter-correlated, so they are close to a single axis of "
          "variation. The FDR correction across them is still applied (Benjamini-Hochberg is valid under "
          "positive dependence), but four significant q-values here are ONE finding seen four ways, not "
          "four independent findings, and the Results should say so. Reporting richness plus one evenness "
          "metric in the main text, with the rest in a supplement, is the honest presentation.")

# ----------------------------------------------------------------
# Does taxonomic assignment success predict diversity?
#
# Diversity is computed on confirmed genus-level reads only. If the share
# of reads a host's community leaves unassigned varies between host
# species -- and section 02.5 shows it does -- then part of any apparent
# species difference could be a property of the reference database rather
# than of the birds. That is a testable proposition, not just a caveat,
# so it is tested here: if assignment success is unrelated to diversity,
# the concern is closed; if it is related, every species-level result has
# to be read with it in mind and section 06.14 quantifies the impact.
# ----------------------------------------------------------------
alpha_data["Unresolved_share"] = excluded_share_per_sample.reindex(alpha_data.index)
assignment_bias = pd.DataFrame([
    {"Metric": metric,
     "Spearman_rho": float(alpha_data[["Unresolved_share", metric]].corr(method="spearman").iloc[0, 1]),
     "p_value": float(spearmanr(alpha_data["Unresolved_share"], alpha_data[metric]).pvalue)}
    for metric in ALPHA_METRICS])
assignment_bias["q_value"] = fdr_within_family(
    dict(zip(assignment_bias["Metric"], assignment_bias["p_value"])))["q_value"].values
assignment_bias.to_csv(STATISTICS_DIR / "assignment_rate_vs_alpha_diversity.csv", index=False)

print("\nIs diversity related to how much of a bird's library could be assigned to a genus?")
print(assignment_bias.round(4).to_string(index=False))
_worst_rho = assignment_bias.loc[assignment_bias["Spearman_rho"].abs().idxmax()]
if (assignment_bias["q_value"] < ALPHA_SIGNIFICANCE).any():
    print(f"[CONFOUND] Assignment success is associated with diversity (strongest: "
          f"{_worst_rho['Metric']}, rho = {_worst_rho['Spearman_rho']:.3f}, q = {_worst_rho['q_value']:.4g}). "
          f"Because the unassigned share also differs between host species, part of the species effect "
          f"could be database coverage rather than biology. Section 06.14 refits the species test on birds "
          f"whose libraries were well assigned, which is the test that separates the two.")
else:
    print(f"No association survives correction (strongest |rho| = {abs(_worst_rho['Spearman_rho']):.3f} for "
          f"{_worst_rho['Metric']}, q = {_worst_rho['q_value']:.4g}). Diversity does not track assignment "
          f"success, so the between-species differences are not explained by it. Section 06.14 confirms "
          f"this on the fitted models.")

# ----------------------------------------------------------------
# Sampling design: MUST be checked before any model is specified.
# ----------------------------------------------------------------
design_reports = sampling_design_report(alpha_data)
print("\nSpecies x Locality design (samples entering the rarefied analyses):")
print(design_reports["species_by_locality"].to_string())
print("\nSpecies x Sex design:")
print(design_reports["species_by_sex"].to_string())

if not design_reports["localities_single_species"].empty:
    print("\n[DESIGN WARNING] Localities with a single species -- Locality is confounded with Species "
          "there and they are excluded from the inter-specific model (06.2):")
    print(design_reports["localities_single_species"].to_string())

_conc = design_reports["species_locality_concentration"]
_highly_concentrated = _conc[_conc["Max_share_in_one_locality"] >= 0.80]
if not _highly_concentrated.empty:
    print("\n[DESIGN WARNING] Species largely confined to ONE locality. Even after the exclusion above, "
          "their Species effect is only weakly separable from that locality's effect; this is a "
          "limitation of the sampling design and must be stated when interpreting section 06.0:")
    print(_highly_concentrated.round(3).to_string())

if not design_reports["within_species_low_n_localities"].empty:
    print(f"\n[DESIGN WARNING] Species x Locality cells with N < {MIN_N_PER_LEVEL} "
          f"(too small for Locality to enter that species' intra-specific model):")
    print(design_reports["within_species_low_n_localities"].to_string(index=False))

if not design_reports["within_species_low_n_sex_levels"].empty:
    print(f"\n[DESIGN WARNING] Species x Sex cells with N < {MIN_N_PER_LEVEL}:")
    print(design_reports["within_species_low_n_sex_levels"].to_string(index=False))

_with_index = {"species_by_locality", "species_by_sex", "localities_single_species", "species_locality_concentration"}
for _name, _report in design_reports.items():
    _report.to_csv(STATISTICS_DIR / f"design_{_name}.csv", index=_name in _with_index)

print("\n05.3 verification")
print("-" * 60)
assert (species_summary[("Richness", "count")].reindex(SPECIES_ORDER).fillna(0) > 0).all()
for _name in design_reports:
    assert (STATISTICS_DIR / f"design_{_name}.csv").exists()
print("OK -- section 05.0 complete. Sections 06.0/07.0 read `design_reports` before choosing model terms.")

In [ ]:
# Table · 05.3 — Alpha diversity summarized by host species
show_table(species_summary, "Alpha diversity summarized by host species, from alpha_data.", index=True)

In [ ]:
# Table · 05.3 — Community dominance by host species
show_table(dominance_by_species, "Monodominance by host species, from the rarefied matrix: how often one genus holds at least {:.0%} of a bird's reads, and which genus does so most often.".format(MONODOMINANCE_THRESHOLD), index=True)

In [ ]:
# Table · 05.3 — Community dominance bird by bird
show_table(dominance.reset_index(), "The same monodominance measure bird by bird, so an individual can be looked up rather than inferred from its species mean.", page_length=15)

In [ ]:
# 05.4 — Figure: alpha diversity of each bird, by host species
# ----------------------------------------------------------------------
# The descriptive figure shows all four metrics computed in 05.2 and every
# species, including any that the models in 06.0 hold back -- this is the
# archive of what was measured, and a reader comparing the figure with the
# models should be able to see exactly what the models leave out. The panels
# say which metrics are modeled and which are not.
_n_desc_metrics = len(ALPHA_METRICS_COMPUTED)
_desc_rows = int(np.ceil(_n_desc_metrics / 2))
fig, axes = plt.subplots(_desc_rows, 2, figsize=(12.5, 4.5 * _desc_rows), squeeze=False)
metric_axis_labels = {
    "Richness": f"Observed {RICHNESS_LABEL}\n(log scale)",
    "Shannon": "Shannon diversity H'\n(natural log)",
    "Simpson": "Simpson diversity (1 - D)\n(probability two reads differ)",
    "Pielou": "Pielou's evenness J'\n(0 = uneven, 1 = perfectly even)",
}
present_species = [s for s in SPECIES_ORDER if (alpha_data["Species_code"] == s).any()]
_n_per_species = {s: int((alpha_data["Species_code"] == s).sum()) for s in present_species}

for ax, metric in zip(axes.flatten(), ALPHA_METRICS_COMPUTED):
    for i, s in enumerate(present_species, start=1):
        vals = alpha_data.loc[alpha_data["Species_code"] == s, metric].dropna().values
        ax.boxplot([vals], positions=[i], widths=0.55, showfliers=False, patch_artist=True,
                   medianprops=dict(color="black", linewidth=1.6),
                   whiskerprops=dict(color="0.45"), capprops=dict(color="0.45"),
                   boxprops=dict(facecolor=SPECIES_COLORS[s], alpha=0.28,
                                 edgecolor=SPECIES_COLORS[s], linewidth=1.3))
        jitter = spawn_rng().normal(0, 0.06, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals, s=30, zorder=3, marker=SPECIES_MARKERS[s],
                   color=SPECIES_COLORS[s], edgecolor="white", linewidth=0.6, alpha=0.95)
    # Richness on a log axis. It spans two orders of magnitude here, so on
    # a linear axis a handful of high birds compress every box into the
    # bottom fifth of the panel and the between-species differences the
    # figure exists to show become invisible. The model is on a log link,
    # so a log axis is also the scale the test is performed on.
    if metric == "Richness":
        ax.set_yscale("log")
        ax.yaxis.set_major_locator(mpl.ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0), numticks=12))
        ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
        ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
    ax.set_xticks(range(1, len(present_species) + 1))
    # n printed under each species, so the figure carries its own sample
    # sizes: a 5-bird box and a 41-bird box look identical otherwise.
    ax.set_xticklabels([f"{SPECIES_LABELS_ITALIC.get(s, s)}\nn = {_n_per_species[s]}"
                        for s in present_species], rotation=18, ha="right", fontsize=8.5)
    ax.set_ylabel(metric_axis_labels[metric])
    ax.set_title(metric if metric in ALPHA_METRICS else f"{metric} (descriptive only)")

# Any panel left over when the metrics do not fill the grid is removed
# rather than left as an empty pair of axes.
for _spare in axes.flatten()[_n_desc_metrics:]:
    _spare.remove()

handles = [Line2D([0], [0], marker=SPECIES_MARKERS[s], color="none", markerfacecolor=SPECIES_COLORS[s],
                  markeredgecolor="white", markersize=9, label=species_legend_label(s))
           for s in present_species]
fig.legend(handles=handles, loc="lower center", ncol=len(present_species), bbox_to_anchor=(0.5, -0.075),
           title="Species (origin): warm colors = invasive, cool colors = native", title_fontsize=9)
fig.suptitle(f"Alpha diversity of the cloacal bacterial community, by host species "
             f"(rarefied to {rarefaction_depth:,} reads)", fontsize=14, fontweight="bold", y=1.0)
figure_caption(fig,
    f"Each point is one bird; boxes show the median and interquartile range. Values are means over "
    f"{N_RAREFACTION_REPLICATES} independent rarefactions to {rarefaction_depth:,} reads "
    f"(N = {len(alpha_data)} birds: " +
    ", ".join(f"{s} {int((alpha_data['Species_code'] == s).sum())}" for s in present_species) +
    f"). Formal tests of the species effect, adjusted for locality, are in section 06.0, which models "
    f"{', '.join(ALPHA_METRICS)}"
    + (f"; {', '.join(m for m in ALPHA_METRICS_COMPUTED if m not in ALPHA_METRICS)} is shown here for "
       f"completeness and archived in alpha_diversity_data.csv, but is not modeled, for the reason given "
       f"in section 06.0" if len(ALPHA_METRICS_COMPUTED) > len(ALPHA_METRICS) else "")
    + ".",
    y=-0.105)
plt.tight_layout()
alpha_desc_fig_path = save_figure(fig, ALPHA_DIR / "alpha_diversity_by_species.png")
plt.close(fig)   # not displayed here: the interactive version is the next cell

print("05.4 verification")
print("-" * 60)
assert alpha_desc_fig_path.exists()
print(f"Saved: {alpha_desc_fig_path}")
print("OK -- descriptive alpha-diversity figure complete.")

In [ ]:
# Interactive view of figure 05.4 — Alpha diversity of every bird, by host species
# ----------------------------------------------------------------------
# The interactive counterpart of figure 05.4, drawn from alpha_data: one
# panel per computed metric, one box and one point cloud per host species,
# and each point carrying the bird it belongs to. All four metrics are
# shown, including the one that is archived rather than modeled, because
# this view is descriptive and a reader comparing it against the models
# should be able to see everything that was measured.
from plotly.subplots import make_subplots

_alpha_species = [s for s in SPECIES_ORDER if (alpha_data["Species_code"] == s).any()]
_metric_titles = {"Richness": "Observed genus richness (log axis)",
                  "Shannon": "Shannon diversity H'",
                  "Simpson": "Simpson diversity (1 - D)",
                  "Pielou": "Pielou's evenness J'"}

fig10_1 = make_subplots(
    rows=2, cols=2, vertical_spacing=0.13, horizontal_spacing=0.09,
    subplot_titles=[f"{m}{'' if m in ALPHA_METRICS else '  (archived, not modeled)'}"
                    for m in ALPHA_METRICS_COMPUTED])

for _i, _metric in enumerate(ALPHA_METRICS_COMPUTED):
    _row, _col = _i // 2 + 1, _i % 2 + 1
    for _species in _alpha_species:
        _block = alpha_data.loc[alpha_data["Species_code"] == _species, [_metric, "Locality", "Sex"]].dropna()
        fig10_1.add_trace(
            go.Box(
                y=_block[_metric].values,
                name=species_html(_species),
                legendgroup=_species, showlegend=(_i == 0),
                boxpoints="all", jitter=0.55, pointpos=0, boxmean=True,
                marker=dict(color=SPECIES_COLORS[_species], size=6,
                            symbol=PLOTLY_MARKERS[SPECIES_MARKERS[_species]],
                            line=dict(color="white", width=0.6)),
                line=dict(color=SPECIES_COLORS[_species], width=1.2),
                fillcolor="rgba(0,0,0,0)",
                customdata=np.column_stack([_block.index.values,
                                            _block["Locality"].astype(str).values,
                                            _block["Sex"].astype(str).values]),
                hovertemplate=("<b>%{customdata[0]}</b><br>"
                               + species_html(_species, origin=True) + "<br>"
                               + "Locality: %{customdata[1]}<br>Sex: %{customdata[2]}<br>"
                               + _metric + ": %{y:.3f}<extra></extra>")),
            row=_row, col=_col)
    fig10_1.update_yaxes(title_text=_metric_titles[_metric], row=_row, col=_col)
    fig10_1.update_xaxes(showticklabels=False, row=_row, col=_col)

# Richness spans two orders of magnitude, so a linear axis would compress
# every box into the bottom of its panel; this is the same log axis the
# count model is fitted on.
fig10_1.update_yaxes(type="log", row=1, col=1)
fig10_1.update_layout(
    template=PLOTLY_TEMPLATE, height=820, boxgap=0.35,
    title=dict(text=f"Alpha diversity of each bird, by host species "
                    f"(rarefied to {rarefaction_depth:,} reads, mean of {N_RAREFACTION_REPLICATES} draws)",
               x=0.5, xanchor="center", font=dict(size=16)),
    legend=dict(orientation="h", yanchor="bottom", y=-0.09, xanchor="center", x=0.5,
                title_text="Host species — click to hide, double-click to isolate"),
    margin=dict(t=90, b=110))
fig10_1.show()

assert len(fig10_1.data) == len(ALPHA_METRICS_COMPUTED) * len(_alpha_species)

## 06.0 — Between-species alpha diversity

Models each alpha-diversity metric as a function of `Species + Locality` on the multi-species
dataset `inter_specific_data`, which holds every rarefied bird whose locality contains at least
two host species. Two exclusions are applied before any model is fitted, both for reasons of
identifiability rather than of results. Localities where only one species was sampled are dropped:
there, species and site are the same variable and the model could not separate them. A host species
with fewer birds than `MIN_N_FOR_SPECIES_MODEL` (02.2) is also held back, because a species mean
estimated from a handful of birds alongside a locality effect carries an interval that overlaps
every other species, and — since the pairwise contrasts and the compact letter display are built
from all species at once — that one unresolvable mean degrades the comparisons among the species
that are well sampled. Such a species keeps every descriptive result and contributes no
coefficient; section 06.2 names it and prints the final design.

### Distribution families

Each response is modeled on the scale it lives on, with the family following from the support of
the response rather than from inspecting the outcome:

| Response | Support | Family |
| --- | --- | --- |
| Richness | counts | Poisson, quasi-Poisson or negative binomial, chosen from the observed dispersion |
| Shannon *H'* | continuous, strictly positive | Gamma with a log link |
| Simpson | strictly inside (0, 1) | Beta regression, with the precision submodel chosen by likelihood-ratio test |

The count family is the one exception decided from the data, by the Pearson dispersion of a Poisson
fit: these counts can be either over- or under-dispersed depending on rarefaction depth, and a
negative binomial imposed on under-dispersed counts does not converge reliably. Shannon takes the
Gamma because *H'* is bounded below by zero and its smallest species mean sits close enough to that
bound for a symmetric family to put an appreciable share of its predictive distribution below it;
section 06.6 quantifies exactly that, and 06.9 confirms the fit with quantile residuals.

Pielou's evenness *J'* is computed and carried in `alpha_diversity_data.csv`, but it is
*J' = H' / log S* — a function of the two metrics already modeled — so modeling it adds a fourth
correlated test rather than new information. It is reported descriptively in section 05.4 and
archived with the other metrics, so a reader who wants it has the per-bird values.

### Testing

Every term is tested by comparing the full model against the same model without that term, so the
result does not depend on which level happens to be the reference. Multiplicity is controlled
within families that are defined in advance: the three metrics form one family for the Species
effect and another for the Locality effect, corrected separately. Post-hoc pairwise comparisons
between species are run only for metrics whose global test survives correction, and are corrected
again within each metric's own set of comparisons.

Two robustness checks close the section: 06.13 repeats the global species test across a span of
rarefaction depths, and 06.14 repeats it on the subset of birds whose libraries were well assigned
taxonomically, which separates a biological difference between host species from a difference in
how well each host's microbiota is represented in the reference database.

In [ ]:
# 06.1 — Modeling imports, model-fitting helpers and contrast machinery
# ----------------------------------------------------------------------
import patsy
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import beta as beta_dist
from scipy.stats import chi2 as chi2_dist
from scipy.stats import gamma as gamma_dist
from scipy.stats import nbinom as nbinom_dist
from scipy.stats import norm as norm_dist
from scipy.stats import poisson as poisson_dist
from scipy.stats import shapiro
from statsmodels.discrete.discrete_model import NegativeBinomial
from statsmodels.othermod.betareg import BetaModel

SPECIES_TERM = "C(Species_code)"
LOCALITY_TERM = "C(Locality)"


def _has_usable_covariance(result) -> bool:
    """True when the fit produced a usable parameter covariance matrix."""
    try:
        cov = np.asarray(result.cov_params(), dtype=float)
    except Exception:
        return False
    return bool(np.all(np.isfinite(cov))) and cov.size > 0


def fit_negative_binomial(formula: str, data: pd.DataFrame):
    """Fit an NB2 model, falling back through optimizers if the Hessian will not invert.

    statsmodels' default Newton step can fail to invert the Hessian on
    small, strongly separated designs (`HessianInversionWarning`, after
    which `cov_params()` raises and every standard error, confidence
    interval and post-hoc contrast is unavailable). Poisson estimates are
    used as starting values, and the optimizer is retried, so the model
    either returns a usable covariance matrix or raises an explicit error
    -- never a fit that looks converged but has no standard errors.
    """
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        poisson_start = smf.poisson(formula, data=data).fit(disp=False).params.values
        start = np.append(poisson_start, 1.0)
        attempts = [
            {"disp": False},
            {"disp": False, "start_params": start},
            {"disp": False, "start_params": start, "method": "bfgs", "maxiter": 2000},
            {"disp": False, "start_params": start, "method": "nm", "maxiter": 6000},
        ]
        last = None
        for kwargs in attempts:
            try:
                result = NegativeBinomial.from_formula(formula, data=data).fit(**kwargs)
            except Exception as exc:
                last = exc
                continue
            if np.isfinite(result.llf) and _has_usable_covariance(result):
                return result
            last = result
    raise RuntimeError(
        f"Negative-binomial model did not produce a usable covariance matrix for:\n    {formula}\n"
        f"Every optimizer was tried. This usually means the design is (nearly) separated -- a species or "
        f"locality level with too few observations, or a level whose richness barely overlaps the others. "
        f"Check the design cross-tabulations in 05.3 before interpreting anything from this term. "
        f"(last outcome: {last})"
    )


# A count response is not automatically over-dispersed. Richness at a
# deep rarefaction depth can be close to Poisson, or even UNDER-dispersed,
# and then the NB dispersion parameter sits on its boundary (alpha -> 0),
# the Hessian is singular, and the fit returns no standard errors at all.
# The count family is therefore chosen by a pre-specified rule on the
# observed dispersion rather than assumed:
#
#   phi = Pearson chi2 / residual df of a Poisson fit
#     phi >  1.2  -> negative binomial, dispersion estimated by MLE (LR tests)
#     phi <= 1.2  -> Poisson, with a quasi-Poisson scale when phi is far
#                    from 1, which handles under-dispersion too (Wald tests)
#
# The NB branch still falls back to quasi-Poisson if the NB fit cannot
# produce a covariance matrix, so the pipeline never stops on a numerical
# failure without saying what it did instead.
POISSON_OVERDISPERSION_THRESHOLD = 1.2


def fit_count_model(formula: str, data: pd.DataFrame, quiet: bool = False) -> dict:
    """Fit the count model chosen by the observed dispersion; report which and why."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        poisson = smf.glm(formula, data=data, family=sm.families.Poisson()).fit()
    dispersion = float(poisson.pearson_chi2 / poisson.df_resid)

    if dispersion > POISSON_OVERDISPERSION_THRESHOLD:
        try:
            nb = fit_negative_binomial(formula, data)
            return {"model": nb, "family": "negative binomial (alpha estimated by MLE)",
                    "key": "negative binomial",
                    "dispersion": dispersion, "test": "LR", "fitter": "nb"}
        except RuntimeError as exc:
            if not quiet:
                print(f"[NOTE] The data are over-dispersed (phi = {dispersion:.2f}) but the negative-binomial "
                      f"fit produced no usable covariance matrix, so a quasi-Poisson model is reported "
                      f"instead. Reason: {str(exc).splitlines()[0]}")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        scaled = smf.glm(formula, data=data, family=sm.families.Poisson()).fit(scale="X2")
    if dispersion < 1 / POISSON_OVERDISPERSION_THRESHOLD:
        family = f"quasi-Poisson, under-dispersed (scale = {dispersion:.2f})"
    elif dispersion > 1.0:
        family = f"quasi-Poisson (scale = {dispersion:.2f})"
    else:
        family = f"Poisson (dispersion = {dispersion:.2f}, no correction needed)"
    # `key` names the distribution for the goodness-of-fit diagnostics in
    # 06.9. A quasi-Poisson is a quasi-likelihood: it has a mean and a
    # variance but no distribution function, so no quantile residual can
    # be defined for it, and 06.9 falls back to Pearson residuals there.
    key = "poisson" if 1.0 / POISSON_OVERDISPERSION_THRESHOLD <= dispersion <= 1.0 else "quasi-poisson"
    return {"model": scaled, "family": family, "key": key,
            "dispersion": dispersion, "test": "Wald", "fitter": "poisson"}


def count_term_test(fit: dict, term: str, formula_full: str, data: pd.DataFrame) -> dict:
    """Global test for one term of a count model, matched to how it was fitted."""
    if fit["test"] == "Wald":
        return wald_test_term(fit["model"], term)
    response = formula_full.split("~")[0].strip()
    remaining = [t.strip() for t in formula_full.split("~")[1].split("+") if t.strip() != term]
    reduced_formula = f"{response} ~ " + (" + ".join(remaining) if remaining else "1")
    return lr_test(fit["model"], fit_negative_binomial(reduced_formula, data))


def fit_beta_model(formula: str, data: pd.DataFrame, precision_formula: str = "~ 1",
                   required: bool = True):
    """Fit a beta-regression model, retrying with other optimizers if the likelihood is not finite.

    `required=False` returns None instead of raising, so one awkward
    nested model cannot stop the whole notebook: the corresponding test
    is then reported as NaN and excluded from the FDR correction, which
    is visible, rather than as a p-value that does not exist.
    """
    attempts = [{}, {"method": "nm", "maxiter": 8000, "maxfun": 8000},
                {"method": "bfgs", "maxiter": 4000}, {"method": "powell", "maxiter": 8000}]
    best = None
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for kwargs in attempts:
            try:
                result = BetaModel.from_formula(formula, data=data,
                                                exog_precision_formula=precision_formula).fit(**kwargs)
            except Exception:
                continue
            if np.isfinite(result.llf):
                if best is None or result.llf > best.llf:
                    best = result
                if _has_usable_covariance(result):
                    return result
    if best is not None:
        return best
    message = (
        f"Beta regression did not converge to a finite log-likelihood for:\n"
        f"    mean: {formula}   precision: {precision_formula}\n"
        f"Check that the response is strictly inside (0, 1) and that every level has enough observations."
    )
    if required:
        raise RuntimeError(message)
    print(f"[WARN] {message}\n       This model is skipped and its test reported as NaN.")
    return None


def check_unit_interval(values: pd.Series, name: str) -> pd.Series:
    """Beta regression needs a response strictly inside (0, 1).

    If a value sits exactly at a boundary, the standard Smithson &
    Verkuilen (2006) squeeze is applied and reported, rather than letting
    the fit fail or silently drop the observation.
    """
    v = pd.to_numeric(values, errors="coerce")
    n_boundary = int(((v <= 0) | (v >= 1)).sum())
    if n_boundary == 0:
        return v
    n = len(v)
    squeezed = (v * (n - 1) + 0.5) / n
    print(f"[NOTE] {name}: {n_boundary} value(s) exactly at 0 or 1; applying the Smithson & Verkuilen (2006) "
          f"squeeze (y*(n-1)+0.5)/n so beta regression is defined. Report this in the Methods.")
    return squeezed


def lr_test(result_full, result_reduced) -> dict:
    """Likelihood-ratio test for two NESTED models (NB, Beta).

    df is the exact difference in the number of estimated parameters,
    which is robust to each family's own df-counting convention.
    """
    if result_full is None or result_reduced is None:
        return {"test": "LR", "statistic": np.nan, "df": 0, "pvalue": np.nan}
    df_diff = len(result_full.params) - len(result_reduced.params)
    stat = 2 * (result_full.llf - result_reduced.llf)
    # A nested full model cannot have a LOWER likelihood than its own
    # reduced model: a negative statistic means one of the two fits did
    # not converge. Returning p = 1 there would silently report "no
    # effect" for what is actually a numerical failure, so it is flagged
    # as NaN instead.
    if df_diff > 0 and stat < -1e-6:
        print(f"[WARN] Likelihood-ratio statistic is negative ({stat:.4g}): the full model converged to a "
              f"worse likelihood than its nested reduced model. Reported as NaN, not as p = 1.")
        return {"test": "LR", "statistic": float(stat), "df": int(df_diff), "pvalue": np.nan}
    stat = max(float(stat), 0.0)
    return {"test": "LR", "statistic": stat, "df": int(df_diff),
            "pvalue": float(chi2_dist.sf(stat, df_diff)) if df_diff > 0 else np.nan}


def wald_test_term(result, term_name: str) -> dict:
    """Joint Wald test for one patsy term in a formula-API OLS/GLM fit.

    Uses statsmodels' own term grouping rather than a hand-built contrast
    matrix, and honors the fit's covariance type, so an HC3 fit gives a
    heteroscedasticity-robust test.

    The reference distribution is recorded, not assumed: statsmodels
    refers the statistic to an F distribution for an ordinary
    least-squares fit and to chi-squared for a GLM or a robust-covariance
    fit. The same numeric statistic means different things under the two,
    so a results table that prints "statistic" and "df" without saying
    which one cannot be checked by a reader.
    """
    result_wald = result.wald_test_terms(skip_single=False, scalar=True)
    row = result_wald.table.loc[term_name]
    distribution = "F" if getattr(result_wald, "distribution", "") == "F" else "chi2"
    return {"test": f"Wald ({distribution})", "statistic": float(row["statistic"]),
            "df": int(row["df_constraint"]), "pvalue": float(row["pvalue"]),
            "distribution": distribution}


def level_contrast_vector(exog_names: list[str], term_prefix: str, level_a: str, level_b: str) -> np.ndarray:
    """Contrast c with c @ params = (linear predictor at level_a) - (at level_b).

    `term_prefix` must be the patsy term exactly as it appears in the
    coefficient names (e.g. "C(Species_code)"). The assert matters: a
    prefix that matched nothing would return an all-zero contrast, which
    is a well-formed vector that estimates exactly 0 without raising.
    Valid as a marginal comparison because these models contain no
    Species:Locality interaction, so the effects are additive.
    """
    c = np.zeros(len(exog_names))
    name_a, name_b = f"{term_prefix}[T.{level_a}]", f"{term_prefix}[T.{level_b}]"
    found = 0
    if name_a in exog_names:
        c[exog_names.index(name_a)] += 1.0; found += 1
    if name_b in exog_names:
        c[exog_names.index(name_b)] -= 1.0; found += 1
    assert found > 0, (
        f"Contrast {level_a} vs {level_b}: neither {name_a!r} nor {name_b!r} is a coefficient of this "
        f"model. Available: {[n for n in exog_names if term_prefix in n]}. "
        f"(One of the two levels being the reference level is expected; neither of them matching is not.)"
    )
    return c


def wald_contrast(params: np.ndarray, cov: np.ndarray, contrast_vector: np.ndarray,
                  transform: str = "identity") -> dict:
    """Wald test and 95% CI for a linear contrast, optionally back-transformed.

    transform="identity": the linear predictor IS the response scale
        (Gaussian), so the contrast is a mean difference.
    transform="exp"     : log link (NB, Gamma), so exp(contrast) is a
        rate ratio / ratio of means.
    """
    params = np.asarray(params, dtype=float)
    diff = float(contrast_vector @ params)
    se = float(np.sqrt(contrast_vector @ np.asarray(cov, dtype=float) @ contrast_vector))
    z = diff / se if se > 0 else np.nan
    pvalue = float(2 * norm_dist.sf(abs(z))) if np.isfinite(z) else np.nan
    ci_low, ci_high = diff - 1.959964 * se, diff + 1.959964 * se
    if transform == "exp":
        return {"estimate": float(np.exp(diff)), "ci_low": float(np.exp(ci_low)),
                "ci_high": float(np.exp(ci_high)), "z": z, "pvalue": pvalue}
    return {"estimate": diff, "ci_low": ci_low, "ci_high": ci_high, "z": z, "pvalue": pvalue}


def quantile_residuals(result, y, family: str, rng: "np.random.Generator | None" = None) -> np.ndarray:
    """Randomized quantile residuals (Dunn & Smyth 1996) for any fitted family.

    Raw or Pearson residuals from a non-Gaussian model are not expected to
    be normal even when the model is right, so a Q-Q plot of them cannot
    be read the way a Q-Q plot of OLS residuals can. Quantile residuals
    fix that: r_i = Phi^-1(F(y_i; theta_i)), where F is the fitted
    distribution for that observation. Under a correctly specified model
    they are exactly standard normal, whatever the family -- so the same
    plot and the same test apply to the count, Gamma and beta models
    alike, and the responses become directly comparable.

    Randomisation is needed only for a discrete response, where F jumps:
    the residual is drawn uniformly across the jump so that the result is
    continuous. Gamma and beta are continuous and need none.
    """
    y = np.asarray(y, dtype=float)
    if family == "gamma":
        mu = np.asarray(result.fittedvalues, dtype=float)
        phi = float(result.scale)                      # Var = phi * mu^2
        u = gamma_dist.cdf(y, a=1.0 / phi, scale=phi * mu)
    elif family == "beta":
        mu = np.asarray(result.model.predict(result.params, which="mean"), dtype=float)
        precision = np.asarray(result.model.predict(result.params, which="precision"), dtype=float)
        u = beta_dist.cdf(y, mu * precision, (1.0 - mu) * precision)
    elif family == "negative binomial":
        mu = np.asarray(result.predict(), dtype=float)
        alpha = float(result.params.get("alpha", np.nan))
        size = 1.0 / alpha
        prob = size / (size + mu)
        lower = nbinom_dist.cdf(y - 1, size, prob)
        upper = nbinom_dist.cdf(y, size, prob)
        u = (rng or np.random.default_rng()).uniform(lower, upper)
    elif family == "poisson":
        mu = np.asarray(result.predict(), dtype=float)
        lower, upper = poisson_dist.cdf(y - 1, mu), poisson_dist.cdf(y, mu)
        u = (rng or np.random.default_rng()).uniform(lower, upper)
    elif family in ("gaussian", "lognormal"):
        # For a log-normal on Y, log Y is Gaussian, so
        # F_Y(y) = Phi((log y - mu)/sigma) and the quantile residual is
        # exactly the standardized residual of the fit on the log scale.
        # The same expression therefore serves both families.
        resid = np.asarray(result.resid, dtype=float)
        return resid / np.std(resid, ddof=1)
    else:
        raise ValueError(f"No quantile-residual definition for family {family!r}.")
    return norm_dist.ppf(np.clip(u, 1e-12, 1 - 1e-12))


def model_residual_check(result, y, family_key: str, dispersion: float | None = None,
                         rng: "np.random.Generator | None" = None) -> dict:
    """Residual diagnostic for one fitted model, with the right residual for its family.

    Returns quantile residuals and a Shapiro-Wilk test where the family
    has a distribution function, and scaled Pearson residuals with no
    test where it does not: a quasi-likelihood specifies only a mean and
    a variance, so normality is not something it claims and testing for
    it would be answering a question the model never asked. In that case
    the informative number is the residual SD, which should be near 1 if
    the variance function is right.
    """
    y = np.asarray(y, dtype=float)
    if family_key == "quasi-poisson":
        mu = np.asarray(result.predict(), dtype=float)
        scale = float(dispersion) if dispersion else 1.0
        residuals = (y - mu) / np.sqrt(np.clip(mu * scale, 1e-12, None))
        return {"kind": "Pearson", "residuals": residuals, "W": np.nan, "pvalue": np.nan,
                "mean": float(np.mean(residuals)), "sd": float(np.std(residuals, ddof=1)),
                "n": int(len(residuals))}
    residuals = quantile_residuals(result, y, family_key, rng)
    return {"kind": "quantile", "residuals": residuals, **residual_normality(residuals)}


def residual_normality(residuals: np.ndarray) -> dict:
    """Shapiro-Wilk on quantile residuals: the single number that says whether the family fits."""
    finite = residuals[np.isfinite(residuals)]
    if len(finite) < 3:
        return {"W": np.nan, "pvalue": np.nan, "mean": np.nan, "sd": np.nan, "n": int(len(finite))}
    w, pvalue = shapiro(finite)
    return {"W": float(w), "pvalue": float(pvalue), "mean": float(finite.mean()),
            "sd": float(finite.std(ddof=1)), "n": int(len(finite))}


def beta_mean_params(result) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Mean-submodel parameters, their covariance block, and their names, from a BetaModel fit."""
    names = list(result.model.exog_names)
    k = len(names)
    return np.asarray(result.params, dtype=float)[:k], np.asarray(result.cov_params(), dtype=float)[:k, :k], names


def beta_contrast(result, level_a: str, level_b: str, term_prefix: str,
                  rng: np.random.Generator, n_sim: int = 20000) -> dict:
    """Beta-regression contrast between two levels, reported on the 0-1 response scale.

    The mean submodel uses a logit link, so a difference of logit-scale
    coefficients is NOT a difference in the mean. Two separate things are
    therefore computed, each with the right tool:
      * the p-value, from the ordinary Wald test on the LOGIT scale,
        where the asymptotic normal approximation applies. A tail
        proportion of the simulated differences would not do: it is a
        Monte-Carlo by-product bounded below by 1/n_sim, not a test
        statistic.
      * the effect size and its 95% CI, by simulating from the asymptotic
        sampling distribution of the mean-submodel parameters and
        back-transforming each draw (a Monte-Carlo delta method), so the
        interval is on the interpretable 0-1 scale.
    """
    params, cov, names = beta_mean_params(result)
    c = level_contrast_vector(names, term_prefix, level_a, level_b)
    logit_scale = wald_contrast(params, cov, c, transform="identity")

    base = {n: 0.0 for n in names}
    base["Intercept"] = 1.0
    row_a, row_b = dict(base), dict(base)
    if f"{term_prefix}[T.{level_a}]" in row_a:
        row_a[f"{term_prefix}[T.{level_a}]"] = 1.0
    if f"{term_prefix}[T.{level_b}]" in row_b:
        row_b[f"{term_prefix}[T.{level_b}]"] = 1.0
    exog_a = np.array([row_a[n] for n in names])
    exog_b = np.array([row_b[n] for n in names])

    draws = rng.multivariate_normal(params, cov, size=n_sim)
    mu_a = 1.0 / (1.0 + np.exp(-(draws @ exog_a)))
    mu_b = 1.0 / (1.0 + np.exp(-(draws @ exog_b)))
    diff = mu_a - mu_b
    return {
        "estimate": float(np.mean(diff)),
        "ci_low": float(np.percentile(diff, 2.5)),
        "ci_high": float(np.percentile(diff, 97.5)),
        "z": logit_scale["z"],
        "pvalue": logit_scale["pvalue"],
    }


print("06.1 verification")
print("-" * 60)
_names = ["Intercept", "C(Species_code)[T.PasDom]", "C(Species_code)[T.PsiKra]"]
_c = level_contrast_vector(_names, SPECIES_TERM, "PasDom", "PsiKra")
assert list(_c) == [0.0, 1.0, -1.0]
_c_ref = level_contrast_vector(_names, SPECIES_TERM, "PasDom", REFERENCE_SPECIES)
assert list(_c_ref) == [0.0, 1.0, 0.0], "Contrast against the reference level must keep only the other term."
try:
    level_contrast_vector(_names, "C(Locality)", "A", "B")
    raise SystemExit("level_contrast_vector should have raised for a non-matching term prefix.")
except AssertionError:
    pass
assert _has_usable_covariance(smf.ols("y ~ x", data=pd.DataFrame({"y": [1.0, 2, 3, 4, 5], "x": [1.0, 2, 3, 4, 6]})).fit())
print("Self-tests passed: contrast vectors are correct, and a term prefix that matches no coefficient")
print("raises rather than returning an all-zero contrast that would estimate exactly 0.")
print("OK -- inferential-modeling imports and helper functions ready.")

In [ ]:
# 06.2 — inter_specific_data: the multi-species modeling set, and its design diagnostics
# ----------------------------------------------------------------------
# Complete cases only. Localities represented by a single species are
# EXCLUDED: with one species at a locality, Species and Locality are not
# jointly identifiable there (design check, 05.3).
excluded_localities = design_reports["localities_single_species"].index.tolist()

inter_specific_data = (
    alpha_data[["Species_code", "Locality"] + ALPHA_METRICS].dropna().copy()
)
n_before_locality_filter = len(inter_specific_data)
inter_specific_data = inter_specific_data.loc[~inter_specific_data["Locality"].isin(excluded_localities)].copy()

# A species with too few birds to estimate its own mean (policy and
# reasoning in 02.2) is held back from the models. It keeps every
# descriptive result -- diversity summaries, the per-bird figure, the
# composition profiles -- and simply does not contribute a coefficient, a
# letter in the compact letter display or a pairwise contrast.
_species_n = inter_specific_data["Species_code"].astype(str).value_counts()
modeled_species = [s for s in SPECIES_ORDER if _species_n.get(s, 0) >= MIN_N_FOR_SPECIES_MODEL]
species_described_only = [s for s in SPECIES_ORDER
                          if _species_n.get(s, 0) > 0 and s not in modeled_species]
inter_specific_data = inter_specific_data.loc[
    inter_specific_data["Species_code"].isin(modeled_species)].copy()
# Dropping a species can leave a locality holding only one of the rest, so
# the identifiability filter is applied again to the reduced design.
_single_after = [loc for loc, n in
                 inter_specific_data.groupby("Locality", observed=True)["Species_code"].nunique().items()
                 if n < 2]
if _single_after:
    inter_specific_data = inter_specific_data.loc[
        ~inter_specific_data["Locality"].isin(_single_after)].copy()
    excluded_localities = sorted(set(excluded_localities) | set(_single_after))

# Setting Species_code as an ORDERED categorical whose first category is
# REFERENCE_SPECIES is what makes plain `C(Species_code)` use that
# species as the treatment reference -- no formula-embedded variable.
# SPECIES_ORDER stays the DISPLAY order everywhere else; only the model's
# category order is rotated, so figures keep a stable species sequence
# while the contrasts get a well-sampled baseline.
_species_present = [s for s in SPECIES_ORDER if s in set(inter_specific_data["Species_code"])]
_model_species_order = ([REFERENCE_SPECIES] + [s for s in _species_present if s != REFERENCE_SPECIES]
                        if REFERENCE_SPECIES in _species_present else _species_present)
inter_specific_data["Species_code"] = pd.Categorical(
    inter_specific_data["Species_code"], categories=_model_species_order, ordered=True)
inter_specific_data["Locality"] = inter_specific_data["Locality"].astype("category")

# Richness carried forward is a mean over rarefaction replicates and is
# therefore not an integer; the negative binomial is a distribution over
# counts, so the NB model is fitted to the rounded value.
inter_specific_data["Richness_count"] = np.rint(inter_specific_data["Richness"]).astype(int)
# Beta regression needs a response strictly in (0, 1).
for _m in BETA_RESPONSE_METRICS:
    inter_specific_data[_m] = check_unit_interval(inter_specific_data[_m], _m)
_rounding_shift = float((inter_specific_data["Richness_count"] - inter_specific_data["Richness"]).abs().mean())

inter_specific_data = inter_specific_data.reset_index(drop=False)
species_present = _species_present

print("06.2 verification")
print("-" * 60)
print(f"Complete cases (rarefied samples): {n_before_locality_filter}")
if excluded_localities:
    print(f"Excluded (single-species) localities: {excluded_localities}")
print(f"MIN_N_FOR_SPECIES_MODEL = {MIN_N_FOR_SPECIES_MODEL}")
print(f"Modeled species        : {modeled_species}")
if species_described_only:
    print(f"Described but not modeled: "
          + ", ".join(f"{s} (N = {int(_species_n.get(s, 0))})" for s in species_described_only)
          + " -- too few birds in the complete-case design to estimate a species mean "
            "alongside the locality effect; all descriptive results retained.")
print(f"Inter-specific analysis N = {len(inter_specific_data)}")
print(f"Reference species for every Species contrast: {REFERENCE_SPECIES}")
print(f"Richness rounded for the NB fit; mean |rounded - unrounded| = {_rounding_shift:.3f} genera "
      f"({100 * _rounding_shift / inter_specific_data['Richness'].mean():.2f}% of the mean richness).")
print("\nFinal Species x Locality design used in the model:")
print(pd.crosstab(inter_specific_data["Species_code"], inter_specific_data["Locality"]).to_string())
assert inter_specific_data["Locality"].nunique() >= 1
assert (inter_specific_data.groupby("Locality", observed=True)["Species_code"].nunique() >= 2).all(), \
    "A single-species locality survived the filter -- Species and Locality would not be identifiable."
assert list(inter_specific_data["Species_code"].cat.categories)[0] == REFERENCE_SPECIES
assert set(species_present) == set(modeled_species), \
    "The modeled species set and the species actually present in the design disagree."
assert (inter_specific_data["Species_code"].astype(str).value_counts()
        >= MIN_N_FOR_SPECIES_MODEL).all(), \
    "A species below MIN_N_FOR_SPECIES_MODEL is still in the modeling set."

# ------------------------------------------------------------------
# Identifiable is not the same as well-separated. Every locality having
# >= 2 species is enough for the model to HAVE a unique solution; it says
# nothing about how precisely Species and Locality can be told apart. When
# a species sits almost entirely in one locality, its coefficient and that
# locality's coefficient are estimated from nearly the same contrast, and
# they trade off against each other -- typically appearing as a large
# positive species effect beside a large negative locality effect that
# very nearly cancel. The variance inflation factor measures exactly this,
# so it is computed rather than left for a reader to infer from the
# coefficient table.
# ------------------------------------------------------------------
_design = patsy.dmatrix(f"{SPECIES_TERM} + {LOCALITY_TERM}", inter_specific_data,
                        return_type="dataframe")
_vif_rows = []
for _col in _design.columns:
    if _col == "Intercept":
        continue
    _others = [c for c in _design.columns if c not in ("Intercept", _col)]
    _r2 = smf.ols(f"Q('{_col}') ~ " + " + ".join(f"Q('{c}')" for c in _others),
                  data=_design).fit().rsquared if _others else 0.0
    _vif_rows.append({"Coefficient": _col, "R2_on_other_terms": _r2,
                      "VIF": (1.0 / (1.0 - _r2)) if _r2 < 1 - 1e-12 else np.inf})
design_vif = pd.DataFrame(_vif_rows).sort_values("VIF", ascending=False)
design_vif.to_csv(STATISTICS_DIR / "design_variance_inflation.csv", index=False)
print("\nVariance inflation of the Species + Locality design "
      "(how far each coefficient is from being determined by the others):")
print(design_vif.round(3).to_string(index=False))

MAX_TOLERABLE_VIF = 5.0
_inflated = design_vif[design_vif["VIF"] >= MAX_TOLERABLE_VIF]
if not _inflated.empty:
    print(f"\n[DESIGN WARNING] {len(_inflated)} coefficient(s) have VIF >= {MAX_TOLERABLE_VIF:.0f}: "
          f"{_inflated['Coefficient'].tolist()}. Their standard errors are inflated by a factor of "
          f"{np.sqrt(_inflated['VIF'].max()):.1f} relative to a balanced design, and the corresponding "
          f"Species and Locality effects are partly interchangeable: the model cannot fully attribute "
          f"that variation to host species rather than to sampling site. The GLOBAL species test is "
          f"unaffected -- it compares nested models rather than reading individual coefficients -- but "
          f"no individual coefficient among these should be interpreted on its own, and the Discussion "
          f"must say that species and site are partially confounded by the sampling design.")
else:
    print(f"\nAll VIFs below {MAX_TOLERABLE_VIF:.0f}: Species and Locality are separable in this design.")
print("OK -- every remaining locality has >= 2 species; Species and Locality are jointly identifiable.")

In [ ]:
# 06.3 — Richness ~ Species + Locality, count family chosen from the observed dispersion
# ----------------------------------------------------------------------
richness_formula_full = f"Richness_count ~ {SPECIES_TERM} + {LOCALITY_TERM}"

richness_fit = fit_count_model(richness_formula_full, inter_specific_data)
richness_model_full = richness_fit["model"]
richness_family = richness_fit["family"]
richness_family_key = richness_fit["key"]
richness_lr_species = count_term_test(richness_fit, SPECIES_TERM, richness_formula_full, inter_specific_data)
richness_lr_locality = count_term_test(richness_fit, LOCALITY_TERM, richness_formula_full, inter_specific_data)

print("06.3 verification")
print("-" * 60)
print(f"Poisson dispersion phi = Pearson chi2 / df = {richness_fit['dispersion']:.3f} "
      f"(> {POISSON_OVERDISPERSION_THRESHOLD} would call for a negative binomial)")
print(f">>> Pre-specified rule selects: {richness_family}")
print()
print(richness_model_full.summary())
if richness_fit["fitter"] == "nb":
    print(f"\nEstimated NB dispersion alpha = {richness_model_full.params.get('alpha', np.nan):.4f} (MLE, not fixed at 1)")
    assert "alpha" in richness_model_full.params.index
print(f"\nGlobal {richness_lr_species['test']} test, Species | Locality: "
      f"stat={richness_lr_species['statistic']:.3f}, df={richness_lr_species['df']}, p={richness_lr_species['pvalue']:.4g}")
print(f"Global {richness_lr_locality['test']} test, Locality | Species: "
      f"stat={richness_lr_locality['statistic']:.3f}, df={richness_lr_locality['df']}, p={richness_lr_locality['pvalue']:.4g}")
assert _has_usable_covariance(richness_model_full), "The richness model has no usable covariance matrix."
print(f"OK -- Richness modeled with {richness_family}.")

In [ ]:
# 06.4 — Figure: residual diagnostics for the richness count model
# ----------------------------------------------------------------------
# `.predict()` rather than `.fittedvalues`: the two are not the same
# across statsmodels classes. For a GLM `.fittedvalues` is the fitted
# mean, but for the discrete NegativeBinomial class it is the linear
# predictor on the log scale, so plotting it against a "fitted richness"
# axis would silently show logs. `.predict()` returns the mean for both.
richness_fitted = np.asarray(richness_model_full.predict(), dtype=float)
assert richness_fitted.min() > 0, "Fitted richness must be positive on the response scale."
_resid_pearson = getattr(richness_model_full, "resid_pearson", None)
richness_pearson_resid = np.asarray(
    _resid_pearson if _resid_pearson is not None
    else (inter_specific_data["Richness_count"].values - richness_fitted) / np.sqrt(richness_fitted),
    dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].scatter(richness_fitted, richness_pearson_resid, alpha=0.75, s=30, color="#0072B2",
                edgecolor="white", linewidth=0.5)
axes[0].axhline(0, color="black", linewidth=1, linestyle="--")
axes[0].axhline(2, color="0.6", linewidth=0.8, linestyle=":")
axes[0].axhline(-2, color="0.6", linewidth=0.8, linestyle=":")
axes[0].set_xlabel(f"Fitted {RICHNESS_LABEL} (response scale)")
axes[0].set_ylabel("Pearson residual")
axes[0].set_title("A. Residuals vs. fitted (no funnel = variance well modeled)")
# With only categorical predictors the fitted values take one value per
# Species x Locality cell, so the points form vertical stripes. Saying so
# stops that being read as a pathology.
axes[0].annotate(f"one stripe per Species x Locality cell\n({len(np.unique(richness_fitted.round(6)))} "
                 f"distinct fitted values)",
                 xy=(0.02, 0.03), xycoords="axes fraction", fontsize=8, color="0.4", va="bottom")

for i, s in enumerate(species_present, start=1):
    mask = (inter_specific_data["Species_code"] == s).values
    jitter = spawn_rng().normal(0, 0.07, size=int(mask.sum()))
    axes[1].scatter(np.full(int(mask.sum()), i) + jitter, richness_pearson_resid[mask],
                    alpha=0.85, s=32, marker=SPECIES_MARKERS[s], color=SPECIES_COLORS[s],
                    edgecolor="white", linewidth=0.5)
axes[1].axhline(0, color="black", linewidth=1, linestyle="--")
axes[1].set_xticks(range(1, len(species_present) + 1))
axes[1].set_xticklabels([SPECIES_LABELS_ITALIC.get(s, s) for s in species_present], rotation=18, ha="right")
axes[1].set_ylabel("Pearson residual")
axes[1].set_title("B. Residuals by species (no group offset = fit is unbiased per species)")

fig.suptitle(f"Genus-richness count model ({richness_family}): residual diagnostics",
             fontsize=13, fontweight="bold")
figure_caption(fig,
    f"Diagnostics for Richness ~ Species + Locality, fitted as {richness_family} "
    f"(Poisson dispersion phi = {richness_fit['dispersion']:.2f}), N = {len(inter_specific_data)}. "
    f"Pearson residuals have mean {np.mean(richness_pearson_resid):.3f} and SD "
    f"{np.std(richness_pearson_resid):.3f}; an SD near 1 indicates the dispersion is correctly modeled.",
    y=-0.03)
plt.tight_layout()
richness_diag_path = save_figure(fig, DIAGNOSTICS_FIG_DIR / "richness_count_model_diagnostics.png")
plt.show(); plt.close(fig)

print("06.4 verification")
print("-" * 60)
print(f"Saved: {richness_diag_path}")
print(f"Pearson residual summary: mean={np.mean(richness_pearson_resid):.3f}, sd={np.std(richness_pearson_resid):.3f}")
if abs(np.std(richness_pearson_resid) - 1) > 0.5:
    print("[NOTE] Pearson residual SD is far from 1 -- the dispersion model may be inadequate.")
print("OK -- Richness residual diagnostics complete.")

In [ ]:
# 06.5 — Shannon ~ Species + Locality, Gamma GLM with a log link
# ----------------------------------------------------------------------
# Shannon diversity is modeled with a Gamma generalized linear model and
# a log link. The family is chosen from the nature of the response, not
# from a search over candidates:
#
#   Support. H' is continuous and strictly positive. A Gamma is defined
#   on (0, infinity); a Gaussian places probability below zero, which for
#   a diversity index is not a small technicality -- several species here
#   have fitted means close enough to zero for that mass to be real.
#
#   Variance. Gamma lets the spread of H' grow with its mean rather than
#   holding it constant, which suits a response bounded below by zero:
#   a group whose mean sits near the boundary cannot vary as widely as
#   one in the middle of the range. The mean-variance relationship these
#   data actually show is measured in 06.6 and reported with its
#   uncertainty, because with a handful of populated cells it is estimated
#   imprecisely and should not be leaned on harder than it deserves.
#
#   Interpretation. With a log link the coefficients are ratios of
#   ARITHMETIC means, which is the quantity the study reports. A model
#   fitted to log(H') would instead describe geometric means, and would
#   give the most strongly dominated birds -- the ones this study is
#   about -- enormous leverage, since log(H') runs to about -6 for them
#   against a mean near -0.6.
#
# Goodness of fit of this model is assessed in 06.9 with quantile
# residuals, and 06.6 shows the evidence behind the family choice.
#
# GLS is deliberately not used: there is no independently justified
# heteroscedasticity structure to put in a GLS weight matrix, so fitting
# one would be a relabeled OLS. A Gaussian model and a log-normal model
# were considered for this response and are not used: section 06.6
# quantifies how much probability a Gaussian would place at impossible
# negative diversities, and a log-normal would describe geometric rather
# than arithmetic means, giving the most strongly dominated birds -- the
# subject of this study -- extreme leverage on the log scale.
shannon_formula_full = f"Shannon ~ {SPECIES_TERM} + {LOCALITY_TERM}"
_shannon_values = inter_specific_data["Shannon"].values
assert np.all(_shannon_values > 0), \
    "Shannon must be strictly positive for a Gamma GLM; a zero would mean a single-genus community."

shannon_gamma = smf.glm(shannon_formula_full, data=inter_specific_data,
                        family=sm.families.Gamma(link=sm.families.links.Log())).fit()

shannon_model_primary = shannon_gamma
shannon_family = "Gamma (log link)"
shannon_family_key = "gamma"
shannon_transform = "exp"
shannon_effect_scale = "ratio of mean Shannon"

shannon_wald_species = wald_test_term(shannon_model_primary, SPECIES_TERM)
shannon_wald_locality = wald_test_term(shannon_model_primary, LOCALITY_TERM)

print("06.5 verification")
print("-" * 60)
print(f"Shannon on {len(inter_specific_data)} birds: min={_shannon_values.min():.4f}, "
      f"max={_shannon_values.max():.4f} -- continuous and strictly positive.")
print(f">>> Reported model: {shannon_family}")
print(shannon_model_primary.summary())
print(f"\nGlobal {shannon_wald_species['test']} test, Species | Locality: "
      f"stat={shannon_wald_species['statistic']:.3f}, df={shannon_wald_species['df']}, "
      f"p={shannon_wald_species['pvalue']:.4g}")
print(f"Global {shannon_wald_locality['test']} test, Locality | Species: "
      f"stat={shannon_wald_locality['statistic']:.3f}, df={shannon_wald_locality['df']}, "
      f"p={shannon_wald_locality['pvalue']:.4g}")
print(f"  (statistic referred to the {shannon_wald_species['distribution']} distribution by statsmodels "
      f"for this fit -- stated explicitly so the test can be reproduced from the table)")

assert np.isfinite(shannon_wald_species["pvalue"]) and np.isfinite(shannon_wald_locality["pvalue"])
print(f"\nOK -- Shannon modeled with {shannon_family}.")

In [ ]:
# 06.6 — Figure: the evidence for modeling Shannon diversity as a Gamma
# ---------------------------------------------------------------------
# The decisive property of Shannon diversity here is its support. H' is
# bounded below by zero and several species have fitted means close to
# that bound, so a Gaussian model -- which places a normal distribution
# around each fitted value -- puts a substantial share of its predictive
# mass at negative diversities. That is not a technicality to be waved
# through with robust standard errors: a model that assigns probability
# to impossible values is describing something other than these data.
# Panel A quantifies exactly how much.
#
# Panel B looks at the variance function, which is the other thing the
# family choice asserts. With only a handful of populated Species x
# Locality cells the slope is estimated imprecisely, so it is shown with
# its uncertainty and treated as corroboration rather than as proof.
#
# Panel C shows influence on the model actually reported.
MIN_CELL_N_FOR_VARIANCE = 3

# -- What a Gaussian would imply, per species --
# The Gaussian fit is made here and used only as the object of this
# diagnostic: it is the model being ruled out, so it needs to be fitted in
# order to show what it would claim, and it is reported nowhere else.
_shannon_gaussian_for_diagnostic = smf.ols(shannon_formula_full, data=inter_specific_data).fit()
_gauss_fitted = np.asarray(_shannon_gaussian_for_diagnostic.fittedvalues, dtype=float)
_gauss_sigma = float(np.sqrt(_shannon_gaussian_for_diagnostic.mse_resid))
_p_negative = pd.DataFrame({
    "Species_code": inter_specific_data["Species_code"].values,
    "Fitted": _gauss_fitted,
    "P_below_zero": norm_dist.cdf(0.0, loc=_gauss_fitted, scale=_gauss_sigma),
}).groupby("Species_code", observed=True).agg(
    Fitted_mean=("Fitted", "mean"), P_below_zero=("P_below_zero", "mean")).reindex(
    [sp for sp in SPECIES_ORDER if sp in set(inter_specific_data["Species_code"])])

# -- The mean-variance relationship, from the cells themselves --
_cells = (inter_specific_data.groupby(["Species_code", "Locality"], observed=True)["Shannon"]
          .agg(["mean", "var", "size"]).dropna())
_cells = _cells[(_cells["size"] >= MIN_CELL_N_FOR_VARIANCE) & (_cells["mean"] > 0) & (_cells["var"] > 0)]
_log_mean, _log_var = np.log(_cells["mean"].values), np.log(_cells["var"].values)
variance_slope, variance_intercept = np.polyfit(_log_mean, _log_var, 1)
_slope_se = float(np.sqrt(np.sum((_log_var - (variance_slope * _log_mean + variance_intercept)) ** 2)
                          / max(len(_log_mean) - 2, 1) / np.sum((_log_mean - _log_mean.mean()) ** 2)))
_slope_ci = (variance_slope - 1.96 * _slope_se, variance_slope + 1.96 * _slope_se)

shannon_cooks_d = shannon_model_primary.get_influence().cooks_distance[0]
influence_threshold = 4 / len(inter_specific_data)
n_influential = int((shannon_cooks_d > influence_threshold).sum())

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.0))

# Panel A: the share of a Gaussian's predictive mass at impossible values.
ax = axes[0]
_order_a = list(_p_negative.index)
_bars = ax.bar(range(len(_order_a)), 100 * _p_negative["P_below_zero"].values,
               color=[SPECIES_COLORS[s] for s in _order_a], edgecolor="white", linewidth=0.8, alpha=0.85)
for _i, _v in enumerate(100 * _p_negative["P_below_zero"].values):
    ax.annotate(f"{_v:.0f}%", xy=(_i, _v), xytext=(0, 3), textcoords="offset points",
                ha="center", fontsize=9, fontweight="bold", color="0.25")
ax.set_xticks(range(len(_order_a)))
ax.set_xticklabels([f"{abbreviated_binomial(s)}\n$\\hat{{\\mu}}$ = {_p_negative.loc[s, 'Fitted_mean']:.2f}"
                    for s in _order_a], fontsize=8)
ax.set_ylabel("Predictive mass below H' = 0 (%)")
ax.set_title(f"A. A Gaussian predicts impossible diversities\n"
             f"$\\hat{{\\sigma}}$ = {_gauss_sigma:.2f} against fitted means of "
             f"{_p_negative['Fitted_mean'].min():.2f}-{_p_negative['Fitted_mean'].max():.2f}", fontsize=10.5)
ax.set_ylim(0, max(100 * _p_negative["P_below_zero"].max() * 1.25, 5))

# Panel B: the measured variance function, with its uncertainty.
ax = axes[1]
for _species in species_present:
    _rows = [(m, v, n) for (sp, _loc), (m, v, n) in _cells.iterrows() if sp == _species]
    if not _rows:
        continue
    _m, _v, _n = zip(*_rows)
    ax.scatter(_m, _v, s=[20 + 6 * k for k in _n], marker=SPECIES_MARKERS[_species],
               color=SPECIES_COLORS[_species], edgecolor="white", linewidth=0.6, alpha=0.9, zorder=3)
_grid = np.linspace(_log_mean.min() - 0.15, _log_mean.max() + 0.15, 50)
ax.plot(np.exp(_grid), np.exp(variance_intercept + variance_slope * _grid),
        color="black", linewidth=1.8, zorder=2,
        label=f"fitted {variance_slope:.2f} (95% CI {_slope_ci[0]:.2f} to {_slope_ci[1]:.2f})")
for _ref, _style, _name in [(2.0, "--", "Gamma: slope 2"), (0.0, "-.", "Gaussian: slope 0")]:
    _c = np.mean(_log_var) - _ref * np.mean(_log_mean)
    ax.plot(np.exp(_grid), np.exp(_c + _ref * _grid), color="0.55", linestyle=_style,
            linewidth=1.2, zorder=1, label=_name)
ax.set_xscale("log"); ax.set_yscale("log")
# The cell means span well under a decade, so the default log locator can
# leave the axis with a single labeled tick. Labelling the minor ticks as
# well keeps the axis readable without abandoning the log scale the
# power-law fit needs.
ax.xaxis.set_minor_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
ax.tick_params(axis="x", which="minor", labelsize=8)
ax.set_xlabel("Mean H' in a Species x Locality cell")
ax.set_ylabel("Variance of H' in that cell")
ax.set_title(f"B. Variance against mean ({len(_cells)} cells)\nslope estimated imprecisely at this n",
             fontsize=10.5)
ax.legend(fontsize=7.5, loc="best", framealpha=0.92)

# Panel C: influence on the reported model.
ax = axes[2]
ax.vlines(np.arange(len(shannon_cooks_d)), 0, shannon_cooks_d, color="#0072B2", linewidth=1.1)
ax.axhline(influence_threshold, color="#D55E00", linewidth=1.3, linestyle="--",
           label=f"4/n = {influence_threshold:.3f}")
_worst = int(np.argmax(shannon_cooks_d))
ax.annotate(f"largest {shannon_cooks_d[_worst]:.3f}\n({inter_specific_data['Species_code'].iloc[_worst]})",
            xy=(_worst, shannon_cooks_d[_worst]), xytext=(8, -2), textcoords="offset points",
            fontsize=8, color="0.3")
ax.set_xlabel("Bird (row of the modeling dataset)")
ax.set_ylabel("Cook's distance")
ax.set_title(f"C. Influence on the reported Gamma model\n{n_influential}/{len(inter_specific_data)} "
             f"above 4/n, none near 1", fontsize=10.5)
ax.legend(fontsize=8.5)

fig.suptitle("Why Shannon diversity is modeled with a Gamma GLM rather than a Gaussian one",
             fontsize=13.5, fontweight="bold", y=1.02)
figure_caption(fig,
    f"A: for each host species, the share of the predictive distribution of an ordinary-least-squares "
    f"fit that falls below zero, computed as Phi(-mu_hat/sigma_hat) with sigma_hat = "
    f"{_gauss_sigma:.2f} averaged over that species' birds. Shannon diversity cannot be negative, so "
    f"this is mass the Gaussian model spends on outcomes that cannot occur -- most heavily for the "
    f"species with the lowest diversity, which are the ones this study is about. A Gamma is defined on "
    f"(0, infinity) and spends none. B: mean against variance across Species x Locality cells with at "
    f"least {MIN_CELL_N_FOR_VARIANCE} birds, on log-log axes; the fitted slope is {variance_slope:.2f} "
    f"(95% CI {_slope_ci[0]:.2f} to {_slope_ci[1]:.2f}), which with {len(_cells)} cells is too "
    f"imprecise to distinguish the candidate variance functions on its own and is reported as "
    f"corroboration rather than as the basis of the choice. C: Cook's distance for the Gamma GLM "
    f"reported in 06.5; the 4/n line is a screening threshold, and what matters is that no bird "
    f"approaches 1.",
    y=-0.08)
plt.tight_layout()
shannon_diag_path = save_figure(fig, DIAGNOSTICS_FIG_DIR / "shannon_family_justification.png")
plt.show(); plt.close(fig)

print("06.6 verification")
print("-" * 60)
print("What an ordinary-least-squares fit of the same formula would imply, by species:")
print(_p_negative.assign(Pct_below_zero=lambda d: (100 * d["P_below_zero"]).round(1))
      [["Fitted_mean", "Pct_below_zero"]].round(3).to_string())
print(f"  Residual SD of that fit: {_gauss_sigma:.3f}")
_worst_species = _p_negative["P_below_zero"].idxmax()
print(f"  -> For {_worst_species}, {100 * _p_negative.loc[_worst_species, 'P_below_zero']:.0f}% of the "
      f"Gaussian's predictive mass sits below zero, at diversities that cannot exist. This is the "
      f"clearest reason the response is not modeled as Gaussian, and it does not depend on any test.")

print(f"\nMean-variance relationship across {len(_cells)} Species x Locality cells:")
print(_cells.round(4).to_string())
print(f"  slope on log-log axes = {variance_slope:.3f} (SE {_slope_se:.3f}, "
      f"95% CI {_slope_ci[0]:.2f} to {_slope_ci[1]:.2f})")
if _slope_ci[0] <= 2.0 <= _slope_ci[1]:
    print("  -> Consistent with the Gamma variance function, though the interval is wide.")
elif variance_slope > 0.5:
    print("  -> The variance does increase with the mean, but the interval does not reach 2. Report the "
          "slope as measured: the family rests on the support of the response and on the fit "
          "comparison in 06.5, not on this slope.")
else:
    print("  [NOTE] No clear increase of variance with the mean at this sample size. Report the slope "
          "honestly; the case for the Gamma family then rests on the support of the response and on "
          "the fit comparison in 06.5.")

print(f"\nInfluence on the reported model: {n_influential}/{len(inter_specific_data)} birds above 4/n, "
      f"largest Cook's distance {shannon_cooks_d.max():.3f}")
if shannon_cooks_d.max() < 1.0:
    print("  No single bird approaches Cook's D = 1, so no individual observation drives the fit.")
else:
    print("  [WARNING] At least one bird has Cook's D >= 1 and materially moves the fit; refit without "
          "it and report whether the conclusion changes.")
assert shannon_diag_path.exists()
assert len(_cells) >= 3, "Too few populated cells to estimate a mean-variance relationship."
print(f"\nSaved: {shannon_diag_path}")
print("OK -- family justification and influence complete.")

In [ ]:
# 06.7 — Simpson ~ Species + Locality, beta regression with a tested precision submodel
# ----------------------------------------------------------------------
# Simpson lies strictly inside (0, 1) (support report, 05.3), so beta
# regression, not OLS. The precision submodel carries the same
# Species + Locality terms as the mean submodel by default; 06.8 tests
# formally whether that complexity is needed, rather than assuming
# constant precision.
#
# Both precision formulas are spelled out as plain strings containing no
# notebook-level variables. `BetaModel.from_formula` evaluates
# `exog_precision_formula` in its own namespace, where the notebook's
# globals are not visible, so a formula that referred to one would fail
# there even though the identical formula works for the mean submodel.
beta_precision_formula_varying = f"~ {SPECIES_TERM} + {LOCALITY_TERM}"
beta_precision_formula_constant = "~ 1"


def beta_mean_formula(metric: str, terms: list[str]) -> str:
    return f"{metric} ~ " + (" + ".join(terms) if terms else "1")


# Step 1, per metric: fit the full mean submodel twice -- once with a
# varying precision submodel, once with constant precision -- and let a
# likelihood-ratio test decide which precision specification to carry
# forward into the mean-effect tests. Fixing the precision submodel
# BEFORE testing mean effects is the right model-building order, and it
# is also what keeps 06.8 numerically stable: pairing a reduced mean
# submodel with a full varying-precision submodel is a specification
# neither model-building order arrives at, and the optimizer handles it
# poorly.
beta_precision_choice, beta_models = {}, {}
for _metric in BETA_RESPONSE_METRICS:
    _terms = [SPECIES_TERM, LOCALITY_TERM]
    varying = fit_beta_model(beta_mean_formula(_metric, _terms), inter_specific_data,
                             beta_precision_formula_varying, required=False)
    constant = fit_beta_model(beta_mean_formula(_metric, _terms), inter_specific_data,
                              beta_precision_formula_constant, required=True)
    precision_test = lr_test(varying, constant)
    use_varying = bool(np.isfinite(precision_test["pvalue"]) and precision_test["pvalue"] < ALPHA_SIGNIFICANCE)
    beta_precision_choice[_metric] = {
        "formula": beta_precision_formula_varying if use_varying else beta_precision_formula_constant,
        "label": "varying by Species + Locality" if use_varying else "constant",
        "test": precision_test,
    }
    beta_models[_metric] = varying if use_varying else constant

print("06.7 verification")
print("-" * 60)
print("Precision submodel, tested rather than assumed:")
for _metric in BETA_RESPONSE_METRICS:
    _c = beta_precision_choice[_metric]; _t = _c["test"]
    print(f"  {_metric}: LR varying vs constant = {_t['statistic']:.3f}, df={_t['df']}, "
          f"p={_t['pvalue']:.4g}  ->  precision used: {_c['label']}")
for _metric, _model in beta_models.items():
    print(f"\n===== {_metric}: Beta regression (logit mean, log precision; "
          f"precision {beta_precision_choice[_metric]['label']}) =====")
    print(_model.summary())
    assert np.isfinite(_model.llf), f"{_metric}: Beta model did not produce a finite log-likelihood."
print(f"\nOK -- {BETA_RESPONSE_METRICS} fitted by beta regression; precision submodel chosen by a "
      f"likelihood-ratio test.")

In [ ]:
# 06.8 — Likelihood-ratio tests for the beta-regression mean submodels
# ----------------------------------------------------------------------
# With the precision submodel fixed in 06.7, each mean-submodel term is
# tested against the same model with that term dropped from the MEAN
# submodel only. A model that will not converge yields a NaN p-value,
# which is excluded from the FDR correction and reported as a failure --
# never as a p-value that would read as "no effect".
beta_tests = {}
for _metric in BETA_RESPONSE_METRICS:
    _prec = beta_precision_choice[_metric]["formula"]
    _full = beta_models[_metric]
    _reduced_species = fit_beta_model(beta_mean_formula(_metric, [LOCALITY_TERM]),
                                      inter_specific_data, _prec, required=False)
    _reduced_locality = fit_beta_model(beta_mean_formula(_metric, [SPECIES_TERM]),
                                       inter_specific_data, _prec, required=False)
    beta_tests[_metric] = {"species": lr_test(_full, _reduced_species),
                           "locality": lr_test(_full, _reduced_locality)}

beta_lr_species = {m: beta_tests[m]["species"] for m in BETA_RESPONSE_METRICS}
beta_lr_locality = {m: beta_tests[m]["locality"] for m in BETA_RESPONSE_METRICS}

print("06.8 verification")
print("-" * 60)
for _metric, _t in beta_tests.items():
    print(f"{_metric} (precision {beta_precision_choice[_metric]['label']}):")
    print(f"  LR Species | Locality : stat={_t['species']['statistic']:.3f} df={_t['species']['df']} "
          f"p={_t['species']['pvalue']:.4g}")
    print(f"  LR Locality | Species : stat={_t['locality']['statistic']:.3f} df={_t['locality']['df']} "
          f"p={_t['locality']['pvalue']:.4g}")
assert set(beta_tests) == set(BETA_RESPONSE_METRICS)
print(f"OK -- global LR tests complete for {BETA_RESPONSE_METRICS}.")

In [ ]:
# 06.9 — Goodness of fit: quantile residuals for every reported model
# -------------------------------------------------------------------
# Each response is modeled with the distribution its support calls for,
# so their raw and Pearson residuals are not on a common footing and none
# of them is expected to be normal even when the model is right. Quantile
# residuals put them all on the same scale: r = Phi^-1(F(y; theta)) is
# exactly standard normal whenever the fitted distribution is the correct
# one, so a single Q-Q plot and a single Shapiro-Wilk test are valid for
# the count model, the Gamma model and the beta model alike.
#
# The models checked here are the ones reported in 06.3 to 06.7. The
# least-squares diagnostics in 06.6 serve a different purpose: they are
# the evidence on which the Shannon family was chosen, and the fit they
# examine is the one that choice rules out.
_qres_rng = spawn_rng()

_DIAGNOSTIC_SOURCE = {
    "Richness": lambda: (richness_model_full, richness_family_key, "Richness_count"),
    "Shannon": lambda: (shannon_model_primary, shannon_family_key, "Shannon"),
    **{_m: (lambda _m=_m: (beta_models[_m], "beta", _m)) for _m in BETA_RESPONSE_METRICS},
}
diagnostic_models = []
for _metric in ALPHA_METRICS:
    _model, _fam, _column = _DIAGNOSTIC_SOURCE[_metric]()
    diagnostic_models.append((_metric, _model, _fam, inter_specific_data[_column].values))
assert len(diagnostic_models) == len(ALPHA_METRICS), \
    "Every modeled metric needs a goodness-of-fit panel."

quantile_residual_table, quantile_residual_values = [], {}
for _metric, _model, _fam, _y in diagnostic_models:
    _check = model_residual_check(_model, _y, _fam, richness_fit["dispersion"], _qres_rng)
    quantile_residual_values[_metric] = _check["residuals"]
    quantile_residual_table.append({"Metric": _metric, "Family": _fam, "Residual": _check["kind"],
                                    **{k: v for k, v in _check.items()
                                       if k not in ("residuals", "kind")}})
quantile_residual_table = pd.DataFrame(quantile_residual_table)
quantile_residual_table.to_csv(STATISTICS_DIR / "model_goodness_of_fit.csv", index=False)

# When every model yields the same kind of residual, one y-label on the
# left of each row is enough. When they are mixed -- a quasi-likelihood
# among likelihood fits -- every panel is labeled, so that a label on
# the leftmost panel alone cannot be read as applying to the whole row.
_residual_kinds = {row["Residual"] for row in quantile_residual_table.to_dict("records")}
_label_every_panel = len(_residual_kinds) > 1

_n_panels = len(diagnostic_models)
fig, axes = plt.subplots(2, _n_panels, figsize=(4.4 * _n_panels, 8.2), squeeze=False)
for _col, (_metric, _model, _fam, _y) in enumerate(diagnostic_models):
    _r = quantile_residual_values[_metric]
    _row = quantile_residual_table.set_index("Metric").loc[_metric]

    # Top: Q-Q against the standard normal the residuals should follow.
    ax = axes[0, _col]
    _theoretical = norm_dist.ppf((np.arange(1, len(_r) + 1) - 0.5) / len(_r))
    ax.scatter(_theoretical, np.sort(_r), s=22, color="#0072B2",
               edgecolor="white", linewidth=0.4, zorder=3)
    _lim = [min(_theoretical.min(), _r.min()) - 0.3, max(_theoretical.max(), _r.max()) + 0.3]
    ax.plot(_lim, _lim, color="#D55E00", linewidth=1.5, zorder=2)
    ax.set_xlim(_lim); ax.set_ylim(_lim)
    ax.set_xlabel("Theoretical normal quantiles")
    _res_name = "Pearson residual" if not np.isfinite(_row["pvalue"]) else "Quantile residual"
    ax.set_ylabel(_res_name if (_col == 0 or _label_every_panel) else "")
    if not np.isfinite(_row["pvalue"]):
        # Quasi-likelihood: Pearson residuals are plotted instead, and a
        # normality test on them would not mean anything.
        ax.set_title(f"{_metric} - {_fam}\nno distribution function; Pearson residuals shown",
                     fontsize=10.5)
    else:
        _verdict = "consistent" if _row["pvalue"] >= ALPHA_SIGNIFICANCE else "departs"
        ax.set_title(f"{_metric} - {_fam}\nShapiro-Wilk {format_p(_row['pvalue'])} ({_verdict})",
                     fontsize=10.5)

    # Bottom: residuals against the fitted mean, on the response scale.
    ax = axes[1, _col]
    if _fam == "beta":
        _mu = np.asarray(_model.model.predict(_model.params, which="mean"), dtype=float)
    elif _fam == "lognormal":
        # Fitted on the log scale; exponentiating returns the fitted
        # geometric mean, which is the quantity this family models.
        _mu = np.exp(np.asarray(_model.fittedvalues, dtype=float))
    elif _fam == "gaussian":
        _mu = np.asarray(_model.fittedvalues, dtype=float)
    else:
        _mu = np.asarray(_model.predict(), dtype=float)
    for _s in species_present:
        _m = (inter_specific_data["Species_code"] == _s).values
        ax.scatter(_mu[_m], _r[_m], s=26, marker=SPECIES_MARKERS[_s], color=SPECIES_COLORS[_s],
                   edgecolor="white", linewidth=0.4, alpha=0.9, zorder=3)
    ax.axhline(0, color="black", linewidth=1, linestyle="--")
    for _h in (-2, 2):
        ax.axhline(_h, color="0.7", linewidth=0.8, linestyle=":")
    if _metric == "Richness":
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
    # "Richness" reads naturally in lower case, the two index names do not,
    # so the metric name is used as it is written everywhere else.
    ax.set_xlabel(f"Fitted {'richness' if _metric == 'Richness' else _metric}"
                  + (" (geometric mean)" if _fam == "lognormal" else ""))
    ax.set_ylabel(_res_name if (_col == 0 or _label_every_panel) else "")
    _outside = int((np.abs(_r) > 1.959964).sum())
    ax.set_title(f"{100 * _outside / len(_r):.1f}% of residuals beyond +/-1.96\n(5% expected)", fontsize=10)

handles = [Line2D([0], [0], marker=SPECIES_MARKERS[s], color="none", markerfacecolor=SPECIES_COLORS[s],
                  markeredgecolor="white", markersize=9, label=species_legend_label(s))
           for s in species_present]
fig.legend(handles=handles, loc="lower center", ncol=len(species_present), bbox_to_anchor=(0.5, -0.05),
           title="Species (origin)", title_fontsize=9)
fig.suptitle("Goodness of fit of the reported diversity models: randomized quantile residuals",
             fontsize=14, fontweight="bold", y=1.0)
figure_caption(fig,
    f"Quantile residuals (Dunn & Smyth 1996) for the models reported in sections 06.3-06.7, "
    f"N = {len(inter_specific_data)} birds. r = Phi^-1(F(y; theta)) is exactly standard normal when the "
    f"fitted distribution is correct, whatever that distribution is, so the responses can be judged "
    f"on the same scale. Top row: Q-Q against the standard normal, with the identity line; points on the "
    f"line mean the family and its variance function describe the data. Bottom row: the same residuals "
    f"against the fitted mean, which exposes any remaining trend in location or spread that the Q-Q plot "
    f"would not show; dotted lines mark +/-1.96, inside which 95% of residuals should fall. A discrete "
    f"response needs the residual to be randomized across the jump in its distribution function, so the "
    f"count panel changes slightly between runs; the continuous responses do not.",
    y=-0.09)
plt.tight_layout()
gof_fig_path = save_figure(fig, DIAGNOSTICS_FIG_DIR / "model_goodness_of_fit.png")
plt.show(); plt.close(fig)

print("06.9 verification")
print("-" * 60)

_testable = quantile_residual_table[quantile_residual_table["pvalue"].notna()]
_poor = _testable[_testable["pvalue"] < ALPHA_SIGNIFICANCE]
_untestable = quantile_residual_table[quantile_residual_table["pvalue"].isna()]["Metric"].tolist()
if _untestable:
    print(f"\n[NOTE] {_untestable} are fitted by quasi-likelihood, which specifies a mean and a variance "
          f"but no distribution, so no quantile residual exists for them. Their Pearson residuals are "
          f"reported instead: an SD near 1 means the variance function is right, which is the whole of "
          f"what a quasi-likelihood claims.")
if _poor.empty:
    print(f"\nEvery model with a distribution to test is consistent with it (Shapiro-Wilk "
          f"p >= {ALPHA_SIGNIFICANCE} on the quantile residuals). The distributional assumptions behind "
          f"the tests in 06.10 hold.")
else:
    print(f"\n[WARNING] {_poor['Metric'].tolist()} depart from their fitted distribution. Their p-values "
          f"rest on an assumption the data do not support, and the effect sizes are more trustworthy than "
          f"the tests. Report this, and lean on the metrics whose residuals behave.")

assert quantile_residual_table["n"].eq(len(inter_specific_data)).all(), \
    "A quantile-residual set does not cover every modeled bird."
assert np.isfinite(quantile_residual_table["sd"]).all()
print(f"\nSaved: {gof_fig_path}")
print(f"Saved: {STATISTICS_DIR / 'model_goodness_of_fit.csv'}")
print(f"OK -- goodness-of-fit diagnostics complete for all {len(diagnostic_models)} reported models.")

In [ ]:
# Table · 06.9 — Goodness of fit of every between-species model
show_table(quantile_residual_table, "Goodness of fit of every reported between-species model, on inter_specific_data. A Shapiro-Wilk p at or above the significance level means the residuals are consistent with the fitted distribution.")

In [ ]:
# 06.10 — Global Species and Locality tests across the metrics, with FDR within each family
# ----------------------------------------------------------------------
# ONE family = "the global Species tests of the inter-specific model",
# one q-value per modeled metric. Explicitly NOT pooled with the
# intra-specific tests in section 07.0, which form their own separate
# families.
#
# The metrics are not independent: Simpson and Shannon are both functions
# of the same abundance vector, and richness enters both of them.
# Benjamini-Hochberg is valid under positive dependence, which is exactly
# this situation, so it remains appropriate; but the q-values should be
# read as several views of one biological question, not as independent
# findings.
# Assembled from ALPHA_METRICS rather than written out metric by metric,
# so that the family of tests the correction is applied to is exactly the
# set of metrics this notebook models.
METRIC_TEST_LABEL = {
    "Richness": f"Richness ({'NB' if richness_fit['fitter'] == 'nb' else 'Poisson'}, {richness_lr_species['test']})",
    "Shannon": f"Shannon (Gamma, {shannon_wald_species['test']})",
    **{m: f"{m} (Beta, LR)" for m in BETA_RESPONSE_METRICS},
}
_species_tests_by_metric = {"Richness": richness_lr_species, "Shannon": shannon_wald_species,
                            **beta_lr_species}
_locality_tests_by_metric = {"Richness": richness_lr_locality, "Shannon": shannon_wald_locality,
                             **beta_lr_locality}
_stats_species = {METRIC_TEST_LABEL[m]: _species_tests_by_metric[m] for m in ALPHA_METRICS}
_stats_locality = {METRIC_TEST_LABEL[m]: _locality_tests_by_metric[m] for m in ALPHA_METRICS}
global_tests_species = {k: v["pvalue"] for k, v in _stats_species.items()}
global_tests_locality = {k: v["pvalue"] for k, v in _stats_locality.items()}

global_fdr_species = fdr_within_family(global_tests_species)
global_fdr_locality = fdr_within_family(global_tests_locality)
for _tab, _src, _term in [(global_fdr_species, _stats_species, "Species_code"),
                          (global_fdr_locality, _stats_locality, "Locality")]:
    _tab.insert(0, "Term", _term)
    _tab.insert(3, "statistic", [_src[c]["statistic"] for c in _tab["Comparison"]])
    _tab.insert(4, "df", [_src[c]["df"] for c in _tab["Comparison"]])
    _tab["Metric"] = [c.split(" ")[0] for c in _tab["Comparison"]]

global_effect_summary = pd.concat([global_fdr_species, global_fdr_locality], ignore_index=True)
global_effect_summary.to_csv(STATISTICS_DIR / "inter_specific_global_effect_tests.csv", index=False)

print("06.10 verification")
print("-" * 60)
print(f"Global effect tests, inter-specific model (N = {len(inter_specific_data)}).")
assert len(global_effect_summary) == 2 * len(ALPHA_METRICS)
print(f"\nSaved: {STATISTICS_DIR / 'inter_specific_global_effect_tests.csv'}")
print("OK -- post-hoc comparisons (06.11) run only for metrics with Significant_FDR == True.")

In [ ]:
# Table · 06.10 — Global Species and Locality tests for each modeled metric
show_table(global_effect_summary, "Global Species and Locality tests for each modeled metric on inter_specific_data, with Benjamini-Hochberg q-values computed within each term's own family.")

In [ ]:
# 06.11 — Post-hoc pairwise species comparisons, where the global test survives correction
# ----------------------------------------------------------------------
_posthoc_rng = spawn_rng()
pairs = list(combinations(species_present, 2))
metric_significant = global_fdr_species.set_index("Metric")["Significant_FDR"].to_dict()

# (model, kind, effect-scale label) per metric. `kind` decides how the
# contrast is computed and back-transformed.
POSTHOC_SPEC = {
    "Richness": (richness_model_full, "exp", "rate ratio of genus richness"),
    "Shannon": (shannon_model_primary, shannon_transform, shannon_effect_scale),
    **{m: (beta_models[m], "beta", f"difference in mean {m} (0-1 scale)")
       for m in BETA_RESPONSE_METRICS},
}
assert set(POSTHOC_SPEC) == set(ALPHA_METRICS), \
    "Every modeled metric needs a post-hoc specification, and no others."

pairwise_records = []
for metric, (model, kind, scale_label) in POSTHOC_SPEC.items():
    if not metric_significant.get(metric, False):
        continue
    for a, b in pairs:
        if kind == "beta":
            res = beta_contrast(model, a, b, SPECIES_TERM, _posthoc_rng)
        else:
            names = list(model.params.index)
            c = level_contrast_vector(names, SPECIES_TERM, a, b)
            res = wald_contrast(model.params.values, model.cov_params().values, c, transform=kind)
        pairwise_records.append({"Metric": metric, "Comparison": f"{a} vs {b}",
                                 "Scale": scale_label, **res})

pairwise_table = pd.DataFrame(pairwise_records)

print("06.11 verification")
print("-" * 60)
if pairwise_table.empty:
    print("No metric had a significant (FDR) global Species effect -- no pairwise comparisons, by design.")
else:
    # FDR within each metric's own pairwise family, never pooled across metrics.
    adjusted_blocks = []
    for metric, block in pairwise_table.groupby("Metric", sort=False):
        qtab = fdr_within_family(dict(zip(block["Comparison"], block["pvalue"])))
        block = block.merge(qtab[["Comparison", "q_value", "Significant_FDR"]], on="Comparison", how="left")
        adjusted_blocks.append(block)
    pairwise_table = pd.concat(adjusted_blocks, ignore_index=True)
    pairwise_table["Stars"] = pairwise_table["q_value"].map(significance_stars)
    print(f"{len(pairs)} pairwise comparisons per significant metric; "
          f"FDR corrected within each metric's own family.")
    # A whole table of exactly zero estimates would mean the contrast
    # vectors matched no coefficients.
    assert not ((pairwise_table["estimate"] == 0).all()), \
        "All pairwise estimates are exactly 0 -- the contrast vectors matched no coefficients."
pairwise_table.to_csv(STATISTICS_DIR / "inter_specific_posthoc_pairwise.csv", index=False)
print(f"\nSaved: {STATISTICS_DIR / 'inter_specific_posthoc_pairwise.csv'}")
print("OK -- post-hoc pairwise comparisons complete.")

In [ ]:
# Table · 06.11 — Post-hoc pairwise species comparisons
show_table(pairwise_table, "Post-hoc pairwise species comparisons on inter_specific_data, FDR-adjusted within each metric. Search a species code to pull out every comparison it takes part in.", page_length=15)

In [ ]:
# 06.12 — Figure: estimated marginal mean diversity per host species, with a compact letter display
# ----------------------------------------------------------------------
# The main between-species figure. It shows where each species sits
# rather than how far each one lies from an arbitrary baseline: the
# model-based estimated marginal mean per species, averaged over
# localities and back-transformed to the response scale, with its 95%
# interval. The pairwise post-hoc result is encoded as a compact letter
# display, so that the level of each species and the groupings among them
# can be read off the same panel.
#
# Plotting contrasts against a reference species instead would make the
# figure depend on which species that is. A joint test over several
# coefficients that share one baseline can be far more significant than
# any of those coefficients individually, so a panel of contrasts can
# show intervals straddling the null under a highly significant global
# test -- both correct, but not readable together.

def estimated_marginal_means(model, kind: str, species_list: list[str],
                             locality_levels: list[str]) -> pd.DataFrame:
    """Fitted value per species, averaged over localities, on the response scale.

    Built as a linear contrast on the link scale (intercept + species
    dummy + the MEAN of the locality dummies, i.e. a balanced average
    over localities rather than one weighted by this study's unequal
    sampling), then back-transformed. The interval is computed on the
    link scale and back-transformed too, so it cannot fall outside the
    response's support -- a richness interval never goes negative and a
    Simpson interval never leaves (0, 1).
    """
    if kind == "beta":
        params, cov, names = beta_mean_params(model)
    else:
        params, cov, names = (np.asarray(model.params, dtype=float),
                              np.asarray(model.cov_params(), dtype=float),
                              list(model.params.index))
    # The NB fit carries an extra dispersion parameter (alpha) after the
    # regression coefficients; it is not part of the linear predictor.
    k = len(names)
    if params.shape[0] > k:
        params, cov = params[:k], cov[:k, :k]
    names_no_alpha = [n for n in names if n != "alpha"]
    if len(names_no_alpha) < len(names):
        keep = [i for i, n in enumerate(names) if n != "alpha"]
        params, cov, names = params[keep], cov[np.ix_(keep, keep)], names_no_alpha

    non_reference_localities = [l for l in locality_levels
                                if f"{LOCALITY_TERM}[T.{l}]" in names]
    inv = {"exp": np.exp, "beta": lambda v: 1.0 / (1.0 + np.exp(-v)), "identity": lambda v: v}
    back = inv["beta"] if kind == "beta" else inv.get(kind, inv["identity"])

    rows = []
    for s in species_list:
        c = np.zeros(len(names))
        c[names.index("Intercept")] = 1.0
        key = f"{SPECIES_TERM}[T.{s}]"
        if key in names:
            c[names.index(key)] = 1.0
        # Balanced average over localities: every non-reference locality
        # dummy gets 1/n_localities (the reference locality contributes 0).
        if non_reference_localities:
            w = 1.0 / (len(non_reference_localities) + 1)
            for l in non_reference_localities:
                c[names.index(f"{LOCALITY_TERM}[T.{l}]")] = w
        link = float(c @ params)
        se = float(np.sqrt(c @ cov @ c))
        rows.append({"Species": s, "estimate": float(back(link)),
                     "ci_low": float(back(link - 1.959964 * se)),
                     "ci_high": float(back(link + 1.959964 * se)),
                     "link_estimate": link, "link_se": se})
    return pd.DataFrame(rows).set_index("Species")


def compact_letter_display(levels: list[str], not_different: set[frozenset],
                           rank_value: dict) -> dict:
    """Assign letters so that two levels share a letter iff they are NOT significantly different.

    Each letter is a maximal clique in the "not significantly different"
    graph, which is the standard construction. Cliques are found with a
    plain Bron-Kerbosch enumeration -- with a handful of species the graph
    is tiny, so there is no reason to approximate. Letters are then ordered
    by the group's mean rank value, so that 'a' is the high-diversity end
    and the display reads left to right like the data.
    """
    adjacency = {l: {m for m in levels if m != l and frozenset((l, m)) in not_different}
                 for l in levels}

    cliques = []
    def expand(current: set, candidates: set, excluded: set):
        if not candidates and not excluded:
            cliques.append(set(current))
            return
        pivot = max(candidates | excluded, key=lambda v: len(adjacency[v])) if (candidates | excluded) else None
        for v in list(candidates - (adjacency[pivot] if pivot else set())):
            expand(current | {v}, candidates & adjacency[v], excluded & adjacency[v])
            candidates = candidates - {v}
            excluded = excluded | {v}
    expand(set(), set(levels), set())

    # A level that differs from everything forms its own singleton clique.
    for l in levels:
        if not any(l in c for c in cliques):
            cliques.append({l})
    cliques.sort(key=lambda c: -np.mean([rank_value.get(l, 0.0) for l in c]))

    letters = {l: "" for l in levels}
    for i, clique in enumerate(cliques):
        letter = chr(ord("a") + i)
        for l in clique:
            letters[l] += letter
    return {l: "".join(sorted(v)) for l, v in letters.items()}


_locality_levels = list(inter_specific_data["Locality"].cat.categories)
emm_tables, cld_by_metric = {}, {}
for metric, (model, kind, scale_label) in POSTHOC_SPEC.items():
    emm = estimated_marginal_means(model, kind, species_present, _locality_levels)
    emm_tables[metric] = emm
    if not pairwise_table.empty and metric in set(pairwise_table["Metric"]):
        block = pairwise_table[pairwise_table["Metric"] == metric]
        not_diff = {frozenset(row["Comparison"].split(" vs "))
                    for _, row in block.iterrows() if not bool(row["Significant_FDR"])}
        cld_by_metric[metric] = compact_letter_display(
            species_present, not_diff, emm["estimate"].to_dict())
    else:
        cld_by_metric[metric] = {s: "" for s in species_present}

emm_out = pd.concat({m: t for m, t in emm_tables.items()}, names=["Metric"]).reset_index()
emm_out["CLD"] = [cld_by_metric[m].get(s, "") for m, s in zip(emm_out["Metric"], emm_out["Species"])]
emm_out.to_csv(STATISTICS_DIR / "inter_specific_marginal_means.csv", index=False)

# Contrasts against the reference species are written out as well: they
# are what a regression table in the manuscript reports, even though the
# figure above shows the marginal means instead.
_effect_rng = spawn_rng()
others = [s for s in species_present if s != REFERENCE_SPECIES]
effect_rows = []
for metric, (model, kind, scale_label) in POSTHOC_SPEC.items():
    for s in others:
        if kind == "beta":
            res = beta_contrast(model, s, REFERENCE_SPECIES, SPECIES_TERM, _effect_rng)
        else:
            names = list(model.params.index)
            c = level_contrast_vector(names, SPECIES_TERM, s, REFERENCE_SPECIES)
            res = wald_contrast(model.params.values, model.cov_params().values, c, transform=kind)
        effect_rows.append({"Metric": metric, "Species": s, "Scale": scale_label, **res})
effects_table = pd.DataFrame(effect_rows)
effects_table.to_csv(STATISTICS_DIR / "inter_specific_effect_sizes.csv", index=False)

AXIS_LABEL = {
    "Richness": f"Estimated {RICHNESS_LABEL}",
    "Shannon": ("Estimated Shannon diversity H'\n(geometric mean)" if shannon_family_key == "lognormal"
                else "Estimated Shannon diversity H'"),
    "Simpson": "Estimated Simpson diversity (1 - D)",
    "Pielou": "Estimated Pielou's evenness J'",
}
q_by_metric = global_fdr_species.set_index("Metric")["q_value"].to_dict()
_n_by_species = inter_specific_data["Species_code"].value_counts()

fig, axes = plt.subplots(1, len(ALPHA_METRICS), figsize=(4.4 * len(ALPHA_METRICS), 5.8), squeeze=False)
axes = axes.ravel()
for ax, metric in zip(axes, ALPHA_METRICS):
    emm = emm_tables[metric]
    # Species ordered by the estimate itself: the figure's job is to show
    # where each species sits, so the eye should not have to reorder it.
    order = emm["estimate"].sort_values(ascending=False).index.tolist()
    y_pos = np.arange(len(order))[::-1]
    for y, s in zip(y_pos, order):
        row = emm.loc[s]
        ax.errorbar(row["estimate"], y,
                    xerr=[[max(row["estimate"] - row["ci_low"], 0)],
                          [max(row["ci_high"] - row["estimate"], 0)]],
                    fmt=SPECIES_MARKERS[s], markersize=10, capsize=4, linewidth=1.9,
                    color=SPECIES_COLORS[s], markeredgecolor="white", markeredgewidth=0.9)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"{SPECIES_LABELS_ITALIC.get(s, s)}\n(n = {int(_n_by_species.get(s, 0))})"
                        for s in order], fontsize=9)
    ax.set_xlabel(AXIS_LABEL[metric])
    q = q_by_metric.get(metric, np.nan)
    ax.set_title(f"{metric}\nglobal species effect: {format_p(q, 'q')} {significance_stars(q)}",
                 fontsize=11)
    if metric == "Richness":
        ax.set_xscale("log")
        # A log axis whose data span less than one decade has no major
        # tick inside the view, so the default locator plus a silenced
        # minor formatter leaves the axis with no numbers at all. Ticks
        # are therefore placed at 1-2-3-5 within each decade and labeled
        # explicitly, which survives both a narrow and a wide range.
        ax.xaxis.set_major_locator(mpl.ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 3.0, 5.0),
                                                         numticks=12))
        ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
        ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
    ax.margins(y=0.22)
    # Compact letters, placed in a reserved strip to the RIGHT of the data
    # so they can never sit on top of an interval.
    lo, hi = ax.get_xlim()
    if ax.get_xscale() == "log":
        ax.set_xlim(lo, hi * (hi / lo) ** 0.22)
    else:
        ax.set_xlim(lo, hi + 0.22 * (hi - lo))
    for y, s in zip(y_pos, order):
        ax.annotate(cld_by_metric[metric].get(s, ""),
                    xy=(0.985, y), xycoords=("axes fraction", "data"),
                    ha="right", va="center", fontsize=11, fontweight="bold", color="0.25")

fig.suptitle("Alpha diversity by host species: model-estimated means, adjusted for locality",
             fontsize=14, fontweight="bold", y=1.04)
fig.text(0.5, 0.995,
         "Points are estimated marginal means with 95% confidence intervals; "
         "species sharing a letter are not distinguishable after FDR correction",
         ha="center", va="top", fontsize=10, color="0.35")
figure_caption(fig,
    f"Estimated marginal means from Species + Locality models (N = {len(inter_specific_data)} birds, "
    f"{len(species_present)} species), averaged over localities with equal weight and back-transformed "
    f"to the response scale. Richness: {richness_family}, log axis. Shannon: {shannon_family}. "
    f"Simpson: beta regression. Bold letters are a compact letter display of the "
    f"post-hoc pairwise comparisons in section 06.11 (FDR-adjusted within each metric): two species "
    f"share a letter when their difference does not survive correction. "
    + (f"{', '.join(SPECIES_CODE_TO_NAME[s] for s in species_described_only)} is not shown: it has too "
       f"few birds in the complete-case design to carry a species mean (02.2) and is reported "
       f"descriptively in sections 05.3, 05.4 and 09.1-09.4. " if species_described_only else "")
    + f"Panel titles give the "
    f"FDR-adjusted q-value of the global species test (*** q<0.001, ** q<0.01, * q<0.05, ns not "
    f"significant). Regression coefficients relative to "
    f"{SPECIES_CODE_TO_NAME[REFERENCE_SPECIES]} are in "
    f"results/statistics/inter_specific_effect_sizes.csv."
    + (f" Richness is reported at a fixed {rarefaction_depth:,} reads and is depth-truncated "
       f"(section 04.4): it is comparable between birds but is not an estimate of total richness."
       if RICHNESS_IS_DEPTH_TRUNCATED else ""),
    y=-0.04)
plt.tight_layout()
inter_effects_fig_path = save_figure(fig, ALPHA_DIR / "inter_specific_alpha_diversity_effects.png")
plt.close(fig)   # not displayed here: the interactive version is the next cell

print("06.12 verification")
print("-" * 60)

# A compact letter display is only meaningful if it is CONSISTENT with
# the pairwise table it came from: two species share a letter exactly
# when their comparison was non-significant. Checked, not assumed.
for _metric, _letters in cld_by_metric.items():
    if pairwise_table.empty or _metric not in set(pairwise_table["Metric"]):
        continue
    for _, _row in pairwise_table[pairwise_table["Metric"] == _metric].iterrows():
        _a, _b = _row["Comparison"].split(" vs ")
        _shares = bool(set(_letters.get(_a, "")) & set(_letters.get(_b, "")))
        assert _shares == (not bool(_row["Significant_FDR"])), (
            f"CLD disagrees with the post-hoc table for {_metric} {_a} vs {_b}: "
            f"shares letter={_shares}, significant={bool(_row['Significant_FDR'])}")
print("\nCompact letter display verified against the post-hoc table for every metric and pair.")

assert inter_effects_fig_path.exists()
assert not effects_table["estimate"].eq(0).all()
assert emm_out["estimate"].notna().all(), "A marginal mean failed to compute."
print(f"\nSaved: {inter_effects_fig_path}")
print(f"Saved: {STATISTICS_DIR / 'inter_specific_effect_sizes.csv'}")
print(f"Saved: {STATISTICS_DIR / 'inter_specific_marginal_means.csv'}")
print("OK -- inter-specific effects figure complete.")

In [ ]:
# Interactive view of figure 06.12 — Estimated marginal means per host species
# ----------------------------------------------------------------------
# The interactive counterpart of figure 06.12, drawn from emm_tables and
# cld_by_metric: the model-estimated mean of each host species, averaged
# over localities with equal weight and back-transformed to the response
# scale, with its 95% confidence interval and its compact-letter-display
# group. Hovering gives the exact bounds, which a printed figure can only
# convey through the length of a bar.
#
# The species are held in one order across all three panels, so that a row
# is the same species everywhere and the panels can be read across. The
# order is the ranking on the first modeled metric; the letters beside
# each point are the compact letter display, and two species sharing a
# letter within a panel are not distinguishable after FDR correction.
_q_by_metric = global_fdr_species.set_index("Metric")["q_value"].to_dict()
_n_by_species = inter_specific_data["Species_code"].astype(str).value_counts()
_shared_order = emm_tables[ALPHA_METRICS[0]]["estimate"].sort_values().index.tolist()
_row_label = {sp: species_html(sp) for sp in _shared_order}

fig10_2 = make_subplots(
    rows=1, cols=len(ALPHA_METRICS), horizontal_spacing=0.05, shared_yaxes=True,
    subplot_titles=[f"{m}<br><sub>global species effect: "
                    f"{format_p(_q_by_metric.get(m, np.nan), 'q')} "
                    f"{significance_stars(_q_by_metric.get(m, np.nan))}</sub>"
                    for m in ALPHA_METRICS])

for _i, _metric in enumerate(ALPHA_METRICS, start=1):
    _emm = emm_tables[_metric]
    for _species in _shared_order:
        _row = _emm.loc[_species]
        _letter = cld_by_metric.get(_metric, {}).get(_species, "")
        fig10_2.add_trace(
            go.Scatter(
                x=[_row["estimate"]], y=[_row_label[_species]],
                mode="markers", name=species_html(_species),
                legendgroup=_species, showlegend=(_i == 1),
                error_x=dict(type="data", symmetric=False,
                             array=[max(_row["ci_high"] - _row["estimate"], 0)],
                             arrayminus=[max(_row["estimate"] - _row["ci_low"], 0)],
                             color=SPECIES_COLORS[_species], thickness=1.6, width=7),
                marker=dict(color=SPECIES_COLORS[_species], size=13,
                            symbol=PLOTLY_MARKERS[SPECIES_MARKERS[_species]],
                            line=dict(color="white", width=1.1)),
                customdata=[[_row["ci_low"], _row["ci_high"], _letter or "-",
                             int(_n_by_species.get(_species, 0))]],
                hovertemplate=(f"<b>{species_html(_species, origin=True)}</b><br>"
                               f"{_metric}: %{{x:.3f}}<br>"
                               "95% CI: %{customdata[0]:.3f} to %{customdata[1]:.3f}<br>"
                               "Letter group: %{customdata[2]}<br>"
                               "n = %{customdata[3]} birds<extra></extra>")),
            row=1, col=_i)
    # The letters are drawn as one column at the right of the panel rather
    # than beside each point, so that a wide confidence interval cannot
    # push a letter on top of its own error bar. The axis is widened to
    # leave room for that column.
    _lo, _hi = float(_emm["ci_low"].min()), float(_emm["ci_high"].max())
    if _metric == "Richness":
        # The same log axis the count model is fitted on, and the scale on
        # which richness differences between these species are readable.
        _lo10, _hi10 = np.log10(max(_lo, 1e-9)), np.log10(_hi)
        _pad = 0.08 * (_hi10 - _lo10) or 0.1
        _axis_range = [_lo10 - _pad, _hi10 + 3.0 * _pad]
        _letter_x = 10 ** (_hi10 + 1.9 * _pad)
        fig10_2.update_xaxes(type="log", range=_axis_range, row=1, col=_i)
    else:
        _pad = 0.08 * (_hi - _lo) or 0.1
        fig10_2.update_xaxes(range=[_lo - _pad, _hi + 3.0 * _pad], row=1, col=_i)
        _letter_x = _hi + 1.9 * _pad
    fig10_2.add_trace(
        go.Scatter(
            x=[_letter_x] * len(_shared_order),
            y=[_row_label[sp] for sp in _shared_order],
            mode="text", showlegend=False, hoverinfo="skip",
            text=[f"<b>{cld_by_metric.get(_metric, {}).get(sp, '')}</b>" for sp in _shared_order],
            textposition="middle center", textfont=dict(size=13, color="#444444")),
        row=1, col=_i)
    fig10_2.update_xaxes(title_text=f"Estimated {_metric}", row=1, col=_i)

fig10_2.update_layout(
    template=PLOTLY_TEMPLATE, height=120 + 78 * len(_shared_order),
    title=dict(text=f"Model-estimated alpha diversity per host species, adjusted for locality "
                    f"(N = {len(inter_specific_data)} birds, {len(species_present)} species)",
               x=0.5, xanchor="center", font=dict(size=16)),
    legend=dict(orientation="h", yanchor="bottom", y=-0.42, xanchor="center", x=0.5,
                title_text="Host species"),
    margin=dict(t=110, b=140, l=170, r=40))
fig10_2.show()

assert len(fig10_2.data) == len(ALPHA_METRICS) * (len(species_present) + 1), \
    "Each panel holds one trace per species plus its column of compact-letter-display letters."
assert set(_shared_order) == set(species_present), \
    "The shared row order must cover exactly the modeled species."

In [ ]:
# Table · 06.12 — Estimated marginal means per host species
show_table(emm_out, "Estimated marginal means per host species, back-transformed to the response scale and averaged over localities with equal weight. These are the values plotted in figure 06.12.")

In [ ]:
# Table · 06.12 — Effect sizes relative to the reference species
show_table(effects_table, "Effect sizes relative to the reference species, with confidence intervals on the scale each family reports: a rate ratio for richness, a ratio of means for Shannon and an odds ratio for Simpson.")

In [ ]:
# 06.13 — Sensitivity: the species effect recomputed across a span of rarefaction depths
# ----------------------------------------------------------------------
# The depth chosen in 04.4 is a trade-off, so the main conclusion has to
# be shown not to hinge on it. The global Species test for every metric
# is recomputed at a shallower and a deeper depth, each with its own
# rarefaction, its own sample set (a deeper depth necessarily keeps fewer
# libraries) and its own models. Only the depth changes.
SENSITIVITY_REPLICATES = 15


def species_tests_at_depth(depth: int, n_replicates: int = SENSITIVITY_REPLICATES) -> dict:
    ids = sequencing_depth.index[sequencing_depth >= depth]
    reps = rarefy_matrix_multiple(community_matrix_analysis.loc[ids], depth, n_replicates, spawn_rng)
    reps = [drop_zero_sum_taxa(m)[0] for m in reps]
    point, _ = alpha_diversity_over_replicates(reps)
    data = analysis_metadata.loc[point.index, ["Species_code", "Locality"]].join(point).dropna()
    data = data.loc[~data["Locality"].isin(excluded_localities)]
    keep_species = [s for s in SPECIES_ORDER if (data["Species_code"] == s).sum() >= 3]
    data = data.loc[data["Species_code"].isin(keep_species)].copy()
    data["Species_code"] = pd.Categorical(data["Species_code"], categories=keep_species, ordered=True)
    data["Locality"] = data["Locality"].astype("category")
    data["Richness_count"] = np.rint(data["Richness"]).astype(int)

    out = {"Depth": depth, "N": len(data), "N_species": len(keep_species)}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _f = f"Richness_count ~ {SPECIES_TERM} + {LOCALITY_TERM}"
        out["Richness"] = count_term_test(fit_count_model(_f, data, quiet=True), SPECIES_TERM, _f, data)["pvalue"]
        gauss = smf.ols(f"Shannon ~ {SPECIES_TERM} + {LOCALITY_TERM}", data=data).fit(cov_type="HC3")
        out["Shannon"] = wald_test_term(gauss, SPECIES_TERM)["pvalue"]
        for metric in BETA_RESPONSE_METRICS:
            data[metric] = check_unit_interval(data[metric], metric)
            bfull = fit_beta_model(f"{metric} ~ {SPECIES_TERM} + {LOCALITY_TERM}", data, beta_precision_formula_varying)
            bred = fit_beta_model(f"{metric} ~ {LOCALITY_TERM}", data, beta_precision_formula_varying)
            out[metric] = lr_test(bfull, bred)["pvalue"]
    return out


# Choosing the comparison depths. They are picked to SPAN the feasible
# range, de-duplicated against the selected depth, and asserted to be
# distinct: a sensitivity analysis that compared a depth against itself,
# or against its immediate neighbor, would report agreement whatever the
# data did.
MIN_SENSITIVITY_RETENTION = 0.70

def choose_sensitivity_depths(summary: pd.DataFrame, selected: int, n_extra: int = 2) -> list[int]:
    """Up to `n_extra` candidate depths spanning the usable range, never equal to `selected`."""
    usable = summary[summary["Pct_samples_retained"] >= 100.0 * MIN_SENSITIVITY_RETENTION]
    usable = usable[usable["N_in_smallest_species"] >= 3]
    options = sorted({int(d) for d in usable["Depth"] if int(d) != int(selected)})
    if not options:
        return []
    if len(options) <= n_extra:
        return options
    # Spread the picks across the available range rather than clustering
    # next to the selected depth. The question is whether a materially
    # different depth changes the answer, so neighboring candidates
    # would not test anything.
    idx = np.linspace(0, len(options) - 1, n_extra).round().astype(int)
    return sorted({options[i] for i in idx})

sensitivity_depths = choose_sensitivity_depths(depth_summary, rarefaction_depth, n_extra=2)

sensitivity_rows = [{"Depth": rarefaction_depth, "N": len(inter_specific_data),
                     "N_species": len(species_present),
                     **{m: global_tests_species[METRIC_TEST_LABEL[m]] for m in ALPHA_METRICS}}]
for _d in sensitivity_depths:
    sensitivity_rows.append(species_tests_at_depth(_d))

depth_sensitivity = pd.DataFrame(sensitivity_rows).sort_values("Depth").reset_index(drop=True)
depth_sensitivity["Selected"] = depth_sensitivity["Depth"] == rarefaction_depth
depth_sensitivity["Depth_ratio_vs_selected"] = depth_sensitivity["Depth"] / rarefaction_depth
depth_sensitivity.to_csv(STATISTICS_DIR / "inter_specific_depth_sensitivity.csv", index=False)

print("06.13 verification")
print("-" * 60)
print(f"Unadjusted p-value of the GLOBAL SPECIES effect, recomputed at {len(depth_sensitivity)} "
      f"rarefaction depths")
print(f"(each with its own rarefaction, {SENSITIVITY_REPLICATES} replicates, and its own retained sample set).")

# A single tested depth, or depths that barely differ, cannot support a
# robustness claim -- say so rather than printing reassuring agreement.
_tested = depth_sensitivity["Depth"].tolist()
assert len(set(_tested)) == len(_tested), f"Duplicate depths in the sensitivity table: {_tested}"
_span = max(_tested) / min(_tested)
if len(_tested) < 2:
    print("\n[WARN] Only the selected depth could be tested -- no depth-robustness claim can be made.")
elif _span < 2.0:
    print(f"\n[WARN] The tested depths span only a factor of {_span:.2f}. That is too narrow to "
          f"demonstrate robustness to the depth choice; state the sensitivity analysis as inconclusive.")
else:
    _agree = [m for m in ALPHA_METRICS
              if len({bool(p < ALPHA_SIGNIFICANCE) for p in depth_sensitivity[m]}) == 1]
    print(f"\nTested depths span a factor of {_span:.1f} "
          f"({min(_tested):,} to {max(_tested):,} reads).")
    print(f"Metrics whose species effect reaches the same conclusion at every tested depth: "
          f"{_agree if _agree else 'none'} ({len(_agree)}/{len(ALPHA_METRICS)})")
    if len(_agree) < len(ALPHA_METRICS):
        print(f"[NOTE] {sorted(set(ALPHA_METRICS) - set(_agree))} change conclusion with depth -- report them "
              f"as depth-sensitive and lean on the metrics that do not.")
print(f"\nSaved: {STATISTICS_DIR / 'inter_specific_depth_sensitivity.csv'}")
print("OK -- depth-sensitivity check complete.")

In [ ]:
# Table · 06.13 — The species effect across a span of rarefaction depths
show_table(depth_sensitivity, "The between-species test repeated across a span of rarefaction depths, each depth rarefied afresh from community_matrix_analysis. The conclusion should not change with the depth; the columns show whether it does.")

In [ ]:
# 06.14 — Sensitivity: the species effect on birds whose libraries assigned well to genus
# ----------------------------------------------------------------------
# Every diversity value in this notebook is computed on the confirmed
# genus-level matrix, and the share of a library that reaches a confirmed
# genus varies between host species (02.5). A host whose microbiota is
# thinly represented in the reference database is therefore described
# from a smaller and non-randomly chosen part of its community than a
# host that assigns well, and a between-species difference in diversity
# could in principle be a between-species difference in database
# coverage.
#
# The test: refit the global species model on the subset of birds whose
# libraries assigned well, and see whether the conclusion survives. If it
# does, database coverage is not driving it. Birds excluded here are not
# bad data -- they are birds whose communities this reference cannot
# describe on equal terms.
MAX_UNRESOLVED_SHARE = 0.50

well_assigned_ids = [i for i in inter_specific_data["SampleID"]
                     if float(excluded_share_per_sample.get(i, 1.0)) <= MAX_UNRESOLVED_SHARE]
assignment_subset = inter_specific_data.loc[
    inter_specific_data["SampleID"].isin(well_assigned_ids)].copy()

_kept_species = [s for s in species_present
                 if (assignment_subset["Species_code"] == s).sum() >= MIN_N_PER_LEVEL]
assignment_subset = assignment_subset.loc[assignment_subset["Species_code"].isin(_kept_species)].copy()
assignment_subset["Species_code"] = pd.Categorical(
    assignment_subset["Species_code"],
    categories=[c for c in inter_specific_data["Species_code"].cat.categories if c in _kept_species],
    ordered=True)
assignment_subset["Locality"] = assignment_subset["Locality"].astype("category")

print("06.14 verification")
print("-" * 60)
print(f"Birds retained (unresolved share <= {MAX_UNRESOLVED_SHARE:.0%}): "
      f"{len(assignment_subset)}/{len(inter_specific_data)}")
print("\nPer species, before and after the assignment filter:")
_before_a = inter_specific_data["Species_code"].value_counts().reindex(species_present).fillna(0).astype(int)
_after_a = assignment_subset["Species_code"].value_counts().reindex(species_present).fillna(0).astype(int)
print(pd.DataFrame({"Full model": _before_a, "Well assigned": _after_a,
                    "Dropped": _before_a - _after_a}).to_string())
_lost_species = [s for s in species_present if s not in _kept_species]
if _lost_species:
    print(f"\n[NOTE] {_lost_species} fall below {MIN_N_PER_LEVEL} well-assigned birds and cannot be kept in "
          f"this refit. That is itself the finding for them: their communities are largely invisible to "
          f"this reference database, and their diversity estimates rest on a small, non-random fraction of "
          f"their reads. Species-level statements about them are not supported.")

assignment_sensitivity_rows = []
if len(_kept_species) >= 2:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _f = f"Richness_count ~ {SPECIES_TERM} + {LOCALITY_TERM}"
        assignment_sensitivity_rows.append(
            {"Metric": "Richness",
             **count_term_test(fit_count_model(_f, assignment_subset, quiet=True),
                               SPECIES_TERM, _f, assignment_subset)})
        _gauss = smf.ols(f"Shannon ~ {SPECIES_TERM} + {LOCALITY_TERM}",
                         data=assignment_subset).fit(cov_type="HC3")
        assignment_sensitivity_rows.append({"Metric": "Shannon", **wald_test_term(_gauss, SPECIES_TERM)})
        for _metric in BETA_RESPONSE_METRICS:
            assignment_subset[_metric] = check_unit_interval(assignment_subset[_metric], _metric)
            _full = fit_beta_model(f"{_metric} ~ {SPECIES_TERM} + {LOCALITY_TERM}",
                                   assignment_subset, beta_precision_formula_constant)
            _red = fit_beta_model(f"{_metric} ~ {LOCALITY_TERM}",
                                  assignment_subset, beta_precision_formula_constant)
            assignment_sensitivity_rows.append({"Metric": _metric, **lr_test(_full, _red)})

if assignment_sensitivity_rows:
    assignment_sensitivity = pd.DataFrame(assignment_sensitivity_rows)
    assignment_sensitivity["N"] = len(assignment_subset)
    assignment_sensitivity["N_species"] = len(_kept_species)
    assignment_sensitivity["p_full_model"] = [global_tests_species[METRIC_TEST_LABEL[m]]
                                              for m in assignment_sensitivity["Metric"]]
    assignment_sensitivity.to_csv(STATISTICS_DIR / "assignment_rate_sensitivity.csv", index=False)
    print(f"\nGlobal species effect on well-assigned birds only (N = {len(assignment_subset)}, "
          f"{len(_kept_species)} species), beside the full-model p-value.")

    _still = [r["Metric"] for _, r in assignment_sensitivity.iterrows()
              if np.isfinite(r["pvalue"]) and r["pvalue"] < ALPHA_SIGNIFICANCE]
    if len(_still) == len(assignment_sensitivity):
        print("\n-> The species effect holds on every metric when only well-assigned birds are used. "
              "The between-species differences in diversity are therefore not an artefact of uneven "
              "taxonomic assignment, and can be discussed as a property of the communities.")
    else:
        _gone = [m for m in assignment_sensitivity["Metric"] if m not in _still]
        print(f"\n-> [WARNING] {_gone} lose significance once poorly assigned birds are removed. For those "
              f"metrics the species effect cannot be separated from differences in reference-database "
              f"coverage, and must be reported as confounded rather than as a biological result.")
    assert assignment_sensitivity["pvalue"].notna().any(), "No sensitivity test produced a p-value."
print("OK -- section 06.0 complete.")

In [ ]:
# Table · 06.14 — The species effect on well-assigned birds only
if assignment_sensitivity_rows:
    show_table(assignment_sensitivity, "The between-species test repeated on the birds whose libraries "
               "assigned well to genus, which separates a biological difference between host species "
               "from a difference in how well each host's microbiota is represented in the reference "
               "database.")

## 07.0 — Within-species analysis

Within each host species separately, models each alpha-diversity metric as a function of
`Locality + Sex + body condition`. This is a different question from section 06.0 and is kept
strictly apart from it: sex and body-condition effects are not assumed to be common across
species, so each species gets its own dataset, its own terms and its own tests, and species are
never pooled.

### Model complexity follows replication, not total N

A term enters a species' model only if every one of its levels has at least `MIN_N_PER_LEVEL`
individuals *in that species*. Species and terms that fail this are documented and excluded rather
than fitted on a handful of birds. A species with too few complete cases for even the simplest
model is reported descriptively, from all of its samples, and no model is fitted for it.

Because sex is only known for part of the sample, a species' dataset is restricted to
sex-determined birds only when `Sex` actually enters its model. That restriction is correct for
the sex term but costs birds from the locality and body-condition tests, which do not need it, so
section 07.6 also refits the body-condition term on every bird with a body-condition value and
reports the two side by side.

### Body condition

Body condition is the scaled mass index of Peig & Green (2009), computed per species with the
standardized major axis slope of log(mass) on log(tarsus). The strength of each species'
mass–tarsus relationship is reported alongside it, because the index only carries a genuine
structural-size correction where that relationship exists: where it does not, the index is body
mass standardized to the mean tarsus and is described that way. A residual condition index is
computed as a secondary, sensitivity covariate.

In [ ]:
# 07.1 — Morphometric completeness: which birds can carry a body-condition value
# ----------------------------------------------------------------------
body_condition_columns = ["Mass", "Tarsus"]
missing_body_condition = analysis_metadata[body_condition_columns].isna().any(axis=1)
unmatched_body_condition = analysis_metadata.loc[missing_body_condition, ["Species_code"] + body_condition_columns]
unmatched_body_condition.to_csv(QC_REPORTS_DIR / "unmatched_body_condition_samples.csv")

print("07.1 verification")
print("-" * 60)
print(f"Samples missing Mass and/or Tarsus (excluded from body-condition covariates): "
      f"{int(missing_body_condition.sum())}/{len(analysis_metadata)}")
if missing_body_condition.any():
    print(unmatched_body_condition.to_string())
print(f"Saved: {QC_REPORTS_DIR / 'unmatched_body_condition_samples.csv'}")
print("OK -- dropped (never imputed) downstream.")

In [ ]:
# 07.2 — Body condition per species: scaled mass index (SMA slope) and its reliability
# ----------------------------------------------------------------------
# Both indices are fitted SEPARATELY within each species: the mass-tarsus
# allometry is species-specific, so one pooled regression across five
# species would not be biologically meaningful.
#
# SMI_i = M_i * (L_0 / L_i) ** b_SMA
#
# where b_SMA is the STANDARDISED MAJOR AXIS slope of log(M) on log(L),
# obtained as b_OLS / r (Peig & Green 2009, eq. 2). Using b_OLS instead
# attenuates the slope by the factor r and, when r is small, produces an
# "index" that is essentially uncorrected body mass.
MIN_ALLOMETRY_R2 = 0.10   # below this, the tarsus correction is reported as unreliable


def compute_body_condition(metadata: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    complete = metadata.dropna(subset=["Mass", "Tarsus"]).copy()
    complete = complete.loc[(complete["Mass"] > 0) & (complete["Tarsus"] > 0)]
    log_mass, log_tarsus = np.log(complete["Mass"]), np.log(complete["Tarsus"])

    condition_index = pd.Series(index=complete.index, dtype=float)
    smi = pd.Series(index=complete.index, dtype=float)
    allometry_log = []

    for species in [s for s in SPECIES_ORDER if s in set(complete["Species_code"])]:
        mask = (complete["Species_code"] == species).values
        if mask.sum() < 3:
            print(f"[WARN] {species}: N={int(mask.sum())} < 3 complete Mass/Tarsus records -- body condition not computed.")
            continue
        y, x = log_mass[mask], log_tarsus[mask]
        ols_fit = sm.OLS(y, sm.add_constant(x)).fit()
        b_ols = float(ols_fit.params.iloc[1])
        r = float(np.sign(b_ols) * np.sqrt(max(ols_fit.rsquared, 0.0)))
        # Standardised major axis slope. |r| can be ~0 in this dataset,
        # which makes b_SMA numerically unstable; guarded and reported.
        b_sma = b_ols / r if abs(r) > 1e-8 else np.nan
        condition_index.loc[complete.index[mask]] = ols_fit.resid

        l0 = float(complete.loc[complete.index[mask], "Tarsus"].mean())
        if np.isfinite(b_sma):
            smi.loc[complete.index[mask]] = (
                complete.loc[complete.index[mask], "Mass"]
                * (l0 / complete.loc[complete.index[mask], "Tarsus"]) ** b_sma
            )
        else:
            smi.loc[complete.index[mask]] = complete.loc[complete.index[mask], "Mass"]
        allometry_log.append({
            "Species_code": species, "N": int(mask.sum()),
            "b_OLS": b_ols, "r": r, "b_SMA": b_sma,
            "R2": float(ols_fit.rsquared), "slope_p_value": float(ols_fit.pvalues.iloc[1]),
            "L0_mean_tarsus": l0,
            "Allometry_reliable": bool(ols_fit.rsquared >= MIN_ALLOMETRY_R2 and ols_fit.pvalues.iloc[1] < ALPHA_SIGNIFICANCE),
        })

    out = complete[["Species_code"]].copy()
    out["ConditionIndex"] = condition_index
    out["SMI"] = smi
    return out, pd.DataFrame(allometry_log)


body_condition, allometry_log = compute_body_condition(analysis_metadata)
body_condition.to_csv(PROCESSED_DATA_DIR / "body_condition.csv")
allometry_log.to_csv(STATISTICS_DIR / "body_condition_allometry_log.csv", index=False)

print("07.2 verification")
print("-" * 60)

_weak = allometry_log.loc[~allometry_log["Allometry_reliable"], "Species_code"].tolist()
if _weak:
    print(f"\n[IMPORTANT] For {_weak}, mass and tarsus are only weakly (or not significantly) related "
          f"(R2 < {MIN_ALLOMETRY_R2} and/or slope p >= {ALPHA_SIGNIFICANCE}). For those species the scaled mass "
          f"index carries almost no structural-size correction and should be interpreted, and described in the "
          f"Methods, as BODY MASS standardized to the mean tarsus rather than as a validated condition index. "
          f"This is a property of the morphometric data (tarsus measured over a narrow range, measurement "
          f"error relative to that range), not of the code.")
assert body_condition[["ConditionIndex", "SMI"]].notna().all().all()
assert (body_condition["SMI"] > 0).all()
print(f"\nBody condition computed for {len(body_condition)}/{len(analysis_metadata)} samples.")
print("OK -- SMI (SMA slope) is the primary covariate; ConditionIndex is the secondary/sensitivity one.")

In [ ]:
# Table · 07.2 — Per-species allometry and the reliability of each slope
show_table(allometry_log, "Per-species allometric fits of log body mass on log tarsus, from analysis_metadata. The scaled mass index uses the standardized major axis slope; the reliability columns say for which species that slope can actually be estimated.")

In [ ]:
# 07.3 — Per-species datasets and term selection, driven by each species' own replication
# ----------------------------------------------------------------------
# Term inclusion is DATA-DRIVEN and decided in this order:
#   1. start from every rarefied sample of the species that has an SMI;
#   2. decide whether Sex is identifiable (>= MIN_N_PER_LEVEL males AND females);
#   3. ONLY IF Sex enters the model, restrict to sex-determined birds.
#      Restricting earlier would discard unknown-sex individuals from
#      models that never use Sex, and those birds carry information about
#      locality and body condition;
#   4. re-check which localities still have >= MIN_N_PER_LEVEL individuals
#      in the resulting dataset, and keep Locality only if >= 2 remain.
intra_specific_base = alpha_data[["Species_code", "Locality", "Sex"] + ALPHA_METRICS].join(
    body_condition[["ConditionIndex", "SMI"]], how="inner")

intra_specific_datasets: dict[str, pd.DataFrame] = {}
intra_specific_term_decisions: dict[str, dict] = {}

for species in SPECIES_ORDER:
    data = intra_specific_base.loc[intra_specific_base["Species_code"] == species].copy()
    n_all = len(data)

    sex_counts = data.loc[data["Sex"].isin(["Male", "Female"]), "Sex"].value_counts()
    include_sex = bool(len(sex_counts) >= 2 and (sex_counts >= MIN_N_PER_LEVEL).all())
    n_unknown_sex_dropped = 0
    if include_sex:
        n_unknown_sex_dropped = int((~data["Sex"].isin(["Male", "Female"])).sum())
        data = data.loc[data["Sex"].isin(["Male", "Female"])].copy()
        data["Sex"] = pd.Categorical(data["Sex"], categories=["Male", "Female"])

    locality_counts = data["Locality"].value_counts()
    identifiable_localities = locality_counts[locality_counts >= MIN_N_PER_LEVEL].index.tolist()
    dropped_localities = [l for l in locality_counts.index if l not in identifiable_localities]
    include_locality = len(identifiable_localities) >= 2
    if include_locality:
        data = data.loc[data["Locality"].isin(identifiable_localities)].copy()
        data["Locality"] = pd.Categorical(data["Locality"], categories=identifiable_localities)

    intra_specific_datasets[species] = data
    intra_specific_term_decisions[species] = {
        "N_all_rarefied": n_all, "N": len(data),
        "include_locality": include_locality, "include_sex": include_sex,
        "identifiable_localities": identifiable_localities, "dropped_localities": dropped_localities,
        "n_unknown_sex_dropped": n_unknown_sex_dropped,
    }

print("07.3 verification")
print("-" * 60)
for species, d in intra_specific_term_decisions.items():
    print(f"{species}: N={d['N']} (of {d['N_all_rarefied']} rarefied samples with SMI)")
    print(f"    Sex      : {'INCLUDED' if d['include_sex'] else 'excluded'}"
          + (f" -- {d['n_unknown_sex_dropped']} unknown-sex bird(s) dropped because Sex is a model term"
             if d["include_sex"] else " -- too few of at least one sex; unknown-sex birds are therefore KEPT"))
    print(f"    Locality : {'INCLUDED' if d['include_locality'] else 'excluded'} "
          f"(kept {d['identifiable_localities']}, dropped {d['dropped_localities']})")
    assert d["N"] == len(intra_specific_datasets[species])

# What including Sex COSTS. Restricting to sexed birds is the right thing
# to do for a Sex term, but the same restriction then applies to the
# Locality and body-condition tests fitted on that dataset, and those are
# the terms the study is actually about. In a section where almost
# nothing reaches significance, the difference between "no effect" and
# "not enough birds to see one" is the whole interpretation, so the cost
# is quantified here rather than left implicit.
sex_restriction_cost = pd.DataFrame([
    {"Species": s,
     "N_with_SMI": d["N_all_rarefied"],
     "N_modeled": d["N"],
     "Birds_lost_to_sexing": d["N_all_rarefied"] - d["N"],
     "Pct_lost": 100.0 * (d["N_all_rarefied"] - d["N"]) / d["N_all_rarefied"] if d["N_all_rarefied"] else np.nan,
     "Sex_in_model": d["include_sex"]}
    for s, d in intra_specific_term_decisions.items()])
sex_restriction_cost.to_csv(STATISTICS_DIR / "intra_specific_sample_loss.csv", index=False)
print("\nCost of the terms selected above, in birds:")
print(sex_restriction_cost.round(1).to_string(index=False))
_heavy = sex_restriction_cost[sex_restriction_cost["Pct_lost"] >= 20.0]
if not _heavy.empty:
    print(f"\n[POWER WARNING] {', '.join(_heavy['Species'])} lose >= 20% of their birds because Sex is in "
          f"the model and unsexed birds must then be dropped. Those birds are still available for the "
          f"Locality and body-condition questions, which do not need sex. Section 07.6 therefore refits "
          f"the body-condition term on every bird with an SMI, and reports that alongside, so a null "
          f"result there is not silently a null produced by a smaller sample.")
print(f"\nOK -- per-species datasets built; terms enter only with >= {MIN_N_PER_LEVEL} per level.")

In [ ]:
# 07.4 — Species with too few birds to model, described from all of their samples
# ----------------------------------------------------------------------
# Pre-specified policy, documented here and not decided after seeing any
# p-value: a species is DESCRIPTIVE-ONLY -- summary statistics and figures,
# no fitted inferential model -- if it has too few complete-case
# individuals for even the simplest one-term model, or if neither
# Locality nor Sex is identifiable after the 07.3 exclusions.
MIN_N_FOR_INFERENCE = 15

descriptive_only_species = [
    s for s, d in intra_specific_term_decisions.items()
    if d["N"] < MIN_N_FOR_INFERENCE or not (d["include_locality"] or d["include_sex"])
]
inferential_species = [s for s in SPECIES_ORDER if s not in descriptive_only_species]

print("07.4 verification")
print("-" * 60)
print(f"MIN_N_FOR_INFERENCE = {MIN_N_FOR_INFERENCE}")
print(f"Descriptive-only species : {descriptive_only_species}")
print(f"Modeled species         : {inferential_species}")

for species in descriptive_only_species:
    d = intra_specific_term_decisions[species]
    reason = (f"N = {d['N']} < {MIN_N_FOR_INFERENCE}" if d["N"] < MIN_N_FOR_INFERENCE
              else "neither Locality nor Sex is identifiable")
    print(f"\n{species} ({SPECIES_CODE_TO_NAME[species]}): descriptive only -- {reason}.")
    # The descriptive summary uses EVERY rarefied sample of the species,
    # not the model-ready subset: a species excluded from modeling should
    # still be described from all the data there is.
    all_samples = alpha_data.loc[alpha_data["Species_code"] == species, ALPHA_METRICS]
    if all_samples.empty:
        print("  No rarefied samples available for this species.")
        continue
    desc = all_samples.agg(["count", "mean", "std", "min", "max"])
    print(desc.round(3).to_string())
    desc.to_csv(STATISTICS_DIR / f"intra_specific_{species}_descriptive_only.csv")

assert len(inferential_species) >= 1, "No species has enough data for an intra-specific model."
# The two thresholds are separate on purpose -- a within-species model has to
# estimate three covariates from one species' birds and needs more of them
# than a single species mean does -- but they must not contradict each other.
# A species modeled here but not between species would be a species the
# notebook treats as unanalyzable in one section and analyzable in another.
assert set(inferential_species) <= set(modeled_species), (
    f"{sorted(set(inferential_species) - set(modeled_species))} would be modeled within species "
    f"while being held out of the between-species models. MIN_N_FOR_INFERENCE "
    f"({MIN_N_FOR_INFERENCE}) must stay at or above MIN_N_FOR_SPECIES_MODEL "
    f"({MIN_N_FOR_SPECIES_MODEL}).")
print("\nOK -- descriptive-only species documented and summarized from all their samples.")

In [ ]:
# 07.5 — Fit each species' model with the families used in section 06.0
# ----------------------------------------------------------------------
# The families match section 06.0 exactly, because a response does not
# change its nature when the question moves from between species to
# within one: Richness keeps the dispersion-driven count model on the
# rounded replicate mean, Shannon the Gamma GLM with a log link, and
# Simpson the beta regression. Fixing the families in advance rather than
# re-selecting them per species matters here: with 18-30 birds per
# species a per-species selection would be unstable, and the same choice
# applied identically everywhere keeps the five sets of coefficients
# comparable with one another and with the between-species model.
SEX_TERM = "C(Sex)"


def build_intra_specific_formula(response: str, decision: dict) -> str:
    terms = []
    if decision["include_locality"]:
        terms.append(LOCALITY_TERM)
    if decision["include_sex"]:
        terms.append(SEX_TERM)
    terms.append("SMI")
    return f"{response} ~ " + " + ".join(terms)


def fit_intra_model(family: str, formula: str, data: pd.DataFrame):
    if family == "count":
        return fit_count_model(formula, data, quiet=True)["model"]
    if family == "gamma":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return smf.glm(formula, data=data,
                           family=sm.families.Gamma(link=sm.families.links.Log())).fit()
    if family == "gaussian":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return smf.ols(formula, data=data).fit(cov_type="HC3")
    return fit_beta_model(formula, data, "~ 1")


_INTRA_FAMILY_BY_METRIC = {"Richness": ("count", "Richness_count"), "Shannon": ("gamma", "Shannon"),
                           **{_m: ("beta", _m) for _m in BETA_RESPONSE_METRICS}}
INTRA_FAMILIES = [(_m, *_INTRA_FAMILY_BY_METRIC[_m]) for _m in ALPHA_METRICS]
# Maps the family label used when fitting to the distribution name the
# quantile-residual diagnostic needs. The count family is not fixed in
# advance -- it is chosen per species from the observed dispersion -- so
# its key comes back from the fitter itself.
INTRA_FAMILY_KEY_BY_FAMILY = {"gaussian": "gaussian", "gamma": "gamma", "beta": "beta"}

intra_specific_models: dict[str, dict] = {}
for species in inferential_species:
    data = intra_specific_datasets[species].copy()
    data["Richness_count"] = np.rint(data["Richness"]).astype(int)
    for _m in BETA_RESPONSE_METRICS:
        data[_m] = check_unit_interval(data[_m], f"{species}/{_m}")
    decision = intra_specific_term_decisions[species]
    intra_specific_models[species] = {}

    for metric, family, response in INTRA_FAMILIES:
        formula_full = build_intra_specific_formula(response, decision)
        family_key = None
        if family == "count":
            count_fit = fit_count_model(formula_full, data, quiet=True)
            model_full, family_label = count_fit["model"], count_fit["family"]
            family_key = count_fit["key"]
        else:
            model_full = fit_intra_model(family, formula_full, data)
            family_label = {"gaussian": "Gaussian OLS, HC3 robust SE",
                            "gamma": "Gamma GLM (log link)"}.get(family, "Beta regression")

        term_tests = {}
        for term_label, term in [("Locality", LOCALITY_TERM), ("Sex", SEX_TERM), ("SMI", "SMI")]:
            if term not in formula_full:
                continue
            if family in ("gaussian", "gamma"):
                # Wald, matching how the same term is tested for these
                # families in section 06.0. A likelihood-ratio test on a
                # Gamma GLM would have to profile out the estimated
                # dispersion, which is exactly the approximation the Wald
                # test avoids here.
                term_tests[term_label] = wald_test_term(model_full, term)
            elif family == "count":
                term_tests[term_label] = count_term_test(count_fit, term, formula_full, data)
            else:
                remaining = [t.strip() for t in formula_full.split("~")[1].split("+") if t.strip() != term]
                formula_reduced = f"{response} ~ " + (" + ".join(remaining) if remaining else "1")
                term_tests[term_label] = lr_test(model_full, fit_intra_model(family, formula_reduced, data))

        intra_specific_models[species][metric] = {
            "formula": formula_full, "model": model_full, "family": family,
            "family_key": family_key or INTRA_FAMILY_KEY_BY_FAMILY.get(family),
            "dispersion": count_fit["dispersion"] if family == "count" else None,
            "family_label": family_label, "response": response, "term_tests": term_tests, "N": len(data),
        }

print("07.5 verification")
print("-" * 60)
for species in inferential_species:
    print(f"\n{SPECIES_CODE_TO_NAME[species]} ({species}), N = {intra_specific_models[species]['Richness']['N']}:")
    for metric, fit_info in intra_specific_models[species].items():
        print(f"  {metric:9s} [{fit_info['family_label']}]  {fit_info['formula']}")
        for term_label, test in fit_info["term_tests"].items():
            print(f"      {term_label:9s}: {test['test']:4s} stat={test['statistic']:8.3f} "
                  f"df={test['df']} p={test['pvalue']:.4g}")
        assert len(fit_info["term_tests"]) >= 1, f"{species}/{metric} has no testable term."

# Goodness of fit for every per-species model, by the same quantile-residual
# criterion used for the between-species models in 06.9. Sixteen Q-Q plots
# would not earn their space, so the diagnostic is reported as one table:
# what matters is whether any model fails, not the shape of each one.
INTRA_FAMILY_KEYS = {"count": None, "gaussian": "gaussian", "beta": "beta"}
_intra_gof_rng = spawn_rng()
intra_goodness_of_fit = []
for species in inferential_species:
    _data = intra_specific_datasets[species]
    for metric, fit_info in intra_specific_models[species].items():
        _family = fit_info["family"]
        _key = fit_info.get("family_key") or INTRA_FAMILY_KEYS.get(_family)
        _y = np.rint(_data[metric]).astype(int).values if _family == "count" else _data[metric].values
        if _key is None:
            continue
        try:
            _check = model_residual_check(fit_info["model"], _y, _key,
                                          fit_info.get("dispersion"), _intra_gof_rng)
        except Exception as exc:                        # noqa: BLE001 -- reported, never silent
            print(f"[WARN] Goodness of fit unavailable for {species}/{metric}: {exc}")
            continue
        intra_goodness_of_fit.append({"Species": species, "Metric": metric, "Family": _key,
                                      "Residual": _check["kind"],
                                      **{k: v for k, v in _check.items()
                                         if k not in ("residuals", "kind")}})

if intra_goodness_of_fit:
    intra_goodness_of_fit = pd.DataFrame(intra_goodness_of_fit)
    intra_goodness_of_fit.to_csv(STATISTICS_DIR / "intra_specific_goodness_of_fit.csv", index=False)
    _bad = intra_goodness_of_fit[intra_goodness_of_fit["pvalue"] < ALPHA_SIGNIFICANCE]
    _n_testable = int(intra_goodness_of_fit["pvalue"].notna().sum())
    if _bad.empty:
        print(f"All {_n_testable} models with a distribution to test are consistent with it; the "
              f"{len(intra_goodness_of_fit) - _n_testable} quasi-likelihood fits have residual SDs listed "
              f"above, which should sit near 1.")
    else:
        print(f"[WARNING] {len(_bad)}/{len(intra_goodness_of_fit)} models depart from their distribution: "
              f"{', '.join(f'{r.Species}/{r.Metric}' for r in _bad.itertuples())}. Their p-values in 07.6 "
              f"rest on an assumption these data do not support.")
    print(f"Saved: {STATISTICS_DIR / 'intra_specific_goodness_of_fit.csv'}")

print("\nOK -- per-species models fitted with data-driven terms.")

In [ ]:
# Table · 07.5 — Goodness of fit of every within-species model
if len(intra_goodness_of_fit):
    show_table(intra_goodness_of_fit, "Goodness of fit of every within-species model, by quantile "
               "residuals on each species' own dataset. The families are the same as in section 06.0, "
               "so a response keeps its distribution whether the question is asked between species or "
               "within one.", page_length=15)

In [ ]:
# 07.6 — Within-species term tests, FDR by (species, term), and the body-condition sensitivity
# ----------------------------------------------------------------------
# FDR family: for a given species and a given term (e.g. "PasDom, Sex"),
# the family is that term's test across the modeled diversity metrics --
# exactly the family definition used in 06.10, kept separate per species,
# because species are hierarchically separate questions and are never
# pooled.
intra_global_tests_df = pd.DataFrame([
    {"Species": species, "Term": term_label, "Metric": metric,
     "statistic": test["statistic"], "df": test["df"], "pvalue": test["pvalue"], "N": fit_info["N"]}
    for species in inferential_species
    for metric, fit_info in intra_specific_models[species].items()
    for term_label, test in fit_info["term_tests"].items()
])

intra_fdr_blocks = []
for (species, term_label), block in intra_global_tests_df.groupby(["Species", "Term"], sort=False):
    qtab = fdr_within_family(dict(zip(block["Metric"], block["pvalue"])))
    qtab = qtab.rename(columns={"Comparison": "Metric"})
    qtab.insert(0, "Term", term_label); qtab.insert(0, "Species", species)
    qtab = qtab.merge(block[["Metric", "statistic", "df", "N"]], on="Metric", how="left")
    intra_fdr_blocks.append(qtab)
intra_global_effect_summary = pd.concat(intra_fdr_blocks, ignore_index=True)
intra_global_effect_summary["Stars"] = intra_global_effect_summary["q_value"].map(significance_stars)
intra_global_effect_summary.to_csv(STATISTICS_DIR / "intra_specific_global_effect_tests.csv", index=False)

print("07.6 verification")
print("-" * 60)
n_tests = len(intra_global_effect_summary)
n_sig = int(intra_global_effect_summary["Significant_FDR"].sum())
print(f"\n{n_sig}/{n_tests} intra-specific tests are significant after FDR correction within their family.")
if n_sig == 0:
    print("[INTERPRETATION] No within-species effect of locality, sex or body condition on alpha diversity "
          "survives correction. With this design (18-30 sex-determined birds per species) the analysis can "
          "only detect large effects, so this is best reported as 'no detectable effect at this sample size', "
          "with the effect sizes and confidence intervals in 07.8 showing what magnitudes remain compatible "
          "with the data -- not as evidence that no effect exists.")
assert len(intra_global_effect_summary) == len(intra_global_tests_df)

# ------------------------------------------------------------------
# Body condition, refitted on EVERY bird of the species that has an SMI.
#
# The models above drop unsexed birds wherever Sex is a term, which costs
# the body-condition test birds it does not need. This refit puts them
# back: same response, same family, Locality retained where identifiable,
# Sex simply absent. It is reported as a SENSITIVITY analysis, never as a
# replacement -- dropping Sex could in principle bias the SMI slope,
# since body mass differs between the sexes -- but if the two versions
# disagree, the disagreement is itself the finding, and if they agree on
# a null then the null is not merely a sample-size artefact.
# ------------------------------------------------------------------
smi_full_sample_rows = []
for species in inferential_species:
    d = intra_specific_term_decisions[species]
    if not d["include_sex"]:
        continue  # nothing was dropped for this species, so there is nothing to recover
    full = intra_specific_base.loc[intra_specific_base["Species_code"] == species].copy()
    if d["include_locality"]:
        full = full.loc[full["Locality"].isin(d["identifiable_localities"])].copy()
        full["Locality"] = pd.Categorical(full["Locality"], categories=d["identifiable_localities"])
    if len(full) <= len(intra_specific_datasets[species]) or len(full) < MIN_N_FOR_INFERENCE:
        continue  # nothing to gain, or too few birds to fit
    full["Richness_count"] = np.rint(full["Richness"]).astype(int)
    for _m in BETA_RESPONSE_METRICS:
        full[_m] = check_unit_interval(full[_m], f"{species}/{_m} (full sample)")

    # Same families and the same term-test machinery as 07.5; the only
    # difference is that Sex is absent from the formula.
    _terms = ([LOCALITY_TERM] if d["include_locality"] else []) + ["SMI"]
    for metric, family, response in INTRA_FAMILIES:
        formula = f"{response} ~ " + " + ".join(_terms)
        try:
            if family == "count":
                _fit = fit_count_model(formula, full, quiet=True)
                test = count_term_test(_fit, "SMI", formula, full)
            elif family in ("gaussian", "gamma"):
                test = wald_test_term(fit_intra_model(family, formula, full), "SMI")
            else:
                _remaining = [t for t in _terms if t != "SMI"]
                _reduced = f"{response} ~ " + (" + ".join(_remaining) if _remaining else "1")
                test = lr_test(fit_intra_model(family, formula, full),
                               fit_intra_model(family, _reduced, full))
        except Exception as exc:                       # noqa: BLE001 -- reported, never silent
            print(f"[WARN] SMI full-sample refit failed for {species}/{metric}: {exc}")
            continue
        if not np.isfinite(test.get("pvalue", np.nan)):
            print(f"[WARN] SMI full-sample refit for {species}/{metric} gave a non-finite p-value; skipped.")
            continue
        smi_full_sample_rows.append({"Species": species, "Metric": metric,
                                     "N_sex_restricted": len(intra_specific_datasets[species]),
                                     "N_full_sample": len(full),
                                     "statistic": test["statistic"], "df": test["df"],
                                     "pvalue": test["pvalue"]})

if smi_full_sample_rows:
    smi_full_sample = pd.DataFrame(smi_full_sample_rows)
    _blocks = []
    for species, block in smi_full_sample.groupby("Species", sort=False):
        qtab = fdr_within_family(dict(zip(block["Metric"], block["pvalue"])))
        _blocks.append(block.merge(qtab.rename(columns={"Comparison": "Metric"})[
            ["Metric", "q_value", "Significant_FDR"]], on="Metric", how="left"))
    smi_full_sample = pd.concat(_blocks, ignore_index=True)
    smi_full_sample.to_csv(STATISTICS_DIR / "intra_specific_smi_full_sample_sensitivity.csv", index=False)
    print("\nSensitivity: the body-condition (SMI) test refitted on every bird with an SMI,")
    print("with Sex dropped from the model so unsexed birds are not discarded:")
    print(smi_full_sample[["Species", "Metric", "N_sex_restricted", "N_full_sample",
                           "pvalue", "q_value", "Significant_FDR"]].round(4).to_string(index=False))
    _primary_sig = set(map(tuple, intra_global_effect_summary.loc[
        intra_global_effect_summary["Significant_FDR"] & intra_global_effect_summary["Term"].eq("SMI"),
        ["Species", "Metric"]].values))
    _sens_sig = set(map(tuple, smi_full_sample.loc[smi_full_sample["Significant_FDR"],
                                                   ["Species", "Metric"]].values))
    if _primary_sig == _sens_sig:
        print("-> The two versions agree on every species and metric. The body-condition results are not "
              "an artefact of the birds lost to sexing.")
    else:
        print(f"-> [NOTE] They DISAGREE: primary-only {sorted(_primary_sig - _sens_sig)}, "
              f"sensitivity-only {sorted(_sens_sig - _primary_sig)}. Report both, and treat the "
              f"body-condition result as sample-dependent rather than established.")
    assert smi_full_sample["N_full_sample"].ge(smi_full_sample["N_sex_restricted"]).all()

print(f"\nSaved: {STATISTICS_DIR / 'intra_specific_global_effect_tests.csv'}")
print("OK -- section 07.6 complete.")

In [ ]:
# Table · 07.6 — Within-species term tests
show_table(intra_global_effect_summary, "Within-species term tests on each species' own dataset, FDR-adjusted within each (species, term) family. Search a species code to read its block alone.", page_length=15)

In [ ]:
# 07.7 — Post-hoc comparisons for within-species terms with more than two levels
# ----------------------------------------------------------------------
# Sex has exactly two levels wherever it is included, so its global test
# IS the pairwise comparison and no post-hoc step is needed. Locality can
# have more than two identifiable levels, so pairwise comparisons are run
# there, and only when the global Locality test survived FDR.
_intra_posthoc_rng = spawn_rng()
intra_pairwise_records = []

for species in inferential_species:
    decision = intra_specific_term_decisions[species]
    if not decision["include_locality"] or len(decision["identifiable_localities"]) < 3:
        continue
    sig_rows = intra_global_effect_summary.query(
        "Species == @species and Term == 'Locality' and Significant_FDR == True")
    for metric in sig_rows["Metric"]:
        fit_info = intra_specific_models[species][metric]
        model, family = fit_info["model"], fit_info["family"]
        for a, b in combinations(decision["identifiable_localities"], 2):
            if family == "beta":
                res = beta_contrast(model, a, b, LOCALITY_TERM, _intra_posthoc_rng)
            else:
                names = list(model.params.index)
                c = level_contrast_vector(names, LOCALITY_TERM, a, b)
                res = wald_contrast(model.params.values, model.cov_params().values, c,
                                    transform="exp" if family == "count" else "identity")
            intra_pairwise_records.append({"Species": species, "Metric": metric,
                                           "Comparison": f"{a} vs {b}", **res})

intra_pairwise_table = pd.DataFrame(intra_pairwise_records)

print("07.7 verification")
print("-" * 60)
if intra_pairwise_table.empty:
    print("No species had a significant (FDR) Locality effect with >= 3 identifiable levels -- "
          "no pairwise comparisons, by design.")
else:
    adjusted = []
    for (species, metric), block in intra_pairwise_table.groupby(["Species", "Metric"], sort=False):
        qtab = fdr_within_family(dict(zip(block["Comparison"], block["pvalue"])))
        adjusted.append(block.merge(qtab[["Comparison", "q_value", "Significant_FDR"]], on="Comparison", how="left"))
    intra_pairwise_table = pd.concat(adjusted, ignore_index=True)
    intra_pairwise_table["Stars"] = intra_pairwise_table["q_value"].map(significance_stars)
    print(intra_pairwise_table.round(4).to_string(index=False))
intra_pairwise_table.to_csv(STATISTICS_DIR / "intra_specific_posthoc_pairwise.csv", index=False)
print(f"\nSaved: {STATISTICS_DIR / 'intra_specific_posthoc_pairwise.csv'}")
print("OK -- section 07.7 complete.")

In [ ]:
# 07.8 — Figure: body-condition slopes on diversity, within each host species
# ----------------------------------------------------------------------
# One panel per diversity metric; within a panel, one row per species,
# showing the standardized body-condition slope with its 95% CI. SMI is
# standardized (z-scored) WITHIN each species purely for this figure, so
# that slopes from species with different body masses (a 27 g sparrow and
# a 115 g parakeet) are on a comparable "per 1 SD of body condition"
# scale. The models themselves are fitted on the raw SMI.
smi_effect_rows = []
for species in inferential_species:
    data = intra_specific_datasets[species]
    smi_sd = float(data["SMI"].std())
    for metric, fit_info in intra_specific_models[species].items():
        if "SMI" not in fit_info["term_tests"]:
            continue
        model, family = fit_info["model"], fit_info["family"]
        names = list(model.params.index) if family != "beta" else list(model.model.exog_names)
        idx = names.index("SMI")
        if family == "beta":
            params, cov, _ = beta_mean_params(model)
            est, se = float(params[idx]), float(np.sqrt(cov[idx, idx]))
        else:
            est = float(np.asarray(model.params)[idx])
            se = float(np.sqrt(np.asarray(model.cov_params())[idx, idx]))
        est_std, se_std = est * smi_sd, se * smi_sd   # per 1 SD of SMI
        q = float(intra_global_effect_summary.query(
            "Species == @species and Term == 'SMI' and Metric == @metric")["q_value"].iloc[0])
        smi_effect_rows.append({
            "Species": species, "Metric": metric, "Family": family, "N": fit_info["N"],
            "Slope_per_SD": est_std, "ci_low": est_std - 1.96 * se_std, "ci_high": est_std + 1.96 * se_std,
            "q_value": q, "Significant_FDR": bool(q < ALPHA_SIGNIFICANCE),
        })
smi_effects = pd.DataFrame(smi_effect_rows)
smi_effects.to_csv(STATISTICS_DIR / "intra_specific_smi_effects.csv", index=False)

SCALE_NOTE = {"Richness": "log rate", "Shannon": "log ratio", "Simpson": "logit", "Pielou": "logit"}
fig, axes = plt.subplots(1, len(ALPHA_METRICS), figsize=(4.5 * len(ALPHA_METRICS), 4.8), squeeze=False)
axes = axes.ravel()
for ax, metric in zip(axes, ALPHA_METRICS):
    block = smi_effects[smi_effects["Metric"] == metric].set_index("Species").reindex(inferential_species).dropna(how="all")
    y_pos = np.arange(len(block))[::-1]
    for y, (species, row) in zip(y_pos, block.iterrows()):
        significant = bool(row["Significant_FDR"])
        ax.errorbar(row["Slope_per_SD"], y,
                    xerr=[[row["Slope_per_SD"] - row["ci_low"]], [row["ci_high"] - row["Slope_per_SD"]]],
                    fmt=SPECIES_MARKERS[species], markersize=9, capsize=4, linewidth=1.8,
                    color=SPECIES_COLORS[species], markeredgecolor="black" if significant else "white",
                    markeredgewidth=1.4 if significant else 0.8)
    ax.axvline(0, color="black", linewidth=1.2, linestyle="--")
    ax.set_yticks(y_pos)
    ax.set_yticklabels([SPECIES_LABELS_ITALIC.get(s, s) for s in block.index])
    ax.set_xlabel(f"Change in {metric} per +1 SD of SMI\n({SCALE_NOTE[metric]} scale)")
    ax.set_title(metric)
    ax.margins(y=0.3, x=0.12)
    # Reserve a clear column on the right for the q-values, so the labels
    # can never land on a confidence interval whatever the data look like.
    _x0, _x1 = ax.get_xlim()
    _span = _x1 - _x0
    ax.set_xlim(_x0, _x1 + 0.52 * _span)
    for y, (species, row) in zip(y_pos, block.iterrows()):
        ax.text(_x1 + 0.03 * _span, y,
                f"{format_p(row['q_value'], 'q')} {significance_stars(row['q_value'])}",
                ha="left", va="center", fontsize=8, color="0.4")

fig.suptitle("Within-species association between body condition (SMI) and gut alpha diversity",
             fontsize=14, fontweight="bold", y=1.06)
fig.text(0.5, 1.0, "Intervals crossing the dashed line indicate no detectable association; "
                     "a black marker outline would mark an effect significant after FDR correction",
         ha="center", va="top", fontsize=10, color="0.35")
figure_caption(fig,
    "Slopes come from each species' own model (terms selected from that species' replication: " +
    "; ".join(f"{s} N={intra_specific_term_decisions[s]['N']}, "
              f"{'Locality+' if intra_specific_term_decisions[s]['include_locality'] else ''}"
              f"{'Sex+' if intra_specific_term_decisions[s]['include_sex'] else ''}SMI"
              for s in inferential_species) +
    "). Slopes are rescaled to a 1-SD change in SMI within each species so that species of very different body "
    "mass are comparable; bars are 95% CIs. q-values are FDR-adjusted within each species' SMI family "
    f"(the {len(ALPHA_METRICS)} modeled metrics). Species-level effects of locality and sex are in "
    "results/statistics/intra_specific_global_effect_tests.csv.",
    y=-0.06)
plt.tight_layout()
intra_effects_fig_path = save_figure(fig, ALPHA_DIR / "intraspecific_alpha_diversity_effects.png")
plt.show(); plt.close(fig)

print("07.8 verification")
print("-" * 60)
assert intra_effects_fig_path.exists()
print(f"\nSaved: {intra_effects_fig_path}")
print("OK -- section 07.0 complete.")

In [ ]:
# Table · 07.8 — Body-condition slopes within each host species
show_table(smi_effects, "Body-condition slopes within each host species, rescaled to a one-standard-deviation change in SMI so that species of very different body mass are comparable. These are the estimates plotted in figure 07.8.")

## 08.0 — Beta diversity

Bray–Curtis dissimilarities computed on the rarefied matrix from section 05.0, so that every pair
of birds is compared at the same sequencing depth.

### Location and dispersion are tested separately

PERMANOVA tests whether groups differ, but its pseudo-*F* responds both to a shift in the average
community and to a difference in how variable a group is internally. Those are different
biological statements, and on unbalanced designs they are easy to confuse. Every PERMANOVA in this
section is therefore paired with a PERMDISP on the identical set of samples, and each significant
term is classified accordingly: a term whose matched PERMDISP is also significant is reported as a
difference in community *structure*, not as a difference in mean composition.

The PERMANOVA is marginal: each term is tested against the model containing the other terms. The
partition is implemented here from the definition in McArdle and Anderson (2001).

### Ordination

Principal coordinates analysis on the Bray–Curtis matrix, with the share of negative eigenvalues
reported so that the quality of the two-dimensional projection can be judged. The ordination is a
display of the distance matrix; every claim rests on the PERMANOVA, which uses the full matrix. A
non-metric multidimensional scaling of the same distances is fitted for its stress alone, because
that number is what decides which ordination a reader should be shown; the verdict printed in 08.2
follows from the value obtained. The compositional counterpart of this ordination, on Aitchison
distance, is in section 09.7.

### Invasive against native

Section 08.6 compares each invasive species against the pooled native species. Origin is
structurally confounded with locality, because invasive and native birds were not sampled in the
same proportions at every site, so the origin label is permuted **within localities** — the null
this design actually supports. Each invasive species is tested on its own and never pooled with
the other, and only if it carries enough birds to be modeled at all (02.2); the comparison is
framed as similarity rather than convergence, which would imply a temporal claim a
cross-sectional design cannot make.

In [ ]:
# 08.1 — Bray-Curtis distances on the rarefied matrix, with PCoA and PERMDISP functions
# ----------------------------------------------------------------------
import patsy
from sklearn.manifold import MDS


def bray_curtis_distance_matrix(community_matrix: pd.DataFrame) -> pd.DataFrame:
    condensed = pdist(community_matrix.values, metric="braycurtis")
    return pd.DataFrame(squareform(condensed), index=community_matrix.index, columns=community_matrix.index)


def gower_centered_matrix(distance_matrix: pd.DataFrame) -> np.ndarray:
    """G = -1/2 * J D^2 J  (Gower 1966): the Gram matrix PCoA and PERMANOVA both work on."""
    D = np.asarray(distance_matrix, dtype=float)
    n = D.shape[0]
    J = np.eye(n) - np.ones((n, n)) / n
    G = -0.5 * J @ (D ** 2) @ J
    return (G + G.T) / 2.0     # enforce exact symmetry against floating-point drift


def principal_coordinates(distance_matrix: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Classical PCoA: eigendecomposition of the Gower-centered matrix.

    Returns (eigenvalues, coordinates), both sorted by descending
    eigenvalue. Bray-Curtis is not a Euclidean metric, so some
    eigenvalues are negative; their axes are kept (scaled by
    sqrt(|lambda|)) because PERMDISP needs them, and the share of total
    |eigenvalue| they carry is reported rather than ignored.

    Computed from the definition rather than taken from a package, so the
    analysis carries no dependency beyond those listed in 01.1.
    """
    eigvals, eigvecs = np.linalg.eigh(gower_centered_matrix(distance_matrix))
    order = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    return eigvals, eigvecs * np.sqrt(np.abs(eigvals))


def permdisp(distance_matrix: pd.DataFrame, grouping: np.ndarray, n_permutations: int,
             rng: np.random.Generator) -> dict:
    """PERMDISP / betadisper (Anderson 2006): are within-group dispersions homogeneous?

    Each sample's distance to its group centroid is computed in the full
    principal-coordinate space, with the negative (imaginary) axes
    subtracted as Anderson prescribes:
        z_ij = sqrt(| d2_real - d2_imaginary |)
    A one-way ANOVA F-statistic on those distances is then referred to a
    null built by permuting the group labels, following Anderson (2006).

    This matters for interpreting PERMANOVA: a significant PERMANOVA with
    a significant PERMDISP can reflect unequal within-group variability
    rather than a difference in group centroids.
    """
    eigvals, coords = principal_coordinates(distance_matrix)
    tol = 1e-8 * max(1.0, float(np.abs(eigvals).max()))
    real, imaginary = coords[:, eigvals > tol], coords[:, eigvals < -tol]

    grouping = np.asarray(grouping)
    z = np.empty(len(grouping), dtype=float)
    for group in np.unique(grouping):
        idx = np.where(grouping == group)[0]
        d2_real = ((real[idx] - real[idx].mean(axis=0)) ** 2).sum(axis=1)
        d2_imag = (((imaginary[idx] - imaginary[idx].mean(axis=0)) ** 2).sum(axis=1)
                   if imaginary.shape[1] else np.zeros(len(idx)))
        z[idx] = np.sqrt(np.abs(d2_real - d2_imag))

    def anova_f(values: np.ndarray, labels: np.ndarray) -> float:
        groups = np.unique(labels)
        k, n = len(groups), len(values)
        if k < 2 or n <= k:
            return np.nan
        grand = values.mean()
        ss_between = sum(len(values[labels == g]) * (values[labels == g].mean() - grand) ** 2 for g in groups)
        ss_within = sum(((values[labels == g] - values[labels == g].mean()) ** 2).sum() for g in groups)
        if ss_within <= 0:
            return np.nan
        return (ss_between / (k - 1)) / (ss_within / (n - k))

    f_observed = anova_f(z, grouping)
    if not np.isfinite(f_observed):
        return {"F": np.nan, "p_value": np.nan, "group_mean_distance": {}, "distances": z, "N": len(z)}
    perm_f = np.array([anova_f(z, rng.permutation(grouping)) for _ in range(n_permutations)])
    perm_f = perm_f[np.isfinite(perm_f)]
    p = float((np.sum(perm_f >= f_observed) + 1) / (len(perm_f) + 1))
    return {
        "F": float(f_observed), "p_value": p, "N": len(z), "distances": z,
        "group_mean_distance": {str(g): float(z[grouping == g].mean()) for g in np.unique(grouping)},
    }


bray_curtis = bray_curtis_distance_matrix(rarefied_matrix)
bray_curtis.to_csv(DIVERSITY_DIR / "bray_curtis_distance_matrix.csv")

print("08.1 verification")
print("-" * 60)
assert bray_curtis.shape[0] == bray_curtis.shape[1] == rarefied_matrix.shape[0]
assert np.allclose(np.diag(bray_curtis.values), 0.0)
assert np.allclose(bray_curtis.values, bray_curtis.values.T)
assert (bray_curtis.values >= 0).all() and (bray_curtis.values <= 1).all()

# Self-test: on a Euclidean distance matrix, PCoA must reproduce the
# original configuration's inter-point distances exactly.
_pts = np.array([[0.0, 0.0], [3.0, 0.0], [0.0, 4.0], [3.0, 4.0], [1.5, 2.0]])
_d = pd.DataFrame(squareform(pdist(_pts)))
_ev, _co = principal_coordinates(_d)
assert np.allclose(squareform(pdist(_co[:, :2])), _d.values, atol=1e-8), "PCoA self-test failed."
assert (_ev[2:] < 1e-8).all(), "PCoA self-test: a 2-D configuration must have only 2 positive eigenvalues."
print("PCoA self-test passed (exactly reproduces a Euclidean configuration).")
print(f"Bray-Curtis distance matrix: {bray_curtis.shape[0]} x {bray_curtis.shape[0]} samples")
print(f"Saved: {DIVERSITY_DIR / 'bray_curtis_distance_matrix.csv'}")
print("OK -- distance matrix symmetric, zero-diagonal, bounded in [0, 1]; native PCoA/PERMDISP ready.")

In [ ]:
# 08.2 — Ordinate the Bray-Curtis matrix: PCoA axes, negative eigenvalues and NMDS stress
# --------------------------------------------------------------------------
# Bray-Curtis is not strictly Euclidean, so negative eigenvalues are
# expected; what matters is documenting how much of the total variation
# they represent rather than ignoring them. The ordination is a display of
# the distance matrix, not the test: PERMANOVA works on the full matrix
# and is unaffected by how well two axes happen to summarize it.
#
# A non-metric multidimensional scaling of the same distances is fitted
# here for one purpose only: its stress is the statistic that decides
# whether NMDS or PCoA is the ordination to present, and stating that
# number is what makes the choice auditable rather than stylistic. Only
# the stress is used; the NMDS configuration is not plotted, because an
# ordination carries no inference in this notebook and the PCoA axes are
# quantitative in a way NMDS axes are not.
pcoa_eigvals, pcoa_all_coords = principal_coordinates(bray_curtis)
positive_eigvals = pcoa_eigvals[pcoa_eigvals > 0]
negative_eigvals = pcoa_eigvals[pcoa_eigvals < 0]
negative_eigenvalue_share = float(np.abs(negative_eigvals).sum() / np.abs(pcoa_eigvals).sum())
pct_explained_axis1 = 100 * pcoa_eigvals[0] / positive_eigvals.sum()
pct_explained_axis2 = 100 * pcoa_eigvals[1] / positive_eigvals.sum()
pcoa_coords = pd.DataFrame(pcoa_all_coords[:, :2], index=bray_curtis.index, columns=["PCoA1", "PCoA2"])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _nmds = MDS(n_components=2, metric=False, dissimilarity="precomputed",
                random_state=RANDOM_SEED, n_init=8, normalized_stress=True, max_iter=500)
    _nmds.fit(bray_curtis.values)
nmds_stress = float(_nmds.stress_)
NMDS_INTERPRETABLE_STRESS = 0.20     # the conventional ceiling for reading a configuration

print("08.2 verification")
print("-" * 60)
print(f"PCoA: {len(positive_eigvals)} positive / {len(negative_eigvals)} negative eigenvalues")
print(f"Negative-eigenvalue share of total |eigenvalue|: {negative_eigenvalue_share:.4f}")
print(f"Axis 1 explains {pct_explained_axis1:.1f}% and axis 2 {pct_explained_axis2:.1f}% of the positive eigenvalues")
if negative_eigenvalue_share > 0.10:
    print("[NOTE] Negative-eigenvalue share > 10%: two PCoA axes are a rough summary of this matrix, so the "
          "ordination is read as a display and every claim rests on the PERMANOVA, which uses the full "
          "distance matrix rather than the 2-D projection.")
print(f"\nNMDS on the same distances: 2-D normalized stress = {nmds_stress:.4f} "
      f"(configurations are conventionally readable below {NMDS_INTERPRETABLE_STRESS:.2f}).")
if nmds_stress >= NMDS_INTERPRETABLE_STRESS:
    print("-> Too high for the arrangement to be interpreted: at this stress the two-dimensional "
          "layout, including any apparent separation of species, is largely an artifact of the "
          "projection. Section 08.7 therefore presents principal coordinates.")
else:
    print("-> Low enough for the arrangement to be readable. Section 08.7 still presents principal "
          "coordinates, whose axes carry an eigenvalue interpretation that NMDS axes do not, and the "
          "species and locality effects are tested by PERMANOVA on the full distance matrix either way.")
assert np.isfinite(negative_eigenvalue_share)
assert np.isfinite(nmds_stress)
assert pcoa_coords.shape == (bray_curtis.shape[0], 2)
print("OK -- PCoA computed, negative eigenvalues documented, and the ordination choice justified.")

In [ ]:
# 08.3 — Marginal PERMANOVA (McArdle & Anderson 2001) and its null-grouping self-test
# ----------------------------------------------------------------------
N_PERMUTATIONS = 999


def marginal_permanova(distance_matrix: pd.DataFrame, metadata: pd.DataFrame, terms: list[str],
                        n_permutations: int, rng: np.random.Generator) -> tuple[pd.DataFrame, dict]:
    """Marginal (Type-III-style) PERMANOVA for an arbitrary number of terms."""
    sample_ids = distance_matrix.index.tolist()
    G = gower_centered_matrix(distance_matrix.loc[sample_ids, sample_ids])
    n = len(sample_ids)
    identity = np.eye(n)
    total_ss = float(np.trace(G))

    md_here = metadata.loc[sample_ids]
    X_full = np.asarray(patsy.dmatrix("~ " + " + ".join(terms), md_here, return_type="dataframe"))
    rank_full = np.linalg.matrix_rank(X_full)
    H_full = X_full @ np.linalg.pinv(X_full)
    df_residual = n - rank_full
    if df_residual <= 0:
        raise ValueError(f"PERMANOVA has no residual degrees of freedom (n={n}, model rank={rank_full}).")

    term_matrices = {}
    for term in terms:
        reduced = [t for t in terms if t != term]
        X_reduced = np.asarray(patsy.dmatrix("~ " + " + ".join(reduced) if reduced else "~ 1",
                                             md_here, return_type="dataframe"))
        term_matrices[term] = (X_reduced @ np.linalg.pinv(X_reduced),
                               rank_full - np.linalg.matrix_rank(X_reduced))

    def compute_stats(G_mat):
        ss_resid = float(np.trace((identity - H_full) @ G_mat))
        out = {}
        for term, (H_reduced, df_term) in term_matrices.items():
            ss_term = float(np.trace((H_full - H_reduced) @ G_mat))
            out[term] = (ss_term, df_term,
                         (ss_term / df_term) / (ss_resid / df_residual) if df_term > 0 and ss_resid > 0 else np.nan)
        return out

    observed = compute_stats(G)
    perm_F = {term: np.empty(n_permutations) for term in terms}
    for i in range(n_permutations):
        perm_idx = rng.permutation(n)
        stats_perm = compute_stats(G[np.ix_(perm_idx, perm_idx)])
        for term in terms:
            perm_F[term][i] = stats_perm[term][2]

    rows = []
    for term in terms:
        ss_term, df_term, pseudo_f = observed[term]
        finite = perm_F[term][np.isfinite(perm_F[term])]
        rows.append({"Term": term, "SS": ss_term, "df": df_term, "pseudo_F": pseudo_f,
                     "R2": ss_term / total_ss,
                     "p_value": float((np.sum(finite >= pseudo_f) + 1) / (len(finite) + 1)), "N": n})
    return pd.DataFrame(rows), {"total_ss": total_ss, "df_residual": df_residual, "n": n, "sample_ids": sample_ids}


print("08.3 verification")
print("-" * 60)
# Self-test: with a group that has no effect, the pseudo-F must be small
# and the p-value must not be significant.
_rng_test = np.random.default_rng(0)
_x = _rng_test.normal(size=(40, 5))
_dtest = pd.DataFrame(squareform(pdist(_x)), index=[f"s{i}" for i in range(40)], columns=[f"s{i}" for i in range(40)])
_mdtest = pd.DataFrame({"g": ["a", "b"] * 20}, index=_dtest.index)
_res, _ = marginal_permanova(_dtest, _mdtest, ["C(g)"], 199, np.random.default_rng(1))
assert 0 < _res["p_value"].iloc[0] <= 1
assert _res["R2"].iloc[0] < 0.20, "PERMANOVA self-test: a null grouping should explain little variation."
print(f"PERMANOVA self-test passed (null grouping: pseudo-F={_res['pseudo_F'].iloc[0]:.2f}, "
      f"R2={_res['R2'].iloc[0]:.3f}, p={_res['p_value'].iloc[0]:.3f}).")
print(f"N_PERMUTATIONS = {N_PERMUTATIONS}")
print("OK -- native marginal-PERMANOVA function defined and sanity-checked.")

In [ ]:
# 08.4 — Community structure ~ Species + Locality, with PERMDISP on the same samples
# ----------------------------------------------------------------------
# Same complete-case, single-species-locality exclusion and the same
# modeled-species set as the inter-specific alpha dataset (06.2), so the
# global beta test and the global alpha tests describe the same birds and
# answer the same question about the same species.
beta_global_metadata = analysis_metadata.loc[
    analysis_metadata.index.intersection(bray_curtis.index), ["Species_code", "Locality"]].dropna()
beta_global_metadata = beta_global_metadata.loc[~beta_global_metadata["Locality"].isin(excluded_localities)]
beta_global_metadata = beta_global_metadata.loc[
    beta_global_metadata["Species_code"].isin(modeled_species)]
beta_global_ids = beta_global_metadata.index.tolist()
beta_global_distance = bray_curtis.loc[beta_global_ids, beta_global_ids]

_permanova_rng = spawn_rng()
global_permanova, global_permanova_meta = marginal_permanova(
    beta_global_distance, beta_global_metadata, terms=[SPECIES_TERM, LOCALITY_TERM],
    n_permutations=N_PERMUTATIONS, rng=_permanova_rng)
global_permanova["q_value"] = fdr_within_family(
    dict(zip(global_permanova["Term"], global_permanova["p_value"])))["q_value"].values
global_permanova.to_csv(STATISTICS_DIR / "global_permanova.csv", index=False)

# PERMDISP on the EXACT same samples (a mismatched subset would make the
# two tests non-comparable).
_permdisp_rng = spawn_rng()
global_permdisp_species = permdisp(beta_global_distance,
                                   beta_global_metadata["Species_code"].astype(str).values,
                                   N_PERMUTATIONS, _permdisp_rng)
global_permdisp_locality = permdisp(beta_global_distance,
                                    beta_global_metadata["Locality"].astype(str).values,
                                    N_PERMUTATIONS, _permdisp_rng)

print("08.4 verification")
print("-" * 60)
print(f"Global PERMANOVA N = {global_permanova_meta['n']} "
      f"(PERMDISP below uses the identical {len(beta_global_ids)} samples)")
print(f"\nPERMDISP (Species grouping) : F = {global_permdisp_species['F']:.3f}, p = {global_permdisp_species['p_value']:.4g}")
print("  Mean distance to the group centroid, by species (higher = more variable community):")
for _g, _v in sorted(global_permdisp_species["group_mean_distance"].items(), key=lambda kv: -kv[1]):
    print(f"    {_g:8s} {_v:.4f}")
print(f"PERMDISP (Locality grouping): F = {global_permdisp_locality['F']:.3f}, p = {global_permdisp_locality['p_value']:.4g}")

assert global_permanova_meta["n"] == len(beta_global_ids) == beta_global_distance.shape[0]
assert global_permdisp_species["N"] == global_permanova_meta["n"], "PERMDISP and PERMANOVA must use the same samples."
if global_permdisp_species["p_value"] < ALPHA_SIGNIFICANCE:
    print("\n[INTERPRETATION] PERMDISP is significant for Species: part of the PERMANOVA Species effect may "
          "reflect a difference in within-species VARIABILITY rather than purely a shift in the average "
          "community. Report both tests together, and describe the species effect as a difference in "
          "community structure (location and/or dispersion), not as a pure centroid shift.")
else:
    print("\n[INTERPRETATION] PERMDISP is not significant for Species: within-species dispersions are "
          "homogeneous, so the PERMANOVA Species effect can be read as a genuine difference in average "
          "community composition.")
print(f"\nSaved: {STATISTICS_DIR / 'global_permanova.csv'}")
print("OK -- section 08.4 complete.")

In [ ]:
# Table · 08.4 — Marginal PERMANOVA of community structure
show_table(global_permanova, "Marginal PERMANOVA of community structure on the Bray-Curtis matrix of the rarefied matrix, each term tested against the model holding the other.")

In [ ]:
# 08.5 — Within-species community structure ~ Locality + Sex + SMI, with matched PERMDISP
# ----------------------------------------------------------------------
# Reuses the SAME term-inclusion decisions as the intra-specific alpha
# models (07.3), so the two analyses answer the same question on the same
# samples, and descriptive-only species stay descriptive-only here too.
species_permanova_results, species_permdisp_results = {}, {}

for species in inferential_species:
    decision = intra_specific_term_decisions[species]
    species_ids = [i for i in intra_specific_datasets[species].index if i in bray_curtis.index]
    if len(species_ids) < MIN_N_FOR_INFERENCE:
        print(f"[NOTE] {species}: only {len(species_ids)} samples have both a rarefied community and the "
              f"model covariates -- below MIN_N_FOR_INFERENCE, so no per-species PERMANOVA is fitted.")
        continue
    species_metadata = intra_specific_datasets[species].loc[species_ids, ["Locality", "Sex", "SMI"]].copy()
    species_distance = bray_curtis.loc[species_ids, species_ids]

    terms = []
    if decision["include_locality"]:
        terms.append(LOCALITY_TERM)
    if decision["include_sex"]:
        terms.append("C(Sex)")
    terms.append("SMI")

    table, meta = marginal_permanova(species_distance, species_metadata, terms, N_PERMUTATIONS, spawn_rng())
    table["q_value"] = fdr_within_family(dict(zip(table["Term"], table["p_value"])))["q_value"].values
    species_permanova_results[species] = (table, meta)

    disp = {}
    _rng_species = spawn_rng()
    if decision["include_locality"]:
        disp["Locality"] = permdisp(species_distance, species_metadata["Locality"].astype(str).values,
                                    N_PERMUTATIONS, _rng_species)
    if decision["include_sex"]:
        disp["Sex"] = permdisp(species_distance, species_metadata["Sex"].astype(str).values,
                               N_PERMUTATIONS, _rng_species)
    species_permdisp_results[species] = disp

species_permanova_all = pd.concat(
    [t.assign(Species=s) for s, (t, _) in species_permanova_results.items()], ignore_index=True)
species_permanova_all["Stars"] = species_permanova_all["q_value"].map(significance_stars)
species_permanova_all.to_csv(STATISTICS_DIR / "species_permanova.csv", index=False)

# A significant PERMANOVA term whose MATCHED PERMDISP is also significant
# cannot be read as a difference in average community composition: groups
# that differ in how spread out they are produce large pseudo-F values
# without any centroid shift at all. Section 08.4 says this for the global
# test; here it is attached to each individual term, because that is where
# the manuscript's sentences come from, and it is the difference between
# a defensible claim and one a reviewer will overturn.
species_permanova_all["PERMDISP_F"] = np.nan
species_permanova_all["PERMDISP_p"] = np.nan
for _i, _row in species_permanova_all.iterrows():
    _factor = _row["Term"].replace("C(", "").replace(")", "")
    _disp = species_permdisp_results.get(_row["Species"], {}).get(_factor)
    if _disp is not None:
        species_permanova_all.loc[_i, "PERMDISP_F"] = _disp["F"]
        species_permanova_all.loc[_i, "PERMDISP_p"] = _disp["p_value"]
species_permanova_all["Dispersion_confounded"] = (
    species_permanova_all["q_value"].lt(ALPHA_SIGNIFICANCE)
    & species_permanova_all["PERMDISP_p"].lt(ALPHA_SIGNIFICANCE))
species_permanova_all["Interpretation"] = np.where(
    ~species_permanova_all["q_value"].lt(ALPHA_SIGNIFICANCE), "no effect detected",
    np.where(species_permanova_all["Dispersion_confounded"],
             "structure differs (location and/or dispersion) -- NOT a centroid shift",
             "centroid shift (dispersions homogeneous)"))
species_permanova_all.to_csv(STATISTICS_DIR / "species_permanova.csv", index=False)

print("08.5 verification")
print("-" * 60)
for species, (table, meta) in species_permanova_results.items():
    print(f"\n{SPECIES_CODE_TO_NAME[species]} ({species}), N = {meta['n']}:")
    print(table.round(4).to_string(index=False))
    for factor, disp in species_permdisp_results[species].items():
        print(f"  PERMDISP ({factor}, matched N={disp['N']}): F = {disp['F']:.3f}, p = {disp['p_value']:.4g}")
        assert disp["N"] == meta["n"], "PERMDISP and PERMANOVA must use the same samples."

_significant = species_permanova_all[species_permanova_all["q_value"].lt(ALPHA_SIGNIFICANCE)]
_n_sig = len(_significant)
print(f"\n{_n_sig}/{len(species_permanova_all)} per-species PERMANOVA terms are significant after FDR "
      f"correction within each species' own family.")
if _n_sig:
    print("\nHow each significant term may be worded, given its matched PERMDISP:")
    print(_significant[["Species", "Term", "R2", "q_value", "PERMDISP_F", "PERMDISP_p",
                        "Interpretation"]].round(4).to_string(index=False))
    _confounded = _significant[_significant["Dispersion_confounded"]]
    if not _confounded.empty:
        print(f"\n[INTERPRETATION] {len(_confounded)}/{_n_sig} significant term(s) have a significant "
              f"PERMDISP on the same samples: "
              f"{', '.join(f'{r.Species} {r.Term}' for r in _confounded.itertuples())}. "
              f"For these, the groups differ in within-group VARIABILITY, and PERMANOVA's pseudo-F is "
              f"sensitive to that as well as to a shift in the mean community. They must be described as "
              f"'differ in community structure', never as 'differ in mean composition', and the PERMDISP "
              f"result must be reported alongside.")
    _clean = _significant[~_significant["Dispersion_confounded"]]
    if not _clean.empty:
        print(f"[INTERPRETATION] {len(_clean)} significant term(s) have homogeneous dispersions "
              f"({', '.join(f'{r.Species} {r.Term}' for r in _clean.itertuples())}) and CAN be read as a "
              f"genuine difference in average community composition. These are the strongest "
              f"within-species results in the study.")
print(f"Saved: {STATISTICS_DIR / 'species_permanova.csv'}")
print("OK -- section 08.5 complete; PERMDISP matches its PERMANOVA sample set in every case.")

In [ ]:
# Table · 08.5 — Within-species PERMANOVA, each paired with its PERMDISP
show_table(species_permanova_all, "Within-species PERMANOVA on each species' own Bray-Curtis submatrix, each result paired with the PERMDISP computed on the identical samples and with the wording that pairing licenses.", page_length=15)

In [ ]:
# 08.6 — Each invasive species against the pooled natives, permuting within localities
# ----------------------------------------------------------------------
# Framed as SIMILARITY, not "convergence", which would imply a
# directional/temporal claim this cross-sectional design cannot support.
# The two invasive species are tested SEPARATELY against the pooled
# natives, never pooled with each other.
#
# A locality-RESTRICTED permutation is used here (unlike the unrestricted
# permutation in 08.3-08.5): origin is structurally confounded with
# locality, because invasive and native species were not sampled in the
# same proportions at every site. The origin label is therefore permuted
# only WITHIN a locality, which is the null that the design actually
# supports.
def restricted_origin_permanova(distance_matrix: pd.DataFrame, metadata: pd.DataFrame,
                                 n_permutations: int, rng: np.random.Generator) -> dict:
    sample_ids = distance_matrix.index.tolist()
    G = gower_centered_matrix(distance_matrix.loc[sample_ids, sample_ids])
    n = len(sample_ids)
    identity = np.eye(n)

    md_here = metadata.loc[sample_ids]
    X_full = np.asarray(patsy.dmatrix("~ C(Origin) + C(Locality)", md_here, return_type="dataframe"))
    X_reduced = np.asarray(patsy.dmatrix("~ C(Locality)", md_here, return_type="dataframe"))
    H_full, H_reduced = X_full @ np.linalg.pinv(X_full), X_reduced @ np.linalg.pinv(X_reduced)
    df_term = np.linalg.matrix_rank(X_full) - np.linalg.matrix_rank(X_reduced)
    df_residual = n - np.linalg.matrix_rank(X_full)
    if df_term <= 0 or df_residual <= 0:
        raise ValueError("Origin is not identifiable after conditioning on Locality in this subset.")

    ss_observed = float(np.trace((H_full - H_reduced) @ G))
    ss_resid = float(np.trace((identity - H_full) @ G))
    f_observed = (ss_observed / df_term) / (ss_resid / df_residual)

    locality_groups = md_here.groupby("Locality", observed=True).indices
    origin_values = md_here["Origin"].values.copy()
    perm_f = np.empty(n_permutations)
    for i in range(n_permutations):
        permuted = origin_values.copy()
        for _, idx in locality_groups.items():
            permuted[idx] = rng.permutation(origin_values[idx])
        md_perm = md_here.copy()
        md_perm["Origin"] = permuted
        X_perm = np.asarray(patsy.dmatrix("~ C(Origin) + C(Locality)", md_perm, return_type="dataframe"))
        H_perm = X_perm @ np.linalg.pinv(X_perm)
        ss_t = float(np.trace((H_perm - H_reduced) @ G))
        ss_r = float(np.trace((identity - H_perm) @ G))
        perm_f[i] = (ss_t / df_term) / (ss_r / df_residual)
    finite = perm_f[np.isfinite(perm_f)]
    return {"pseudo_F": f_observed, "df": df_term, "df_residual": df_residual,
            "R2": ss_observed / float(np.trace(G)),
            "p_value": float((np.sum(finite >= f_observed) + 1) / (len(finite) + 1)), "N": n}


invasive_similarity_results = {}
for invasive_species in sorted(INVASIVE_SPECIES):
    origin_metadata = analysis_metadata.loc[
        analysis_metadata.index.intersection(bray_curtis.index), ["Species_code", "Locality"]].dropna().copy()
    origin_metadata = origin_metadata.loc[
        (origin_metadata["Species_code"] == invasive_species)
        | (origin_metadata["Species_code"].isin(NATIVE_SPECIES))]
    origin_metadata["Origin"] = np.where(origin_metadata["Species_code"] == invasive_species, "Invasive", "Native")
    shared = origin_metadata.groupby("Locality", observed=True)["Origin"].nunique()
    origin_metadata = origin_metadata.loc[origin_metadata["Locality"].isin(shared[shared >= 2].index)]
    origin_ids = [i for i in origin_metadata.index if i in bray_curtis.index]
    origin_metadata = origin_metadata.loc[origin_ids]
    n_invasive = int((origin_metadata["Origin"] == "Invasive").sum())

    # An invasive species held back from the between-species models (02.2)
    # is held back here too: the same handful of birds cannot support an
    # origin contrast either, and testing it here while omitting it
    # everywhere else would be inconsistent.
    if invasive_species not in modeled_species or n_invasive < MIN_N_FOR_INFERENCE:
        invasive_similarity_results[invasive_species] = {
            "descriptive_only": True, "n_invasive": n_invasive,
            "n_native": len(origin_metadata) - n_invasive}
        continue
    result = restricted_origin_permanova(bray_curtis.loc[origin_ids, origin_ids], origin_metadata,
                                         N_PERMUTATIONS, spawn_rng())
    # The matched PERMDISP was missing here, and this is the one test in
    # the notebook that most needs it: it is the study's headline
    # comparison, and an invasive species whose birds all carry a similar
    # community will produce a large pseudo-F against heterogeneous
    # natives WITHOUT its average community being different at all. Read
    # without this, "invasive differs from native" can mean the opposite
    # of what it appears to.
    origin_disp = permdisp(bray_curtis.loc[origin_ids, origin_ids],
                           origin_metadata["Origin"].astype(str).values,
                           N_PERMUTATIONS, spawn_rng())
    invasive_similarity_results[invasive_species] = {
        "descriptive_only": False, "n_invasive": n_invasive,
        "n_native": len(origin_metadata) - n_invasive, **result,
        "PERMDISP_F": origin_disp["F"], "PERMDISP_p": origin_disp["p_value"],
        "Dispersion_invasive": origin_disp["group_mean_distance"].get("Invasive", np.nan),
        "Dispersion_native": origin_disp["group_mean_distance"].get("Native", np.nan)}

print("08.6 verification")
print("-" * 60)
for species, result in invasive_similarity_results.items():
    name = SPECIES_CODE_TO_NAME[species]
    if result["descriptive_only"]:
        print(f"{name} ({species}): descriptive only -- N_invasive = {result['n_invasive']} "
              f"< MIN_N_FOR_INFERENCE = {MIN_N_FOR_INFERENCE}. No locality-restricted PERMANOVA is fitted; "
              f"any apparent pattern must be described qualitatively only.")
    else:
        print(f"{name} ({species}): N_invasive={result['n_invasive']}, N_native={result['n_native']}, "
              f"pseudo-F={result['pseudo_F']:.3f}, R2={result['R2']:.4f}, p={result['p_value']:.4g}")
        print(f"  Matched PERMDISP (Invasive vs Native): F = {result['PERMDISP_F']:.3f}, "
              f"p = {result['PERMDISP_p']:.4g}   "
              f"[mean distance to centroid: invasive {result['Dispersion_invasive']:.3f}, "
              f"native {result['Dispersion_native']:.3f}]")
        if result["p_value"] < ALPHA_SIGNIFICANCE and result["PERMDISP_p"] < ALPHA_SIGNIFICANCE:
            _more = "more" if result["Dispersion_invasive"] > result["Dispersion_native"] else "less"
            print(f"  [INTERPRETATION] Both tests are significant, and {species} birds are {_more} "
                  f"variable among themselves than the natives. The PERMANOVA result therefore cannot be "
                  f"reported as 'the invasive community differs on average from the native one'. What the "
                  f"data support is that the two groups differ in community STRUCTURE, and the clearest "
                  f"single component of that difference is the difference in within-group homogeneity. "
                  f"State it that way, and report both statistics together.")
        elif result["p_value"] < ALPHA_SIGNIFICANCE:
            print(f"  [INTERPRETATION] PERMANOVA is significant and dispersions are homogeneous, so this "
                  f"can be read as a genuine difference in average community composition between "
                  f"{species} and the pooled natives, within shared localities.")
invasive_similarity_summary = pd.DataFrame(
    [{"Invasive_species": s, "Species_name": SPECIES_CODE_TO_NAME[s], **r}
     for s, r in invasive_similarity_results.items()])
invasive_similarity_summary.to_csv(STATISTICS_DIR / "invasive_native_similarity.csv", index=False)
print(f"\nSaved: {STATISTICS_DIR / 'invasive_native_similarity.csv'}")
print("OK -- section 08.6 complete.")

In [ ]:
# Table · 08.6 — Each invasive species against the pooled natives
show_table(invasive_similarity_summary, "Each invasive species against the pooled natives on the Bray-Curtis matrix, with the origin label permuted within localities and a matched PERMDISP.")

In [ ]:
# 08.7 — Figure: Bray-Curtis ordination and within-species dispersion
# ----------------------------------------------------------------------
species_row = global_permanova.set_index("Term").loc[SPECIES_TERM]
locality_row = global_permanova.set_index("Term").loc[LOCALITY_TERM]

fig = plt.figure(figsize=(13.5, 5.8))
gs = fig.add_gridspec(1, 2, width_ratios=[1.15, 1], wspace=0.26)
ax_pcoa, ax_disp = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])


def draw_ordination(ax, coords, xlabel, ylabel, title):
    """Scatter one ordination, one marker and color per host species, with centroids.

    The ordination is unconstrained, so every bird with a community
    profile is drawn, including those of a species too sparsely sampled
    to enter the models; their points are open rather than filled, and
    the legend says so, so the figure is never read as showing a species
    that the PERMANOVA beside it actually tested.
    """
    for species in SPECIES_ORDER:
        ids = analysis_metadata.index[analysis_metadata["Species_code"] == species].intersection(coords.index)
        if len(ids) == 0:
            continue
        _described_only = species not in modeled_species
        pts = coords.loc[ids]
        ax.scatter(pts.iloc[:, 0], pts.iloc[:, 1],
                   facecolor="none" if _described_only else SPECIES_COLORS[species],
                   edgecolor=SPECIES_COLORS[species] if _described_only else "white",
                   marker=SPECIES_MARKERS[species],
                   label=species_legend_label(species) + (" - described only" if _described_only else ""),
                   alpha=0.85, s=42, linewidth=1.1 if _described_only else 0.6, zorder=3)
        if _described_only:
            continue
        # The species centroid, so the difference the PERMANOVA tests is
        # visible on the figure rather than left to the eye.
        ax.scatter(pts.iloc[:, 0].mean(), pts.iloc[:, 1].mean(), color=SPECIES_COLORS[species],
                   marker=SPECIES_MARKERS[species], s=190, edgecolor="black", linewidth=1.5, zorder=4)
    ax.axhline(0, color="0.85", linewidth=0.7, zorder=0)
    ax.axvline(0, color="0.85", linewidth=0.7, zorder=0)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title, fontsize=11)


# Panel statistics go in the panel TITLE rather than in a text box inside
# the axes, where they can land on top of the data.
draw_ordination(ax_pcoa, pcoa_coords,
                f"PCoA axis 1 ({pct_explained_axis1:.1f}% of variation)",
                f"PCoA axis 2 ({pct_explained_axis2:.1f}% of variation)",
                f"A. Principal coordinates on Bray-Curtis dissimilarity\n"
                f"PERMANOVA species effect: $R^2$ = {species_row['R2']:.3f}, "
                f"{format_p(species_row['p_value'])} {significance_stars(species_row['p_value'])}")

# Panel B: the quantity PERMDISP tests, drawn directly. A difference in
# within-group spread is invisible in an ordination, and on these data it
# is a substantial part of what separates the host species.
disp_values = global_permdisp_species["distances"]
disp_groups = beta_global_metadata["Species_code"].astype(str).values
order_c = [s for s in SPECIES_ORDER if s in set(disp_groups)]
_order_c_plot = list(reversed(order_c))
for i, s in enumerate(_order_c_plot, start=1):
    vals = disp_values[disp_groups == s]
    ax_disp.boxplot([vals], positions=[i], widths=0.55, showfliers=False, patch_artist=True,
                    vert=False,
                    medianprops=dict(color="black", linewidth=1.5),
                    whiskerprops=dict(color="0.45"), capprops=dict(color="0.45"),
                    boxprops=dict(facecolor=SPECIES_COLORS[s], alpha=0.28,
                                  edgecolor=SPECIES_COLORS[s], linewidth=1.3))
    jitter = spawn_rng().normal(0, 0.06, size=len(vals))
    ax_disp.scatter(vals, np.full(len(vals), i) + jitter, s=28, marker=SPECIES_MARKERS[s],
                    color=SPECIES_COLORS[s], edgecolor="white", linewidth=0.5, alpha=0.9, zorder=3)
ax_disp.set_yticks(range(1, len(_order_c_plot) + 1))
ax_disp.set_yticklabels(
    [f"{abbreviated_binomial(s)}\n(n = {int((disp_groups == s).sum())})" for s in _order_c_plot],
    fontsize=8.5)
ax_disp.set_title(f"B. Within-species dispersion\nPERMDISP: F = {global_permdisp_species['F']:.2f}, "
                  f"{format_p(global_permdisp_species['p_value'])} "
                  f"{significance_stars(global_permdisp_species['p_value'])}", fontsize=11)
ax_disp.set_xlabel("Distance to species centroid (Bray-Curtis)")

handles, labels = ax_pcoa.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(labels), bbox_to_anchor=(0.5, -0.08),
           title="Species (origin): warm colors = invasive, cool colors = native", title_fontsize=9)
fig.suptitle("Bacterial community structure by host species: ordination and dispersion",
             fontsize=14, fontweight="bold", y=1.02)
figure_caption(fig,
    f"Bray-Curtis dissimilarities computed on the community matrix rarefied to {rarefaction_depth:,} reads "
    f"(N = {global_permanova_meta['n']} birds). A: each small point is one bird's community, large "
    f"black-outlined symbols are species centroids; axes 1 and 2 together capture "
    f"{pct_explained_axis1 + pct_explained_axis2:.1f}% of the positive eigenvalues, and negative "
    f"eigenvalues account for {negative_eigenvalue_share:.1%} of the total, so the ordination is a summary "
    f"while the test uses the full distance matrix. "
    + (f"Open symbols are birds of "
       f"{', '.join(SPECIES_CODE_TO_NAME[s] for s in species_described_only)}, shown because the "
       f"ordination is unconstrained but omitted from the test and given no centroid: "
       f"too few birds to estimate a species mean (02.2). " if species_described_only else "")
    + f"Marginal PERMANOVA ({N_PERMUTATIONS} permutations): "
    f"Species pseudo-F = {species_row['pseudo_F']:.2f}, R2 = {species_row['R2']:.3f}, "
    f"p = {species_row['p_value']:.3f}, q = {species_row['q_value']:.3f}; Locality pseudo-F = "
    f"{locality_row['pseudo_F']:.2f}, R2 = {locality_row['R2']:.3f}, p = {locality_row['p_value']:.3f}, "
    f"q = {locality_row['q_value']:.3f}. B: distance from each bird to its own species centroid, the "
    f"quantity PERMDISP tests. PERMANOVA is sensitive to a difference in within-group spread as well as "
    f"to a shift in the average community, so the two panels have to be read together. The compositional "
    f"counterpart of panel A, on Aitchison distance, is figure 09.7.",
    y=-0.15)
beta_ordination_fig_path = save_figure(fig, BETA_DIR / "beta_diversity_ordination.png")
plt.show(); plt.close(fig)

print("08.7 verification")
print("-" * 60)
assert beta_ordination_fig_path.exists()
print(f"Saved: {beta_ordination_fig_path}")
print("OK -- section 08.0 complete.")

## 09.0 — Taxonomic composition

Two parts, kept separate because they answer different questions and rest on different
assumptions.

**Descriptive composition (09.1–09.4)** works on the curated counts of every library that passed
the quality floor. It reports which genera are abundant, which are prevalent, and — importantly on
these data — how far those two things diverge. A genus with a high mean relative abundance and a
near-zero median dominates a few birds rather than being a consistent member of the community, so
the per-genus, per-bird distributions are plotted rather than only the species means.

**Compositional analysis (09.5–09.7)** treats the data as what they are: compositions, carrying
relative rather than absolute information. Counts are transformed to centered log-ratio
coordinates, so that the analysis is driven by ratios between genera rather than by the total
depth of a library. Two steps precede the transform. A prevalence filter removes genera too sparse
for a log-ratio to be meaningful, and the remaining zeros are replaced multiplicatively
(Martín-Fernández, Barceló-Vidal & Pawlowsky-Glahn 2003) rather than with a blanket pseudocount,
which preserves the ratios between the parts that were actually observed.

Euclidean distance on CLR coordinates is Aitchison distance, so the PERMANOVA in 09.6 and the
ordination in 09.7 are exact companions of each other. Because this route uses non-rarefied counts
and a different distance, agreement with the Bray–Curtis result of section 08.0 is a genuine
robustness check rather than a restatement.

In [ ]:
# 09.1 — composition_counts: relative abundance and prevalence of every curated genus
# ----------------------------------------------------------------------
# Composition is described on the libraries that passed the quality
# floor, which is the same sample set the rest of the notebook uses. A
# relative abundance computed from a few hundred reads carries almost no
# information about a community but would enter a species mean with the
# same weight as one computed from half a million, so failed libraries
# are excluded here as they are everywhere else.
composition_ids = [i for i in community_matrix_analysis.index if i in set(qc_passing_ids)]
composition_counts = community_matrix_analysis.loc[composition_ids]
composition_metadata = analysis_metadata.loc[composition_ids]
_n_excluded_from_composition = community_matrix_analysis.shape[0] - len(composition_ids)

relative_abundance = composition_counts.div(composition_counts.sum(axis=1), axis=0)
relative_abundance_with_species = relative_abundance.join(composition_metadata["Species_code"])

composition_summary = pd.DataFrame({
    "Genus": composition_counts.columns,
    "Mean_relative_abundance": relative_abundance.mean().values,
    "Median_relative_abundance": relative_abundance.median().values,
    "Prevalence_fraction": (composition_counts > 0).mean().values,
    "N_samples_present": (composition_counts > 0).sum().values,
}).sort_values("Mean_relative_abundance", ascending=False).reset_index(drop=True)
composition_summary.to_csv(TAXONOMY_RESULTS_DIR / "genus_composition_summary.csv", index=False)

print("09.1 verification")
print("-" * 60)
print(f"Samples described here: {len(composition_ids)}/{community_matrix_analysis.shape[0]} "
      f"({_n_excluded_from_composition} failed librar(ies) below the {MIN_LIBRARY_DEPTH:,}-read floor "
      f"excluded, as everywhere else in the notebook)")
print(f"Genera in the confirmed analytical matrix: {len(composition_summary)}")
_top = composition_summary.iloc[0]
print(f"\n[READING THE TABLE] The mean and the median relative abundance differ sharply for the most "
      f"abundant genera (e.g. {_top['Genus']}: mean {_top['Mean_relative_abundance']:.3f} vs median "
      f"{_top['Median_relative_abundance']:.3f}, present in {_top['Prevalence_fraction']:.0%} of samples). "
      f"A high mean with a near-zero median means a genus dominates a FEW birds rather than being a "
      f"consistent member of the community -- which is exactly what figure 09.3 is for.")
assert np.allclose(relative_abundance.sum(axis=1).values, 1.0)
print(f"\nSaved: {TAXONOMY_RESULTS_DIR / 'genus_composition_summary.csv'}")
print("OK -- descriptive composition table complete.")

In [ ]:
# Table · 09.1 — Relative abundance and prevalence of every curated genus
show_table(composition_summary, "Relative abundance and prevalence of every curated genus, from composition_counts. Sort by prevalence and then by mean abundance to separate the genera that are consistently present from those that dominate a few birds.", page_length=15)

In [ ]:
# 09.2 — Figure: mean genus composition of each host species
# ----------------------------------------------------------------------
# Color choice carries meaning in a stacked bar, where adjacent
# segments are read against each other. The palette is a 12-hue
# color-blind-safe qualitative set chosen so that no two hues are
# confusable when stacked, and the two segments that are not taxa are
# deliberately given no hue at all: plain gray for the pooled remaining
# genera, hatched white for excluded and unresolved reads. Both then read
# at a glance as bookkeeping rather than as bacteria.
N_TOP_GENERA_DISPLAY = 12
top_genera = composition_summary["Genus"].head(N_TOP_GENERA_DISPLAY).tolist()

mean_by_species = (relative_abundance_with_species
                   .groupby("Species_code", observed=True)[list(composition_counts.columns)]
                   .mean().reindex(SPECIES_ORDER).dropna(how="all"))
plot_order = list(mean_by_species.index)
plot_matrix = mean_by_species[top_genera].copy()
plot_matrix["Other confirmed genera (pooled)"] = mean_by_species.drop(columns=top_genera).sum(axis=1)

# The excluded/unresolved read share is ONE additional descriptive-only
# segment, computed on the ORIGINAL pre-curation read totals. It is
# bookkeeping, not a taxon, and is never used in any statistical test.
excluded_share_by_species = (excluded_summary.loc[composition_ids]
                             .join(composition_metadata["Species_code"], how="inner")
                             .groupby("Species_code", observed=True)["Excluded_or_unresolved_share"]
                             .mean().reindex(plot_order).fillna(0))
plot_matrix = plot_matrix.mul(1 - excluded_share_by_species, axis=0)
plot_matrix["Excluded / unresolved reads (not a taxon)"] = excluded_share_by_species

fig, ax = plt.subplots(figsize=(11.5, 6.4))
bottom = np.zeros(len(plot_matrix))
x = np.arange(len(plot_matrix))
for i, col in enumerate(plot_matrix.columns):
    if col.startswith("Other confirmed"):
        style = dict(color=COLOR_OTHER, edgecolor="0.55", linewidth=0.6)
    elif col.startswith("Excluded"):
        style = dict(color=COLOR_BOOKKEEPING, edgecolor="0.4", linewidth=0.8, hatch="////")
    else:
        style = dict(color=GENUS_PALETTE[i % len(GENUS_PALETTE)], edgecolor="white", linewidth=0.4)
    label = col if (col.startswith("Other confirmed") or col.startswith("Excluded")) else italic(col)
    ax.bar(x, plot_matrix[col].values, bottom=bottom, width=0.68, label=label, **style)
    bottom += plot_matrix[col].values

ax.set_xticks(x)
ax.set_xticklabels([species_axis_label(s) for s in plot_order])
ax.set_ylabel("Mean relative abundance (fraction of all reads)")
ax.set_ylim(0, 1)
ax.set_title(f"Mean genus-level composition by host species "
             f"(top {N_TOP_GENERA_DISPLAY} genera overall)", pad=40)
ax.text(0.5, 1.075, "Each bar is one species' mean community; bars sum to 1",
        transform=ax.transAxes, ha="center", va="bottom", fontsize=9.5, color="0.35")
for xi, s in zip(x, plot_order):
    ax.annotate(f"n = {int((composition_metadata['Species_code'] == s).sum())}",
                xy=(xi, 1.0), xytext=(0, 4), textcoords="offset points", ha="center",
                fontsize=9, color="0.3")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8.5, title="Genus", title_fontsize=9)
ax.grid(axis="x", visible=False)

figure_caption(fig,
    f"Mean relative abundance per host species, over all confirmed genus-level reads "
    f"(N = {len(composition_metadata)} birds that passed the library-quality floor). Coloured segments are "
    f"the {N_TOP_GENERA_DISPLAY} most abundant "
    f"genera across the whole dataset; gray pools every other confirmed genus; the hatched segment is the "
    f"share of reads that taxonomic curation could not confirm at genus rank (mean "
    f"{100 * float(excluded_share_by_species.mean()):.1f}% per species). The hatched segment is shown for "
    f"read accounting only and is excluded from every statistical analysis in this notebook.",
    y=-0.06)
plt.tight_layout()
composition_bar_path = save_figure(fig, TAXONOMY_FIG_DIR / "genus_composition_stacked_bar.png")
plt.close(fig)   # not displayed here: the interactive version is the next cell

print("09.2 verification")
print("-" * 60)
assert np.allclose(plot_matrix.sum(axis=1).values, 1.0, atol=1e-6), "Stacked segments must sum to 1 per species."
assert composition_bar_path.exists()
print(f"Saved: {composition_bar_path}")
print("OK -- stacked composition bar complete.")

In [ ]:
# Interactive view of figure 09.2 — Mean genus composition of each host species
# ----------------------------------------------------------------------
# Drawn from plot_matrix: the mean relative abundance per host species,
# including the pooled remainder and the unresolved read share. Hovering
# names the genus and the exact share, which a stacked bar cannot do.
_comp_species = [s for s in SPECIES_ORDER if s in plot_matrix.index]
_segment_color = {g: GENUS_PALETTE[i % len(GENUS_PALETTE)] for i, g in enumerate(top_genera)}
_segment_color["Other confirmed genera (pooled)"] = "#BEBEBE"
_segment_color["Excluded / unresolved reads (not a taxon)"] = "#6E6E6E"
# The two pooled segments are not taxa, so they are not set in italics.
_non_taxon_segments = {"Other confirmed genera (pooled)",
                       "Excluded / unresolved reads (not a taxon)"}

fig10_4a = go.Figure()
for _segment in plot_matrix.columns:
    fig10_4a.add_trace(go.Bar(
        y=[species_html(s) for s in _comp_species],
        x=plot_matrix.loc[_comp_species, _segment].values,
        name=_segment if _segment in _non_taxon_segments else html_italic(_segment),
        orientation="h",
        marker=dict(color=_segment_color.get(_segment, "#BEBEBE"),
                    line=dict(color="white", width=0.5)),
        hovertemplate=(f"<b>{_segment if _segment in _non_taxon_segments else html_italic(_segment)}"
                       f"</b><br>%{{y}}<br>%{{x:.1%}} of reads<extra></extra>")))

fig10_4a.update_layout(
    template=PLOTLY_TEMPLATE, barmode="stack", height=460,
    title=dict(text=f"Mean genus composition of each host species "
                    f"(N = {len(composition_metadata)} birds, confirmed genus-level reads)",
               x=0.5, xanchor="center", font=dict(size=16)),
    xaxis=dict(title="Share of reads", tickformat=".0%", range=[0, 1]),
    yaxis=dict(title="", autorange="reversed"),
    legend=dict(title_text="Genus — click to hide, double-click to isolate",
                font=dict(size=10), traceorder="normal"),
    margin=dict(t=90, b=60))
fig10_4a.show()
assert np.allclose(plot_matrix.loc[_comp_species].sum(axis=1).values, 1.0, atol=1e-6), \
    "Stacked segments must account for every read of a species."

In [ ]:
# 09.3 — Figure: bird-to-bird variability behind each species mean, per genus
# ----------------------------------------------------------------------
# A species mean hides how much individual birds vary, which is what
# decides whether a genus is a consistent feature of a species'
# microbiome or is driven by a handful of individuals.
n_cols = 4
n_rows = int(np.ceil(len(top_genera) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.1 * n_cols, 3.1 * n_rows))
axes = np.atleast_1d(axes).flatten()

for ax, genus in zip(axes, top_genera):
    for i, s in enumerate(plot_order, start=1):
        vals = relative_abundance_with_species.loc[
            relative_abundance_with_species["Species_code"] == s, genus].values
        ax.boxplot([vals], positions=[i], widths=0.55, showfliers=False, patch_artist=True,
                   medianprops=dict(color="black", linewidth=1.3),
                   whiskerprops=dict(color="0.5"), capprops=dict(color="0.5"),
                   boxprops=dict(facecolor=SPECIES_COLORS[s], alpha=0.25,
                                 edgecolor=SPECIES_COLORS[s], linewidth=1.1))
        jitter = spawn_rng().normal(0, 0.06, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals, s=16, marker=SPECIES_MARKERS[s],
                   color=SPECIES_COLORS[s], edgecolor="white", linewidth=0.4, alpha=0.85, zorder=3)
    prevalence = float(composition_summary.set_index("Genus").loc[genus, "Prevalence_fraction"])
    ax.set_title(f"{italic(genus)}\npresent in {prevalence:.0%} of birds", fontsize=9.5)
    ax.set_ylabel("Relative abundance")
    ax.set_xticks(range(1, len(plot_order) + 1))
    ax.set_xticklabels([s for s in plot_order], rotation=45, ha="right", fontsize=7.5)
for ax in axes[len(top_genera):]:
    ax.axis("off")

handles = [Line2D([0], [0], marker=SPECIES_MARKERS[s], color="none", markerfacecolor=SPECIES_COLORS[s],
                  markeredgecolor="white", markersize=9, label=species_legend_label(s)) for s in plot_order]
fig.legend(handles=handles, loc="lower center", ncol=len(plot_order), bbox_to_anchor=(0.5, -0.055),
           title="Species (origin)", title_fontsize=9)
fig.suptitle("Bird-to-bird variability behind each species mean", fontsize=14, fontweight="bold", y=1.0)
figure_caption(fig,
    f"One panel per genus, for the {N_TOP_GENERA_DISPLAY} most abundant genera overall. Each point is one "
    f"bird (N = {len(composition_metadata)}); boxes give the median and interquartile range within a species; "
    f"outliers beyond the whiskers are not drawn separately because every individual is already shown. "
    f"Species codes on the x-axis follow the legend. A genus whose box sits near zero while a few points sit "
    f"high is carried by individual birds, not by the species.",
    y=-0.09)
plt.tight_layout()
genus_variability_fig_path = save_figure(fig, TAXONOMY_FIG_DIR / "genus_individual_variability.png")
plt.show(); plt.close(fig)

print("09.3 verification")
print("-" * 60)
assert genus_variability_fig_path.exists()
print(f"Saved: {genus_variability_fig_path}")
print("OK -- individual-variability figure complete.")

In [ ]:
# 09.4 — Figure: prevalence of the most abundant genera across host species
# ----------------------------------------------------------------------
prevalence_by_species = ((composition_counts[top_genera] > 0)
                         .join(composition_metadata["Species_code"])
                         .groupby("Species_code", observed=True)[top_genera].mean()
                         .reindex(plot_order))

fig, ax = plt.subplots(figsize=(11.5, 5.2))
# A single-hue sequential map: prevalence is a quantity, not a category,
# so it must NOT use the categorical palette, and a perceptually uniform
# ramp keeps equal steps looking equal.
im = ax.imshow(prevalence_by_species.values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(top_genera)))
ax.set_xticklabels([italic(g) for g in top_genera], rotation=40, ha="right", fontsize=8.5)
ax.set_yticks(range(len(plot_order)))
ax.set_yticklabels([species_axis_label(s) for s in plot_order], fontsize=9)
for i in range(prevalence_by_species.shape[0]):
    for j in range(prevalence_by_species.shape[1]):
        v = prevalence_by_species.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                color="white" if v < 0.55 else "black", fontweight="bold" if v >= 0.80 else "normal")
cbar = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.035)
# Wrapped onto two lines and given its own room: as one long line the
# label ran past the right edge of the canvas and was cut off.
cbar.set_label("Prevalence: fraction of that\nspecies' birds carrying the genus", fontsize=8.5)
cbar.ax.tick_params(labelsize=8)
ax.set_title(f"Which genera are consistent members of each species' microbiome "
             f"(top {N_TOP_GENERA_DISPLAY} genera)", pad=14)
ax.grid(visible=False)
figure_caption(fig,
    f"Prevalence = the proportion of birds of that species in which the genus was detected at all "
    f"(any non-zero read count), not its abundance. Values >= 0.80 are bold. "
    f"Sample sizes: " + ", ".join(f"{s} n={int((composition_metadata['Species_code'] == s).sum())}" for s in plot_order) +
    ". Compare with figure 09.2, which shows abundance: a genus can be highly prevalent yet rare, or abundant "
    f"in only a few birds.",
    y=-0.08)
plt.tight_layout()
prevalence_heatmap_path = save_figure(fig, TAXONOMY_FIG_DIR / "genus_prevalence_by_species_final.png")
plt.close(fig)   # not displayed here: the interactive version is the next cell

print("09.4 verification")
print("-" * 60)
assert (prevalence_by_species.values >= 0).all() and (prevalence_by_species.values <= 1).all()
assert prevalence_heatmap_path.exists()
print(f"Saved: {prevalence_heatmap_path}")
print("OK -- prevalence heatmap complete. Descriptive composition (09.1-09.4) finished.")

In [ ]:
# Interactive view of figure 09.4 — Prevalence of the most abundant genera
# ----------------------------------------------------------------------
# Drawn from prevalence_by_species: in how many birds of a species each
# genus occurs at all, irrespective of how many reads it takes when it
# does. Read together with the composition above, this is the
# monodominance result of section 05.3: a genus can hold a large share of
# the reads of a species while being present in only a few of its birds.
fig10_4b = go.Figure(data=go.Heatmap(
    z=prevalence_by_species.loc[_comp_species, top_genera].values,
    x=[html_italic(g) for g in top_genera], y=[species_html(s) for s in _comp_species],
    colorscale="Blues", zmin=0, zmax=1,
    colorbar=dict(title="Share of<br>birds", tickformat=".0%"),
    hovertemplate="<b>%{x}</b><br>%{y}<br>present in %{z:.0%} of birds<extra></extra>"))
fig10_4b.update_layout(
    template=PLOTLY_TEMPLATE, height=380,
    title=dict(text=f"Prevalence of the {len(top_genera)} most abundant genera across host species",
               x=0.5, xanchor="center", font=dict(size=16)),
    xaxis=dict(title="", tickangle=-35), yaxis=dict(title="", autorange="reversed"),
    margin=dict(t=90, b=140))
fig10_4b.show()
assert prevalence_by_species.loc[_comp_species, top_genera].max().max() <= 1.0

In [ ]:
# 09.5 — clr_transformed: prevalence filter, multiplicative zero replacement and the CLR transform
# ----------------------------------------------------------------------
# Zero replacement and the CLR transform, implemented from their
# published definitions.
def multiplicative_replacement(proportions: np.ndarray, library_sizes: np.ndarray,
                               delta_fraction: float = 0.65) -> np.ndarray:
    """Replace zeros in a closed composition (Martin-Fernandez et al. 2003).

    NOT a blanket "+1 pseudocount to everything", which perturbs every
    ratio rather than only those containing a structural zero. Each zero
    cell is replaced by delta_i = delta_fraction / N_i, where N_i is that
    sample's FULL library size -- one read out of the whole library is
    its real detection limit, so the replacement value must be derived
    from the library, not from the filtered subset. The non-zero parts
    are then rescaled by (1 - n_zeros_i * delta_i) so the row still sums
    to 1, and ratios between the observed parts are preserved exactly.
    """
    x = np.asarray(proportions, dtype=float).copy()
    totals = np.asarray(library_sizes, dtype=float)
    for i in range(x.shape[0]):
        zeros = x[i] == 0
        n_zeros = int(zeros.sum())
        if n_zeros == 0:
            continue
        delta = delta_fraction / totals[i]
        if n_zeros * delta >= 0.5:
            raise ValueError(
                f"Row {i}: the zero-replacement values would take up more than half of the composition "
                f"({n_zeros} zeros x delta = {n_zeros * delta:.3g}). That sample has too shallow a library "
                f"for a compositional analysis and should have been excluded by the filters above."
            )
        x[i, ~zeros] *= (1.0 - n_zeros * delta)
        x[i, zeros] = delta
    return x


def clr_transform(proportions: np.ndarray) -> np.ndarray:
    """Centered log-ratio (Aitchison 1986): clr(x) = log(x) - mean(log(x)), row-wise."""
    x = np.asarray(proportions, dtype=float)
    if (x <= 0).any():
        raise ValueError("CLR is undefined for zero or negative parts; replace zeros first.")
    log_x = np.log(x)
    return log_x - log_x.mean(axis=1, keepdims=True)


# A stricter prevalence filter than the low-support filter of 02.0 (which
# only removes near-absent genera): compositional analysis is sensitive to
# rare, sparsely observed genera dominating a log-ratio, so a genus must be
# present in a meaningful fraction of samples to be included as its own part.
CLR_MIN_PREVALENCE_FRACTION = 0.20
# A sample also has to carry enough information to BE a composition:
# a handful of reads spread over two parts is not a community profile,
# and its log-ratios would be dominated by the zero replacement. Both
# thresholds are pre-specified, and every excluded sample is listed.
CLR_MIN_NONZERO_PARTS = 3
CLR_MIN_RETAINED_READS = 100

genus_prevalence_fraction = (composition_counts > 0).mean(axis=0)
clr_genera = genus_prevalence_fraction[genus_prevalence_fraction >= CLR_MIN_PREVALENCE_FRACTION].index.tolist()
assert len(clr_genera) >= 3, (
    f"Only {len(clr_genera)} genera reach a prevalence of {CLR_MIN_PREVALENCE_FRACTION:.0%}; "
    f"a compositional analysis needs at least 3 parts. Lower CLR_MIN_PREVALENCE_FRACTION."
)
clr_input_counts = composition_counts[clr_genera].copy()

_retained_reads = clr_input_counts.sum(axis=1)
_nonzero_parts = (clr_input_counts > 0).sum(axis=1)
_keep = (_retained_reads >= CLR_MIN_RETAINED_READS) & (_nonzero_parts >= CLR_MIN_NONZERO_PARTS)
clr_excluded_samples = pd.DataFrame({
    "Retained_reads": _retained_reads[~_keep], "Nonzero_parts": _nonzero_parts[~_keep],
    "Library_size": composition_counts.sum(axis=1)[~_keep],
}).join(analysis_metadata["Species_code"])
clr_excluded_samples.to_csv(QC_REPORTS_DIR / "compositional_analysis_excluded_samples.csv")
clr_input_counts = clr_input_counts.loc[_keep]

# delta comes from the FULL library size, which is the true detection limit.
clr_library_sizes = composition_counts.sum(axis=1).loc[clr_input_counts.index]
clr_proportions = clr_input_counts.div(clr_input_counts.sum(axis=1), axis=0)
n_zero_cells = int((clr_proportions.values == 0).sum())
clr_replaced = multiplicative_replacement(clr_proportions.values, clr_library_sizes.values)
clr_transformed = pd.DataFrame(clr_transform(clr_replaced),
                               index=clr_proportions.index, columns=clr_proportions.columns)
clr_transformed.to_csv(TAXONOMY_RESULTS_DIR / "clr_transformed_matrix.csv")

print("09.5 verification")
print("-" * 60)
# Self-tests against the published properties of both transforms.
_p = np.array([[0.5, 0.3, 0.2], [0.6, 0.4, 0.0]])
_rep = multiplicative_replacement(_p, np.array([10000.0, 10000.0]))
assert np.allclose(_rep.sum(axis=1), 1.0), "Zero replacement must keep each composition closed to 1."
assert (_rep > 0).all(), "Zero replacement must leave no zeros."
assert np.isclose(_rep[1, 0] / _rep[1, 1], _p[1, 0] / _p[1, 1]), "Ratios between observed parts must be preserved."
assert np.allclose(clr_transform(_p[:1]), np.log(_p[:1]) - np.log(_p[:1]).mean()), "CLR definition."
assert np.allclose(clr_transform(_p[:1] * 7), clr_transform(_p[:1])), "CLR must be scale-invariant."
print("Self-tests passed: closure preserved, no zeros left, observed ratios unchanged, CLR scale-invariant.")
print(f"\nFilters: prevalence >= {CLR_MIN_PREVALENCE_FRACTION:.0%}, and per sample "
      f">= {CLR_MIN_RETAINED_READS} retained reads and >= {CLR_MIN_NONZERO_PARTS} non-zero parts.")
print(f"Genera retained for compositional analysis: {len(clr_genera)}/{composition_counts.shape[1]}")
print(f"Samples: {clr_transformed.shape[0]}/{composition_counts.shape[0]} (non-rarefied curated counts)")
if len(clr_excluded_samples):
    print(f"Samples excluded (too little signal in the retained genera): {len(clr_excluded_samples)}; "
          f"listed in {QC_REPORTS_DIR / 'compositional_analysis_excluded_samples.csv'}")
    print(clr_excluded_samples.head(10).to_string())
print(f"Zero cells replaced (multiplicative replacement, not a blanket pseudocount): "
      f"{n_zero_cells}/{clr_proportions.size} ({100 * n_zero_cells / clr_proportions.size:.1f}%)")
assert np.allclose(clr_transformed.sum(axis=1).values, 0.0, atol=1e-6), "CLR rows must sum to ~0 by construction."
assert not clr_transformed.isna().any().any()
print(f"Saved: {TAXONOMY_RESULTS_DIR / 'clr_transformed_matrix.csv'}")
print("OK -- CLR matrix ready (Euclidean distance on it IS Aitchison distance).")

In [ ]:
# 09.6 — Compositional PERMANOVA on Aitchison distance, and agreement with Bray-Curtis
# ----------------------------------------------------------------------
# Aitchison distance is Euclidean distance between CLR coordinates, so
# the distance matrix is a plain `pdist` on `clr_transformed`. This
# section depends only on section 09.5, not on the Bray-Curtis work in
# section 08.0.
aitchison_distance = pd.DataFrame(squareform(pdist(clr_transformed.values, metric="euclidean")),
                                  index=clr_transformed.index, columns=clr_transformed.index)

compositional_metadata = analysis_metadata.loc[
    analysis_metadata.index.intersection(aitchison_distance.index), ["Species_code", "Locality"]].dropna()
compositional_metadata = compositional_metadata.loc[~compositional_metadata["Locality"].isin(excluded_localities)]
compositional_metadata = compositional_metadata.loc[
    compositional_metadata["Species_code"].isin(modeled_species)]
compositional_ids = compositional_metadata.index.tolist()
compositional_distance = aitchison_distance.loc[compositional_ids, compositional_ids]

compositional_permanova, compositional_permanova_meta = marginal_permanova(
    compositional_distance, compositional_metadata, terms=[SPECIES_TERM, LOCALITY_TERM],
    n_permutations=N_PERMUTATIONS, rng=spawn_rng())
compositional_permanova["q_value"] = fdr_within_family(
    dict(zip(compositional_permanova["Term"], compositional_permanova["p_value"])))["q_value"].values
compositional_permanova.to_csv(STATISTICS_DIR / "compositional_permanova_aitchison.csv", index=False)

# PERMDISP on the EXACT same samples, for both terms of the PERMANOVA, as in
# section 08.4. A significant dispersion difference means that term of the
# PERMANOVA cannot be read as a pure shift between group centroids. Both tests
# draw from one spawned generator, so the species result is identical to the
# one this section reported before locality was added.
_comp_permdisp_rng = spawn_rng()
compositional_permdisp_species = permdisp(
    compositional_distance, compositional_metadata["Species_code"].astype(str).values,
    N_PERMUTATIONS, _comp_permdisp_rng)
compositional_permdisp_locality = permdisp(
    compositional_distance, compositional_metadata["Locality"].astype(str).values,
    N_PERMUTATIONS, _comp_permdisp_rng)

print("09.6 verification")
print("-" * 60)
print(f"Compositional PERMANOVA N = {compositional_permanova_meta['n']} "
      f"(Aitchison distance on {len(clr_genera)} genera, non-rarefied)")
print(f"\nPERMDISP on the matched {compositional_permdisp_species['N']} birds:")
print(f"  Species grouping : F = {compositional_permdisp_species['F']:.3f}, "
      f"p = {compositional_permdisp_species['p_value']:.4g}")
print(f"  Locality grouping: F = {compositional_permdisp_locality['F']:.3f}, "
      f"p = {compositional_permdisp_locality['p_value']:.4g}")
print("  Mean distance to the group centroid (higher = more variable community):")
for _label, _disp in (("species", compositional_permdisp_species),
                      ("locality", compositional_permdisp_locality)):
    for _g, _v in sorted(_disp["group_mean_distance"].items(), key=lambda kv: -kv[1]):
        print(f"    {_label:8s} {_g:28s} {_v:.4f}")
assert compositional_permanova_meta["n"] == len(compositional_ids) == compositional_distance.shape[0]
assert compositional_permdisp_species["N"] == compositional_permanova_meta["n"]
assert compositional_permdisp_locality["N"] == compositional_permanova_meta["n"], \
    "PERMDISP and PERMANOVA must be computed on the same birds."

# Agreement with the Bray-Curtis result is a genuine robustness check:
# the two analyses use different distances, different data (rarefied vs
# not) and different assumptions.
_bc = global_permanova.set_index("Term").loc[SPECIES_TERM]
_ait = compositional_permanova.set_index("Term").loc[SPECIES_TERM]
_agree = (_bc["p_value"] < ALPHA_SIGNIFICANCE) == (_ait["p_value"] < ALPHA_SIGNIFICANCE)
print(f"\nSpecies effect on community composition, by two independent routes:")
print(f"  Bray-Curtis, rarefied to {rarefaction_depth:,} reads (08.4) : "
      f"R2={_bc['R2']:.3f}, p={_bc['p_value']:.4g}  (N={int(_bc['N'])})")
print(f"  Aitchison / CLR, non-rarefied counts       (09.6) : "
      f"R2={_ait['R2']:.3f}, p={_ait['p_value']:.4g}  (N={int(_ait['N'])})")
print(f"  -> the two approaches {'AGREE' if _agree else 'DISAGREE'} on whether species structures composition.")
if not _agree:
    print("[NOTE] They disagree. Report both: the Bray-Curtis result depends on the rarefaction depth, the "
          "Aitchison result depends on the prevalence filter and the zero replacement. Neither is 'the' answer.")
for _term, _disp in ((SPECIES_TERM, compositional_permdisp_species),
                     (LOCALITY_TERM, compositional_permdisp_locality)):
    _name = "Species" if _term == SPECIES_TERM else "Locality"
    if _disp["p_value"] < ALPHA_SIGNIFICANCE:
        print(f"\n[INTERPRETATION] PERMDISP is significant for {_name} on Aitchison distance: part of "
              f"that PERMANOVA effect may reflect a difference in within-group VARIABILITY rather than "
              f"purely a shift in the average composition. Report the two tests together.")
    else:
        print(f"\n[INTERPRETATION] PERMDISP is not significant for {_name} on Aitchison distance: the "
              f"within-group dispersions are homogeneous, so that PERMANOVA effect can be read as a "
              f"difference in average composition.")
print(f"\nSaved: {STATISTICS_DIR / 'compositional_permanova_aitchison.csv'}")
print("OK -- compositional PERMANOVA complete.")

In [ ]:
# Table · 09.6 — Compositional PERMANOVA on Aitchison distance
show_table(compositional_permanova, "Compositional PERMANOVA on Aitchison distance between the CLR coordinates of the non-rarefied curated counts, the independent route to the same question as 08.4.")

In [ ]:
# 09.7 — Figure: compositional ordination of clr_transformed, with genus loadings
# ----------------------------------------------------------------------
# Aitchison distance is Euclidean distance on CLR coordinates, so the
# ordination that matches it is an ordinary PCA of `clr_transformed` --
# no eigenvalue correction, no stress, and axis percentages that mean
# what they say. This is the compositional counterpart of the Bray-Curtis
# PCoA in 08.7: it works on the non-rarefied curated counts, is driven by
# ratios between genera rather than by their absolute abundances, and is
# therefore much less dominated by whichever single genus happens to have
# taken over a bird.
clr_centered = clr_transformed.loc[compositional_ids] - clr_transformed.loc[compositional_ids].mean(axis=0)
_u, _s, _vt = np.linalg.svd(clr_centered.values, full_matrices=False)
clr_scores = pd.DataFrame(_u[:, :2] * _s[:2], index=clr_centered.index, columns=["PC1", "PC2"])
clr_variance_explained = 100 * (_s ** 2) / float((_s ** 2).sum())

# Genus loadings: which genera pull the ordination apart. Shown for the
# few with the longest vectors, which is what makes a compositional
# ordination interpretable rather than decorative.
N_LOADINGS_SHOWN = 6
loadings = pd.DataFrame(_vt[:2].T, index=clr_centered.columns, columns=["PC1", "PC2"])
loadings["length"] = np.hypot(loadings["PC1"], loadings["PC2"])
top_loadings = loadings.nlargest(N_LOADINGS_SHOWN, "length")
loadings.sort_values("length", ascending=False).to_csv(
    TAXONOMY_RESULTS_DIR / "aitchison_pca_genus_loadings.csv")

fig, ax = plt.subplots(figsize=(9.5, 7.2))
for species in SPECIES_ORDER:
    ids = compositional_metadata.index[compositional_metadata["Species_code"] == species].intersection(clr_scores.index)
    if len(ids) == 0:
        continue
    pts = clr_scores.loc[ids]
    ax.scatter(pts["PC1"], pts["PC2"], color=SPECIES_COLORS[species], marker=SPECIES_MARKERS[species],
               label=species_legend_label(species), alpha=0.85, s=44,
               edgecolor="white", linewidth=0.6, zorder=3)
    ax.scatter(pts["PC1"].mean(), pts["PC2"].mean(), color=SPECIES_COLORS[species],
               marker=SPECIES_MARKERS[species], s=200, edgecolor="black", linewidth=1.5, zorder=5)

# Loading arrows, scaled to the span of the scores so they are readable
# on the same axes without claiming a shared unit.
_scale = 0.42 * max(clr_scores["PC1"].abs().max(), clr_scores["PC2"].abs().max()) / max(
    top_loadings["length"].max(), 1e-12)
for genus, row in top_loadings.iterrows():
    ax.annotate("", xy=(row["PC1"] * _scale, row["PC2"] * _scale), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color="0.35", linewidth=1.1, alpha=0.9), zorder=4)

# Genera with similar loadings put their arrow tips close together, so
# the labels are decluttered rather than written at the tips: within each
# side of the plot they are pushed apart vertically until they clear each
# other, and a thin leader line keeps each name attached to its arrow.
ax.autoscale_view()
_y_span = ax.get_ylim()[1] - ax.get_ylim()[0]
_min_gap = 0.055 * _y_span
for _side in (1, -1):
    _rows = [(g, r["PC1"] * _scale, r["PC2"] * _scale) for g, r in top_loadings.iterrows()
             if (r["PC1"] >= 0) == (_side > 0)]
    _rows.sort(key=lambda t: -t[2])
    _placed = []
    for _g, _tx, _ty in _rows:
        if _placed and _ty > _placed[-1] - _min_gap:
            _ty = _placed[-1] - _min_gap
        _placed.append(_ty)
        _lx = _tx + _side * 0.04 * (ax.get_xlim()[1] - ax.get_xlim()[0])
        ax.annotate(italic(_g), xy=(_tx, _placed[-1]), xytext=(_lx, _placed[-1]),
                    ha="left" if _side > 0 else "right", va="center",
                    fontsize=8, color="0.2", zorder=6,
                    arrowprops=dict(arrowstyle="-", color="0.65", linewidth=0.6,
                                    shrinkA=0, shrinkB=2))

ax.axhline(0, color="0.85", linewidth=0.7, zorder=0)
ax.axvline(0, color="0.85", linewidth=0.7, zorder=0)
ax.set_xlabel(f"PC1 ({clr_variance_explained[0]:.1f}% of compositional variance)")
ax.set_ylabel(f"PC2 ({clr_variance_explained[1]:.1f}% of compositional variance)")
_ait_row = compositional_permanova.set_index("Term").loc[SPECIES_TERM]
ax.set_title(f"Compositional ordination of the cloacal microbiota (Aitchison / CLR)\n"
             f"PERMANOVA species effect: $R^2$ = {_ait_row['R2']:.3f}, "
             f"{format_p(_ait_row['p_value'])} {significance_stars(_ait_row['p_value'])}",
             fontsize=12.5, fontweight="bold")
ax.legend(loc="best", fontsize=8.5, title="Species (origin)", title_fontsize=9, framealpha=0.92)
figure_caption(fig,
    f"Principal components of the centered log-ratio matrix ({len(clr_genera)} genera passing the "
    f"{100 * CLR_MIN_PREVALENCE_FRACTION:.0f}% prevalence filter, {len(clr_scores)} birds, non-rarefied counts). "
    f"Euclidean distance on these coordinates is Aitchison distance, so this ordination is the exact "
    f"companion of the PERMANOVA in 09.6 and needs no stress or eigenvalue correction. Because CLR works "
    f"on ratios between genera, the ordination is not dominated by whichever genus happens to hold most of "
    f"a bird's reads, which is what distinguishes it from the Bray-Curtis ordination in 08.7. Gray arrows "
    f"are the {N_LOADINGS_SHOWN} genera with the largest loadings; they point toward the birds in which "
    f"that genus is over-represented relative to the community's geometric mean. Large black-outlined "
    f"symbols are species centroids. Full loadings: "
    f"results/taxonomy/aitchison_pca_genus_loadings.csv.",
    y=-0.06)
plt.tight_layout()
clr_ordination_path = save_figure(fig, TAXONOMY_FIG_DIR / "compositional_ordination_aitchison.png")
plt.show(); plt.close(fig)

print("09.7 verification")
print("-" * 60)
print(f"CLR matrix ordinated: {clr_centered.shape[0]} birds x {clr_centered.shape[1]} genera")
print(f"Variance explained: PC1 {clr_variance_explained[0]:.1f}%, PC2 {clr_variance_explained[1]:.1f}% "
      f"(first two axes together {clr_variance_explained[:2].sum():.1f}%)")
print(f"\nGenera with the largest loadings on the first two components:")
print(top_loadings.round(3).to_string())
# CLR coordinates sum to zero within a sample, so the matrix is rank
# deficient by one and the last singular value must be ~0. Checking it
# confirms the transform and the centering are behaving.
assert abs(clr_transformed.sum(axis=1)).max() < 1e-8, "CLR rows must sum to zero."
assert clr_variance_explained[:2].sum() <= 100.0 + 1e-9
assert clr_scores.notna().all().all()
print(f"\nSaved: {clr_ordination_path}")
print("OK -- section 09.0 complete.")

In [ ]:
# Table · 09.7 — Genus loadings on the first two compositional components
show_table(loadings.sort_values("length", ascending=False), "Genus loadings on the first two compositional principal components, from clr_transformed: which genera pull the ordination apart, and in which direction.", index=True, page_length=15)

In [ ]:
# Interactive view of figures 08.7A and 09.7 — Community ordinations, bird by bird
# ----------------------------------------------------------------------
# The interactive counterparts of figures 08.7A and 09.7, drawn from
# pcoa_coords (Bray-Curtis on the rarefied matrix) and clr_scores
# (Aitchison on the CLR of the non-rarefied curated counts). The two
# panels answer the same question by routes that share no assumption:
# one works on relative abundances after subsampling to a common depth,
# the other on ratios between genera without subsampling at all. Hovering
# names the bird, which is what a static ordination cannot do and what a
# reader chasing an outlier actually needs.
#
# Each panel is restricted to the birds its own PERMANOVA was computed on,
# so the statistic printed in a subtitle describes exactly the points
# below it. A species described but not modeled therefore does not appear
# here; its position among the others is in figure 08.7A, where the
# unconstrained ordination shows every bird.
#
# Neither panel carries inference: both PERMANOVA results are computed on
# the full distance matrix, in 08.4 and 09.6.
_pcoa_row = global_permanova.set_index("Term").loc[SPECIES_TERM]
_ait_row = compositional_permanova.set_index("Term").loc[SPECIES_TERM]

fig10_3 = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.09,
    subplot_titles=[
        f"Bray-Curtis, principal coordinates<br><sub>PERMANOVA species effect: "
        f"R&#178; = {_pcoa_row['R2']:.3f}, {format_p(_pcoa_row['p_value'])}</sub>",
        f"Aitchison, compositional principal components<br><sub>PERMANOVA species effect: "
        f"R&#178; = {_ait_row['R2']:.3f}, {format_p(_ait_row['p_value'])}</sub>"])

_panels = [
    (1, pcoa_coords.loc[beta_global_ids], beta_global_metadata, "PCoA1", "PCoA2",
     f"PCoA axis 1 ({pct_explained_axis1:.1f}% of variation)",
     f"PCoA axis 2 ({pct_explained_axis2:.1f}% of variation)"),
    (2, clr_scores, compositional_metadata, "PC1", "PC2",
     f"PC1 ({clr_variance_explained[0]:.1f}% of compositional variance)",
     f"PC2 ({clr_variance_explained[1]:.1f}% of compositional variance)"),
]

for _col, _coords, _meta, _xcol, _ycol, _xlab, _ylab in _panels:
    for _species in SPECIES_ORDER:
        _ids = _meta.index[_meta["Species_code"] == _species].intersection(_coords.index)
        if len(_ids) == 0:
            continue
        _described = _species not in modeled_species
        _pts = _coords.loc[_ids]
        fig10_3.add_trace(
            go.Scatter(
                x=_pts[_xcol].values, y=_pts[_ycol].values, mode="markers",
                name=species_html(_species) + (" (described only)" if _described else ""),
                legendgroup=_species, showlegend=(_col == 1),
                marker=dict(size=9, symbol=PLOTLY_MARKERS[SPECIES_MARKERS[_species]],
                            color="rgba(0,0,0,0)" if _described else SPECIES_COLORS[_species],
                            line=dict(color=SPECIES_COLORS[_species] if _described else "white",
                                      width=1.6 if _described else 0.7)),
                customdata=np.column_stack([_pts.index.values,
                                            _meta.loc[_ids, "Locality"].astype(str).values]),
                hovertemplate=(f"<b>%{{customdata[0]}}</b><br>"
                               f"{species_html(_species, origin=True)}<br>"
                               f"Locality: %{{customdata[1]}}<extra></extra>")),
            row=1, col=_col)
        # A species too sparsely sampled to be modeled is drawn without a
        # centroid, exactly as in the static figures: a centroid through a
        # handful of birds invites a comparison the design cannot support.
        if not _described:
            fig10_3.add_trace(
                go.Scatter(
                    x=[float(_pts[_xcol].mean())], y=[float(_pts[_ycol].mean())], mode="markers",
                    legendgroup=_species, showlegend=False,
                    marker=dict(size=19, symbol=PLOTLY_MARKERS[SPECIES_MARKERS[_species]],
                                color=SPECIES_COLORS[_species],
                                line=dict(color="black", width=1.8)),
                    hovertemplate=(f"<b>{species_html(_species)} centroid</b><br>"
                                   f"n = {len(_ids)} birds<extra></extra>")),
                row=1, col=_col)
    fig10_3.update_xaxes(title_text=_xlab, zeroline=True, zerolinecolor="#e6e6e6", row=1, col=_col)
    fig10_3.update_yaxes(title_text=_ylab, zeroline=True, zerolinecolor="#e6e6e6", row=1, col=_col)

fig10_3.update_layout(
    template=PLOTLY_TEMPLATE, height=560,
    title=dict(text="Bacterial community structure by host species, by two independent routes",
               x=0.5, xanchor="center", font=dict(size=16)),
    legend=dict(orientation="h", yanchor="bottom", y=-0.26, xanchor="center", x=0.5,
                title_text="Host species — large outlined symbols are species centroids"),
    margin=dict(t=110, b=120))
fig10_3.show()

assert len(pcoa_coords.loc[beta_global_ids]) == global_permanova_meta["n"], \
    "The ordination panel must show exactly the birds its PERMANOVA was computed on."

## 11.0 — Summary of outputs

Every section above ends with a verification block — printed diagnostics and `assert` statements —
so that each step can be confirmed before the next one relies on it, and every result table is
rendered as a searchable table in a cell of its own.

**Where the results are.** Statistical tables in `results/statistics/` and `results/taxonomy/`, QC
reports in `results/qc_reports/`, figures in `images_graphs/`, intermediate matrices in
`data/processed/`. Section 11.1 indexes all of them, and writes that index to
`results/output_index.csv`.

**The files a reader will want first**, in the order the questions usually arrive:

- `results/statistics/rarefaction_depth_candidates.csv` and `rarefaction_depth_selection.txt` —
  the rarefaction depth, the trade-off behind it, and the coverage it achieves.
- `results/statistics/alpha_diversity_rarefaction_stability.csv` — evidence that the diversity
  values do not depend on which random subsample was drawn.
- `results/statistics/inter_specific_global_effect_tests.csv`,
  `inter_specific_marginal_means.csv` and `inter_specific_posthoc_pairwise.csv` — the main
  between-species result.
- `results/statistics/inter_specific_depth_sensitivity.csv` and `assignment_rate_sensitivity.csv`
  — that result's robustness to the rarefaction depth and to uneven taxonomic assignment.
- `results/statistics/global_permanova.csv` and `compositional_permanova_aitchison.csv` — the
  community-composition result by two independent routes.
- `results/statistics/dominance_by_species.csv` — how often each host species carries a community
  dominated by a single genus, and which genus.
- `results/qc_reports/unresolved_read_share_by_species.csv` and
  `results/statistics/design_species_locality_concentration.csv` — the two limitations that have
  to be stated when interpreting any species-level result.

**What this notebook reports, and what it does not.** One model per question, and for each response
the family its support calls for rather than the best of a search: a count family for richness
chosen from the observed dispersion, a Gamma with a log link for Shannon diversity, and a beta
regression for Simpson diversity. Every one of them is checked against its own distribution with
randomized quantile residuals in 06.9 and 07.5, and every PERMANOVA is reported with a PERMDISP on
the identical samples, because the two together decide whether a significant result may be described
as a difference in average composition or only as a difference in community structure.

Three things were examined during model development and are deliberately not reported as results.
Pielou's evenness is computed and archived for every bird but not modeled, because *J' = H' / log S*
restates what the richness and Shannon models already carry. A Gaussian and a log-normal family for
Shannon diversity were considered and rejected; section 06.6 is the evidence, and it quantifies how
much probability a Gaussian would place at impossible negative diversities. A non-metric
multidimensional scaling of the Bray–Curtis matrix is fitted in 08.2 for its stress alone, which is
the statistic that decides whether it or principal coordinates should be shown, and its configuration
is not presented.

**Reading this notebook as a web page.** Exporting it produces one self-contained file in which the
tables stay searchable and the interactive figures stay live, with no kernel behind them:

```bash
jupyter nbconvert --to html --embed-images TFM_microbiome_pipeline.ipynb
```

In [ ]:
# 11.1 — Index of everything this notebook wrote
# ----------------------------------------------------------------------
# Built by walking the output directories after the fact rather than from
# a hand-kept list, so the index cannot fall out of step with what was
# actually produced. Each table is listed with its shape, so a reader can
# tell at a glance whether a file holds one row per bird, per genus or per
# test before opening it.
_output_rows = []
for _directory in sorted({RESULTS_DIR, FIGURES_DIR, PROCESSED_DATA_DIR}):
    for _path in sorted(_directory.rglob("*")):
        if not _path.is_file() or _path.name.startswith("."):
            continue
        _shape = ""
        if _path.suffix == ".csv":
            try:
                _frame = pd.read_csv(_path, nrows=None, low_memory=False)
                _shape = f"{len(_frame)} rows x {_frame.shape[1]} cols"
            except Exception:                            # noqa: BLE001 -- listed without a shape
                _shape = "unreadable as a flat table"
        _output_rows.append({
            "Folder": str(_path.parent.relative_to(PROJECT_DIR)),
            "File": _path.name,
            "Kind": {".csv": "table", ".png": "figure", ".txt": "text"}.get(_path.suffix, _path.suffix or "file"),
            "Size_KB": round(_path.stat().st_size / 1024, 1),
            "Shape": _shape})
output_index = pd.DataFrame(_output_rows).sort_values(["Folder", "File"]).reset_index(drop=True)
output_index.to_csv(RESULTS_DIR / "output_index.csv", index=False)

print("10.5 verification")
print("-" * 60)
print(f"\nTotal: {len(output_index)} files, "
      f"{output_index['Size_KB'].sum() / 1024:.1f} MB")
assert len(output_index) > 0, "The notebook produced no output files, which cannot be right."
assert (output_index["Kind"] == "figure").sum() >= 1
print(f"\nSaved: {RESULTS_DIR / 'output_index.csv'}")
print("OK -- section 11.0 complete.")

In [ ]:
# Table · 11.1 — Every file this notebook wrote
show_table(output_index, "Every file this notebook wrote, found by walking the output directories "
           "after the run. Search for a file name to see where it went and how large it is.",
           page_length=15)